# Lightweight Adaptive-Receptive-Field Crowd Counting on ShanghaiTech (< 1M parameters)

> **Default model: `LightARFSCounter`** (Section 6.3) — MobileNetV2 front-end + adaptive receptive-field selection + tiny channel/spatial attention, < 1M parameters. The original VGG16 CNN-Transformer hybrid described below is still available with `cfg.model_type = "hybrid"`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RashmiNair2021/Flask_basic/blob/claude/cnn-transformer-crowd-counting-6h4tom/colab/ShanghaiTech_CNN_Transformer_Crowd_Counting.ipynb)

An end-to-end, GPU-ready pipeline for crowd counting on the **ShanghaiTech** dataset (Part A or Part B),
built around a hybrid architecture that combines:

- **CNN backbone** (VGG16 transfer learning) for local feature extraction
- **Multi-scale feature fusion** (mini feature-pyramid across two backbone depths)
- **Dilated Pyramid Pooling Module (DPPM)** — an ASPP-style block combining *dilated convolutions*
  at multiple rates with a global-pooling *pyramid* branch for multi-scale context
- **Spatial Transformer Network (STN)** to learn perspective/scale-invariant feature warping
- **Transformer encoder** (multi-head self-attention + 2D positional encoding) for long-range,
  global crowd context
- **Graph-based occlusion reasoning module** — a lightweight GNN that lets occluded/ambiguous
  regions borrow evidence from visually-similar regions elsewhere in the scene
- **Image patching** for both training (random-crop patches) and inference (tiled sliding-window),
  plus **mixed precision**, **gradient accumulation**, **backbone freezing**, and **gradient
  checkpointing** for memory efficiency
- **Adaptive learning rate** (`ReduceLROnPlateau`, driven by validation MAE)
- **Best-model checkpointing** — compares every epoch against the best validation result so far
- **Train / validation split** with live loss & MAE (the regression analogue of "accuracy") curves
- A **combined loss**: pixel-wise MSE + SSIM (structural) + a direct counting L1 term — a
  combination shown in crowd-counting literature to be more effective than any single term alone

> Runtime: **Runtime → Change runtime type → GPU (T4/A100)** before executing.


## Latest revision: RMSC-style robustness study (Sections 16-26)

| Added | Where |
|---|---|
| 12 corruptions x 5 severities (RMSC-Net definitions) with fixed per-image seeds; FGSM / PGD / BIM attacks through the scored tiled inference | 16, 17 |
| Study variants of the light model: MSRM (= ARFS blocks) on/off, DAA (channel -> spatial -> new reliability gate) on/off, dilation sets, fusion (add / concat / adaptive). The main model is unchanged (same parameters and state_dict) | 6.1, 6.3, 18 |
| ~20 resumable experiments on Drive: Stage I (clean), Stage II (consistency / corruption / adversarial), Stage III (adversarial + consistency = RMSC-Net) | 19, 20 |
| Robustness evaluation with per-image caches on Drive: corruptions, attacks, scale, occlusion, cross-part generalisation, RD % | 21 |
| Explainable AI: Clean <-> Corrupted <-> Adversarial panels (density, error map, spatial attention, reliability map, encoder features, Grad x Input, integrated gradients), channel response, attention/feature stability (correlation, CKA) | 22 |
| Component-wise ablation table (Baseline, A, B, C, RMSC-Net) + dilation / fusion / loss / robust-training ablations, CSV + LaTeX | 23, 24 |
| Statistics on paired per-image errors: paired t / Wilcoxon (chosen by Shapiro-Wilk, both reported), Cohen's d_z / rank-biserial, bootstrap CIs, Holm correction, Friedman + Kendall's W, seed-level CIs | 25 |
| Auto-generated results and discussion (`results/results_discussion.md`) incl. failure cases | 26 |

## Previous revision: bounded training, everything on Drive, Stage I validation diagnostics

The model, loss, data pipeline, augmentation, optimizer, LR schedule and early-stopping rule are
**unchanged**, so a run reaches the same validation/test MAE and RMSE as before. (The notebook prints
RMSE; crowd-counting papers usually report it as "MSE".) Only the following changed:

| Change | Where |
|---|---|
| `cfg.epochs` lowered **300 -> 150**. It is only an upper limit: early stopping ended the previous run at epoch 113 (best smoothed val MAE at epoch 73), and the LR schedule (`ReduceLROnPlateau`) does not depend on `cfg.epochs`, so training up to the stopping epoch is the same | Section 2 |
| `cfg.epochs_per_session` (optional): train at most N epochs per Colab session, save, and stop cleanly. *Run all* in the next session continues from the next epoch | Sections 2, 10 |
| **Everything on Google Drive** under `<drive_root>/<run_name>/`: resumable `last.pth` (+ `last_prev.pth` backup) every epoch, `best_model.pth` on every new best val MAE, snapshots, plus `results/` with `config.json`, `training_history.csv` (rewritten every epoch), `training_status.json`, `model_summary.json`, every diagnostics/test table as CSV/JSON, every figure as PNG, and a running `results/summary.json` | Sections 2, 6.4, 10, 10.3, 11 |
| **Model summary** (params, GFLOPs at 1024x768, FPS) saved to Drive; it runs under `fork_rng`, so it does not shift the random stream that training uses | Section 6.4 |
| **Stage I diagnostics on the validation set (per image)**, ported from RMSC-Net (`RMSC_Net_Colab.ipynb`, Section 9d): bias (over/under-counting), error by density third, share of the 5 worst images, pred-vs-GT plot, and an enlarged-inference search (scale / threshold) chosen on validation only | Section 10.3 |
| Test evaluation also saves per-image predictions and metrics (MAE, RMSE, NAE) to Drive. The headline numbers stay at normal size; the result with the validation-chosen enlargement is reported separately, and only when the search picked something other than normal size | Section 11 |

**Resuming after a GPU disconnect:** reconnect, then **Runtime -> Run all**. The training cell finds
`last.pth` on Drive and continues from the next epoch with the same optimizer/scheduler/RNG state. A
run that already finished goes straight to the diagnostics and evaluation. Keep `cfg.run_name` the
same to resume, or change it to start a new run.

### Earlier revision: lightweight adaptive receptive-field model + resumable training

| Change | Where |
|---|---|
| New default model **`LightARFSCounter`** (< 1M parameters, checked by an `assert`): MobileNetV2 front-end (strides 8/16/32), **adaptive receptive-field selection** blocks (LigMSANet idea; dilated depthwise branches with per-channel + per-pixel softmax selection) and a ~100-parameter **ECA + spatial attention** gate | Sections 2, 6.1, 6.3 (`cfg.model_type = "light"`) |
| The previous VGG16 CNN-Transformer is kept as `cfg.model_type = "hybrid"` for comparison/ablation | Section 6.2 |
| **Checkpoints on Google Drive** every epoch: full training state (model, optimizer, scheduler, scaler, history, best MAE, RNG), atomic writes, one backup copy, periodic snapshots | Sections 2, 10 |
| **Automatic resume** after a GPU timeout/disconnect: reconnect and **Runtime → Run all**; training continues from the last saved epoch | Section 10 |
| Optional clean stop before a session limit (`cfg.max_session_hours`) | Section 2 |
| **Early stopping** on the smoothed validation MAE (`cfg.early_stop_patience`, `cfg.min_epochs`), resume-safe | Sections 2, 10 |
| NaN-recovery cell 10.1 is now opt-in so *Run all* never re-runs it by accident | Section 10.1 |

Each new run needs its own `cfg.run_name`; reusing a name resumes that run.


## What was changed to fix the low performance (read me first)

**Symptoms in the previous run:** val MAE 208.9 (best, epoch 188) / 295.7 (last epoch),
test MAE 122.3 / RMSE 193.6 on Part A, with train loss still high (train *patch* MAE ~40 on
256x256 crops that average only ~60 people, so the model was **under-fitting**, not just
over-fitting), and the LR already at its 1e-6 floor.

| # | Cause | Cell(s) | Fix |
|---|-------|---------|-----|
| 1 | **Evaluation bug:** val/test images were reflect-padded up to a multiple of `patch_size` (256). The padded strip holds mirrored copies of real people but its GT density is zero, so every one of those heads was counted as error (up to 255 px per side). `predict_count` had the same bug. | 15, 39 | Pad only to a multiple of `output_stride` (<8 px); tiling moved inside `tiled_predict`, which crops back to the real image. |
| 2 | **Non-overlapping tiles** at inference: heads cut by a tile border lost half their context and every tile edge had zero-padding artefacts. | 24, 32 | 50% overlapping sliding window with averaging (`cfg.infer_overlap`). |
| 3 | **STN replaced the feature map with a warped copy.** The density map must stay pixel-aligned with GT; a learned zoom/shift moves predicted heads away from their labels. | 18 | STN output added as a zero-initialised residual branch (identity path kept aligned). |
| 4 | **Too little training signal:** 1 random 256px crop per image per epoch (~31 steps/epoch, ~1/8 of each image) and flip-only augmentation. | 5, 15, 16 | `crops_per_image=4`, count-preserving random scale (0.7-1.3x), random grayscale. |
| 5 | **LR schedule collapsed early:** `ReduceLROnPlateau` read a very noisy 45-image val MAE, so it decayed the LR to 1e-6 before convergence. | 5, 28 | Scheduler steps on a 5-epoch moving average of val MAE; factor 0.5, patience 10. |
| 6 | **Same LR (1e-4, AdamW) for the pretrained VGG and the random new modules**, which degrades ImageNet features early on. | 5, 28, 30 | Parameter groups: backbone at 0.1x LR. |
| 7 | **Mixed precision everywhere**, including the custom head's attention/softmax/affine sampling and the density regression. | 4, 21 | Only the VGG backbone runs in mixed precision (bf16 when supported, incl. T4); fusion → decoder run in float32. (A float16-on-T4 variant was tried and produced NaN from step 0, so it was reverted.) |

> The model architecture changed (STN residual branch), so **old checkpoints are not
> compatible**: re-run training from Section 10 rather than the resume cell in 10.1.


## 1. Environment setup

Only a couple of packages are missing from the default Colab image. Everything else
(`torch`, `torchvision`, `numpy`, `scipy`, `matplotlib`, `opencv`, `tqdm`) ships pre-installed.


In [ ]:
!pip -q install kagglehub h5py --upgrade

In [ ]:
import os
import math
import glob
import json
import random
import zipfile
import shutil
import copy
import time
import datetime
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import scipy.io as sio
from scipy.spatial import KDTree

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset

import torchvision.transforms as T
import torchvision.models as tv_models

import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.auto import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

# bfloat16 has the same exponent range as float32, so activations cannot overflow the way they
# do in float16 (max ~65504 -> inf -> NaN through BatchNorm). Prefer it whenever the GPU supports
# it, including T4, where it is emulated: a bit slower, but a float16 run on T4 produced NaN
# losses from the very first step. float16 + GradScaler is only the fallback for GPUs without
# bf16 support. The custom layers after VGG run in float32 regardless (see the model's forward).
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print("Mixed-precision dtype:", AMP_DTYPE)


## 2. Configuration

Every tunable hyperparameter lives in one place. `patch_size` controls both the random-crop
size used during training *and* the tile size used for sliding-window inference — keeping the
transformer's token count constant and memory bounded regardless of the original image size.


In [ ]:
@dataclass
class CFG:
    # ---- dataset ----
    part: str = "A"                      # "A", "B", or "both" (combine both parts for
                                          # more training data -- see the caveat printed
                                          # in Section 3 about their differing domains)
    dataset_search_roots: tuple = (
        "/content/ShanghaiTech", "/content/shanghaitech",
        "/content/drive/MyDrive/ShanghaiTech",
    )
    density_cache_root: str = "/content/density_cache"
    val_split: float = 0.15              # held out from the official *training* split; kept
                                          # larger than a typical 90/10 split because ShanghaiTech
                                          # per-image counts vary hugely (~100s to ~1500+), so a
                                          # small val set makes the epoch-to-epoch MAE too noisy
                                          # for the scheduler/checkpointing to read reliably

    # ---- image / patch handling ----
    patch_size: int = 256                # multiple of output_stride; used for train crops AND tiled inference
    crops_per_image: int = 4             # [PERF FIX] random crops drawn per training image per epoch.
                                          # With 1 crop/image an epoch was only ~31 steps and saw ~1/8
                                          # of each Part A image -> the model under-fitted badly
                                          # (train patch-MAE ~40 on patches averaging ~60 people)
    scale_aug: tuple = (0.7, 1.3)        # [PERF FIX] random, count-preserving rescale of each training
                                          # image -- head sizes in ShanghaiTech vary enormously and
                                          # flip-only augmentation left the model scale-brittle
    gray_aug_prob: float = 0.1           # [PERF FIX] Part A contains some grayscale images
    infer_overlap: float = 0.5           # [PERF FIX] fraction of overlap between sliding-window tiles
    output_stride: int = 8               # backbone downsampling factor (stride of the density head)
    density_scale: float = 100.0         # scales tiny per-pixel density values for stable gradients

    # ---- model ----
    model_type: str = "light"            # "light" = LightARFSCounter (< 1M params, Section 6.3);
                                          # "hybrid" = original VGG16 CNN-Transformer (~11M params)
    max_params: int = 1_000_000          # parameter budget for the light model (checked in Section 6)
    light_width: int = 64                # channels in the light model's head
    light_blocks: int = 3                # number of adaptive receptive-field selection blocks
    light_dilations: tuple = (1, 2, 3, 5)  # receptive fields the ARFS blocks choose between
    pretrained_backbone: bool = True
    freeze_backbone_stage1: bool = True  # freeze the first backbone block (VGG16 conv1-2 / MobileNetV2 stem)
    transformer_layers: int = 2
    transformer_heads: int = 4
    transformer_ff_dim: int = 512
    graph_grid_size: int = 16
    graph_k: int = 8
    use_grad_checkpoint: bool = False    # flip on if you hit OOM on deep backbones
    use_torch_compile: bool = True       # torch.compile() the model for faster train/inference

    # ---- optimization ----
    batch_size: int = 8
    num_workers: int = 2
    base_lr: float = 5e-4                # LR of the newly-initialised head. The small light model
                                          # trains well at 5e-4; use 1e-4 for model_type="hybrid"
    backbone_lr_mult: float = 0.2        # pretrained backbone LR = base_lr * mult (use 0.1 for "hybrid")
    weight_decay: float = 1e-4
    epochs: int = 150                    # UPPER LIMIT only (was 300). Early stopping ended the previous
                                          # run at epoch 113 (best smoothed val MAE at epoch 73), and the
                                          # LR schedule (ReduceLROnPlateau) does not depend on this value,
                                          # so lowering it does not change training up to that point
    epochs_per_session: int = 0          # > 0: train at most this many epochs in one Colab session, save
                                          # to Drive and stop cleanly; Run all continues from the next
                                          # epoch. 0 = no per-session limit
    grad_accum_steps: int = 1
    amp: bool = True                     # automatic mixed precision
    grad_clip_norm: float = 5.0          # caps gradient norm to prevent exploding-gradient NaNs

    # ---- loss weights ----
    lambda_mse: float = 1.0
    lambda_ssim: float = 0.1
    lambda_count: float = 0.02

    # ---- scheduler ----
    lr_factor: float = 0.5
    lr_patience: int = 10                # epochs (of the *smoothed* val MAE) without improvement
    val_mae_smoothing: int = 5           # [PERF FIX] ReduceLROnPlateau is driven by a moving average of
                                          # the last N val MAEs. Driving it by the raw, very noisy
                                          # 45-image val MAE decayed the LR to its 1e-6 floor long
                                          # before the model had converged
    lr_min: float = 1e-6
    early_stop: bool = True              # stop when the smoothed val MAE hasn't improved for
    early_stop_patience: int = 40        # this many epochs (kept > lr_patience so the LR scheduler
                                          # gets a few decays to rescue progress before stopping)
    early_stop_min_delta: float = 0.0    # an improvement must beat the best by more than this
    min_epochs: int = 60                 # never stop before this epoch (noisy early val MAE)
    warmup_epochs: int = 3                # linear LR ramp-up before ReduceLROnPlateau engages,
                                          # so the scheduler's "best" tracking starts once
                                          # training has actually stabilized

    # ---- inference ----
    use_tta: bool = True                 # average a prediction with its horizontal-flip
                                          # counterpart at test/inference time (not during
                                          # per-epoch validation, to keep training fast)

    # ---- Stage I diagnostics on the validation set (Section 10.3) ----
    diag_infer_scales: tuple = (1.0, 1.25, 1.5, 1.75, 2.0)  # enlarged-inference factors to try
    diag_infer_thresholds: tuple = (0, 500, 1000, 1500)     # only enlarge images predicted >= this
                                                            # many people (0 = enlarge every image)

    # ---- RMSC-style robustness study (Sections 16-26) ----
    run_rmsc_study: bool = True          # train the study's experiments (Section 20). False = only
                                          # analyse what is already on Drive
    abl_experiments: tuple = ()          # () = all experiments in EXPERIMENT_ORDER, else a subset
    abl_seeds: tuple = (42,)             # >1 seed = repeated training runs (seed-level statistics)
    abl_s1_max_epochs: int = 150         # Stage I (clean) upper limit; same recipe/early stopping as main
    abl_s2_epochs: int = 40              # Stage II (corruption/consistency) fine-tuning, upper limit
    abl_s3_epochs: int = 20              # Stage III (adversarial) fine-tuning, upper limit
    abl_s2_lr_mult: float = 0.2          # Stage II LR = base_lr * mult (backbone keeps backbone_lr_mult)
    abl_s3_lr_mult: float = 0.1
    abl_s23_patience: int = 15           # early stopping for Stages II/III (smoothed val MAE)
    abl_s23_min_epochs: int = 10
    abl_epochs_per_session: int = 0      # > 0: at most this many study epochs per Colab session
    abl_dilation_sets: tuple = ((1, 2, 3), (1, 2, 4, 8), (1, 3, 5, 7))  # vs the default (1, 2, 3, 5)
    lambda_cons: float = 0.5             # output (density) consistency weight
    lambda_feat: float = 5.0             # feature consistency weight (1 - cosine similarity)
    adv_eps: float = 4 / 255             # L_inf budget in pixel units for adversarial training/eval
    adv_alpha: float = 2 / 255           # PGD step for adversarial TRAINING
    adv_train_steps: int = 3             # PGD steps for adversarial TRAINING (eval uses adv_eval_steps)
    adv_eval_steps: int = 7
    # evaluation
    run_robust_eval: bool = True
    abl_corruption_severities: tuple = (3,)   # severities used for the "Robustness" ablation columns
    robust_full_models: tuple = ("lightarfs_main", "model_b", "model_c", "rmsc_full")  # full sweeps
    robust_eps_255: tuple = (1, 2, 4, 8)      # attack strengths (x/255) for the full sweep
    robust_scales: tuple = (0.5, 0.75, 1.25, 1.5)
    robust_occlusions: tuple = (0.1, 0.2, 0.3)  # fraction of the image area hidden
    robust_eval_max_images: int = 0           # 0 = whole test set
    # explainability
    xai_models: tuple = ("model_b", "rmsc_full")  # falls back to the main model if not trained
    xai_n_samples: int = 3
    xai_corruption: tuple = ("fog", 3)
    xai_ig_steps: int = 16                    # integrated-gradients steps (0 = skip IG)
    xai_stability_images: int = 30
    xai_stability_severity: int = 3
    # statistics
    stats_n_bootstrap: int = 10000
    stats_alpha: float = 0.05

    # ---- checkpoints / resuming across GPU timeouts ----
    run_name: str = "light_arfs_partA_v2"  # one Drive folder per run. Reusing a name CONTINUES that run
                                          # (or reuses it if it already finished); a new name trains
                                          # from scratch. "light_arfs_partA" is the earlier run (kept
                                          # on Drive, not overwritten)
    use_drive: bool = True               # keep checkpoints on Google Drive so they survive the Colab
                                          # runtime being reset (GPU timeout / disconnect / idle)
    require_drive: bool = True           # stop with an error if Drive is not mounted, instead of
                                          # silently saving to the Colab disk (lost on disconnect)
    drive_root: str = "/content/drive/MyDrive/crowd_counting"
    local_root: str = "/content/checkpoints"  # fallback when Drive isn't available (NOT persistent)
    ckpt_dir: str = ""                   # set below to <drive_root or local_root>/<run_name>
    auto_resume: bool = True             # training cell resumes from <ckpt_dir>/last.pth if it exists
    ckpt_every_epochs: int = 1           # full training state (model+optimizer+scheduler+history+RNG)
    snapshot_every_epochs: int = 25      # also keep a weights-only snapshot every N epochs
    max_session_hours: float = 0.0       # > 0: stop cleanly (state saved) after this many hours, e.g.
                                          # set it a little below your Colab session limit


cfg = CFG()


def setup_checkpoint_dir(cfg):
    root = cfg.local_root
    if cfg.use_drive:
        err = None
        try:
            from google.colab import drive
            if not os.path.isdir("/content/drive/MyDrive"):
                drive.mount("/content/drive")
        except Exception as e:  # not on Colab, or the user declined the Drive prompt
            err = e
        if os.path.isdir("/content/drive/MyDrive"):
            root = cfg.drive_root
        elif cfg.require_drive:
            raise RuntimeError(
                f"Google Drive is not mounted ({err}). Nothing would be saved permanently, so the notebook "
                "stops here. Run this cell again and allow Drive access, or set cfg.require_drive = False "
                f"to save to {cfg.local_root} (lost when the runtime resets).")
        else:
            print(f"Google Drive unavailable ({err}); checkpoints go to {cfg.local_root}, which is "
                  "LOST when the runtime resets.")
    cfg.ckpt_dir = os.path.join(root, cfg.run_name)
    os.makedirs(cfg.ckpt_dir, exist_ok=True)
    probe = os.path.join(cfg.ckpt_dir, ".write_test")  # make sure the folder is really writable
    with open(probe, "w") as f:
        f.write("ok")
    os.remove(probe)
    print("Checkpoint directory:", cfg.ckpt_dir, "(Google Drive)" if root == cfg.drive_root else "(LOCAL, not persistent)")
    describe_run_folder(cfg.ckpt_dir)


def describe_run_folder(ckpt_dir):
    """Say clearly whether this run folder already holds a run, and what will happen to it."""
    def stamp(p):
        return datetime.datetime.fromtimestamp(os.path.getmtime(p)).strftime("%Y-%m-%d %H:%M")
    last, best = os.path.join(ckpt_dir, "last.pth"), os.path.join(ckpt_dir, "best_model.pth")
    if not (os.path.exists(last) or os.path.exists(best)):
        print(f"No earlier run in this folder -- training starts from scratch and is saved here.")
        return
    print("=" * 100)
    print(f"EXISTING RUN FOUND in {ckpt_dir}")
    try:
        if os.path.exists(last):
            st = torch.load(last, map_location="cpu", weights_only=False)
            print(f"  last.pth       : {st['epoch']} epoch(s) done, best val MAE {st['best_val_mae']:.2f} (saved {stamp(last)})")
        if os.path.exists(best):
            bk = torch.load(best, map_location="cpu", weights_only=False)
            print(f"  best_model.pth : epoch {bk['epoch']}, val MAE {bk['val_mae']:.2f} (saved {stamp(best)})")
    except Exception as e:
        print(f"  (could not read the checkpoints: {e})")
    print("  Run all CONTINUES this run, or reuses it as-is if it already finished (no new training).")
    print("  To train from scratch instead, change cfg.run_name -- the old folder stays on Drive untouched.")
    print("=" * 100)


setup_checkpoint_dir(cfg)
os.makedirs(cfg.density_cache_root, exist_ok=True)

# ---- results on Drive (next to the checkpoints): tables, figures, a running summary.json ----
RESULT_DIR = os.path.join(cfg.ckpt_dir, "results")
FIG_DIR = os.path.join(RESULT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)
SUMMARY_PATH = os.path.join(RESULT_DIR, "summary.json")


def _now():
    return datetime.datetime.now().isoformat(timespec="seconds")


def _atomic_write_text(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        f.write(text)
    os.replace(tmp, path)


def save_json(obj, name, verbose=True):
    path = os.path.join(RESULT_DIR, name)
    _atomic_write_text(path, json.dumps(obj, indent=2, default=float))
    if verbose:
        print(f"[Drive] saved {path}")
    return path


def save_csv(df, name, verbose=True, **kwargs):
    path = os.path.join(RESULT_DIR, name)
    _atomic_write_text(path, df.to_csv(**kwargs))
    if verbose:
        print(f"[Drive] saved {path}")
    return path


def save_fig(fig, name, dpi=150, verbose=True):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    if verbose:
        print(f"[Drive] saved {path}")
    return path


def update_summary(section, values):
    """Merge `values` into results/summary.json under `section` (keeps the other sections)."""
    summary = {}
    if os.path.exists(SUMMARY_PATH):
        with open(SUMMARY_PATH) as f:
            summary = json.load(f)
    summary[section] = dict(values, updated=_now())
    _atomic_write_text(SUMMARY_PATH, json.dumps(summary, indent=2, default=float))
    print(f"[Drive] summary.json <- {section}")


save_json(asdict(cfg), "config.json")
print("Results directory:", RESULT_DIR)
print(json.dumps(asdict(cfg), indent=2, default=str))


## 3. Get the ShanghaiTech dataset

Pick **one** of the cells below. The Kaggle route (`tthien/shanghaitech`) is the easiest inside
Colab — it needs your `kaggle.json` API token (Kaggle account → *Settings* → *Create New Token*).

Set `cfg.part` in Section 2 to `"A"`, `"B"`, or `"both"`. The `tthien/shanghaitech` Kaggle dataset already contains both `part_A` and `part_B`, so `"both"` needs no extra download -- it just uses more of what's already there.


In [ ]:
# --- Option A: Kaggle (recommended) ------------------------------------------------
# Upload your kaggle.json first:
# from google.colab import files
# files.upload()  # select kaggle.json
# os.makedirs("/root/.kaggle", exist_ok=True)
# shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
# os.chmod("/root/.kaggle/kaggle.json", 0o600)

DATASET_PATH = None
try:
    import kagglehub
    DATASET_PATH = kagglehub.dataset_download("tthien/shanghaitech")
    print("Downloaded to:", DATASET_PATH)
except Exception as e:
    print("Kaggle download unavailable/failed:", e)
    print("Use Option B (Google Drive) or Option C (manual upload) below instead.")


Using Colab cache for faster access to the 'shanghaitech' dataset.
Downloaded to: /kaggle/input/shanghaitech


In [ ]:
# --- Option B: Google Drive (if you already have a ShanghaiTech.zip in Drive) ------
# from google.colab import drive
# drive.mount('/content/drive')
# zip_path = "/content/drive/MyDrive/ShanghaiTech.zip"
# with zipfile.ZipFile(zip_path, 'r') as zf:
#     zf.extractall("/content/ShanghaiTech")
# DATASET_PATH = "/content/ShanghaiTech"


In [ ]:
# --- Option C: manual upload of a zip file -----------------------------------------
# from google.colab import files
# uploaded = files.upload()  # select your ShanghaiTech.zip
# zip_name = list(uploaded.keys())[0]
# with zipfile.ZipFile(zip_name, 'r') as zf:
#     zf.extractall("/content/ShanghaiTech")
# DATASET_PATH = "/content/ShanghaiTech"


In [ ]:
def find_dataset_root(search_roots):
    candidates = [r for r in search_roots if r and os.path.isdir(r)]
    for root in candidates:
        for dirpath, dirnames, _ in os.walk(root):
            if "part_A" in dirnames or "part_B" in dirnames:
                return dirpath
    # fall back: look for any directory that itself is named part_A/part_B
    for root in candidates:
        for dirpath, dirnames, _ in os.walk(root):
            if os.path.basename(dirpath) in ("part_A", "part_B"):
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not locate part_A / part_B under: " + ", ".join(candidates)
        + ". Run one of the download options above first."
    )


def resolve_part_dirs(dataset_root, part):
    part_dir = os.path.join(dataset_root, f"part_{part}")
    train_img_dir = os.path.join(part_dir, "train_data", "images")
    train_gt_dir = os.path.join(part_dir, "train_data", "ground-truth")
    test_img_dir = os.path.join(part_dir, "test_data", "images")
    test_gt_dir = os.path.join(part_dir, "test_data", "ground-truth")
    for d in (train_img_dir, train_gt_dir, test_img_dir, test_gt_dir):
        assert os.path.isdir(d), f"Missing expected directory: {d}"
    return train_img_dir, train_gt_dir, test_img_dir, test_gt_dir


search_roots = list(cfg.dataset_search_roots)
if DATASET_PATH:
    search_roots = [DATASET_PATH] + search_roots

DATASET_ROOT = find_dataset_root(search_roots)
print("Dataset root:", DATASET_ROOT)

active_parts = ["A", "B"] if cfg.part == "both" else [cfg.part]

# Per-part image lists and ground-truth dirs -- kept separate (rather than merged into one
# flat list) because Part A uses geometry-adaptive Gaussians and Part B uses a fixed sigma
# (see precompute_density_maps in Section 4), and each part's images live in their own
# density-cache directory.
train_image_paths_by_part, test_image_paths_by_part = {}, {}
train_gt_dir_by_part, test_gt_dir_by_part = {}, {}
for p in active_parts:
    train_img_dir, train_gt_dir, test_img_dir, test_gt_dir = resolve_part_dirs(DATASET_ROOT, p)
    train_image_paths_by_part[p] = sorted(glob.glob(os.path.join(train_img_dir, "*.jpg")))
    test_image_paths_by_part[p] = sorted(glob.glob(os.path.join(test_img_dir, "*.jpg")))
    train_gt_dir_by_part[p] = train_gt_dir
    test_gt_dir_by_part[p] = test_gt_dir
    print(f"part_{p}: {len(train_image_paths_by_part[p])} train images, "
          f"{len(test_image_paths_by_part[p])} test images")

if cfg.part == "both":
    print(
        "\nNote: Part A (dense, web-crawled, avg ~501 people/image, variable resolution) and "
        "Part B (sparser, fixed surveillance camera, avg ~123 people/image) are different "
        "domains -- the literature always reports them as separate benchmarks, never pooled. "
        "Combining them gives the model roughly 2x the images and more viewpoint diversity, at "
        "the cost of a more heterogeneous training signal. Section 11 reports test MAE/RMSE "
        "per part as well as combined, so you can check whether either part is hurt by mixing."
    )


Dataset root: /kaggle/input/shanghaitech/ShanghaiTech
part_A: 300 train images, 182 test images


## 4. Ground-truth parsing & geometry-adaptive density maps

ShanghaiTech ships point annotations (one dot per head) in `.mat` files. We convert each
image's point set into a continuous **density map** by splashing a Gaussian at every head
location. Following MCNN/CSRNet, the Gaussian spread (`sigma`) is **geometry-adaptive**: it is
derived from the average distance to each point's nearest neighbours, so dense clusters get
tight kernels and sparse regions get wider ones.

To keep this fast, each Gaussian is rendered into a small local window (sized to its own
`sigma`) rather than blurring the entire image per point — this is itself a memory/CPU
optimisation for a step that would otherwise dominate preprocessing time on the crowded
Part A images.

The resulting maps are cached to disk as `.npy` files so this expensive step runs only once.


In [ ]:
def load_gt_points(mat_path):
    mat = sio.loadmat(mat_path)
    points = mat["image_info"][0, 0][0, 0][0]
    return points.astype(np.float32)


def _gaussian_kernel_2d(sigma):
    radius = max(1, int(3 * sigma))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    kernel /= kernel.sum() + 1e-12
    return kernel, radius


def generate_density_map(img_hw, points, adaptive=True, k=3, beta=0.3,
                          fixed_sigma=15.0, max_sigma=20.0):
    h, w = img_hw
    density = np.zeros((h, w), dtype=np.float32)
    n = points.shape[0]
    if n == 0:
        return density

    if adaptive and n > k:
        tree = KDTree(points)
        dists, _ = tree.query(points, k=k + 1)
        sigmas = dists[:, 1:].mean(axis=1) * beta
        sigmas = np.clip(sigmas, 1.0, max_sigma)
    else:
        sigmas = np.full(n, fixed_sigma, dtype=np.float32)

    for i in range(n):
        x = int(round(float(points[i, 0])))
        y = int(round(float(points[i, 1])))
        if x < 0 or x >= w or y < 0 or y >= h:
            continue
        kernel, radius = _gaussian_kernel_2d(sigmas[i])
        x0, x1 = x - radius, x + radius + 1
        y0, y1 = y - radius, y + radius + 1
        kx0, kx1 = max(0, -x0), kernel.shape[1] - max(0, x1 - w)
        ky0, ky1 = max(0, -y0), kernel.shape[0] - max(0, y1 - h)
        cx0, cx1 = max(0, x0), min(w, x1)
        cy0, cy1 = max(0, y0), min(h, y1)
        density[cy0:cy1, cx0:cx1] += kernel[ky0:ky1, kx0:kx1]
    return density


def precompute_density_maps(image_paths, gt_dir, out_dir, part="A"):
    os.makedirs(out_dir, exist_ok=True)
    adaptive = (part == "A")
    for img_path in tqdm(image_paths, desc=f"Density maps -> {out_dir}"):
        fname = os.path.splitext(os.path.basename(img_path))[0]
        out_path = os.path.join(out_dir, fname + ".npy")
        if os.path.exists(out_path):
            continue
        gt_path = os.path.join(gt_dir, "GT_" + fname + ".mat")
        points = load_gt_points(gt_path)
        with Image.open(img_path) as im:
            w, h = im.size
        density = generate_density_map((h, w), points, adaptive=adaptive)
        np.save(out_path, density.astype(np.float32))


def load_density_map(density_dir, fname):
    return np.load(os.path.join(density_dir, fname + ".npy"))


In [ ]:
train_density_dir_by_part, test_density_dir_by_part = {}, {}
for p in active_parts:
    train_density_dir_by_part[p] = os.path.join(cfg.density_cache_root, f"part_{p}_train")
    test_density_dir_by_part[p] = os.path.join(cfg.density_cache_root, f"part_{p}_test")
    precompute_density_maps(train_image_paths_by_part[p], train_gt_dir_by_part[p],
                             train_density_dir_by_part[p], part=p)
    precompute_density_maps(test_image_paths_by_part[p], test_gt_dir_by_part[p],
                             test_density_dir_by_part[p], part=p)
    print(f"part_{p} density maps cached at:", train_density_dir_by_part[p],
          "and", test_density_dir_by_part[p])


Density maps -> /content/density_cache/part_A_train:   0%|          | 0/300 [00:00<?, ?it/s]

Density maps -> /content/density_cache/part_A_test:   0%|          | 0/182 [00:00<?, ?it/s]

part_A density maps cached at: /content/density_cache/part_A_train and /content/density_cache/part_A_test


## 5. Dataset & train/validation split

- **Training** samples are randomly cropped to `patch_size x patch_size` (padding up first if an
  image is smaller) plus a random horizontal flip — classic patch-based augmentation that also
  bounds the memory needed per training step regardless of the source image resolution.
- Training images are also **randomly rescaled** (count-preserving), occasionally converted to
  grayscale, and each image yields `crops_per_image` random crops per epoch.
- **Validation/test** samples keep their full resolution and are only padded (by < 8 px) up to a
  multiple of `output_stride`. *They are no longer reflect-padded up to a multiple of
  `patch_size`*: the reflected strip contained mirrored copies of real people while its GT density
  was zero, so every such head was counted as an error. Tiling now happens inside `tiled_predict`.
- The GT density map is downsampled (via **sum-pooling**, which preserves the total count) to
  match the network's output stride, keeping the target tensor — and the loss computation — an
  order of magnitude smaller in memory than a full-resolution target.
- The official ShanghaiTech *training* split is further divided into train/validation here; the
  official *test* split is kept untouched as a final held-out test set.
- With `cfg.part = "both"`, the split above is done **per part first, then combined** (`ConcatDataset`) -- so both Part A and Part B are represented proportionally in train, validation, and test, rather than risking one part being under- or over-sampled.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

_to_tensor_norm = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def pad_to_min_size(img_np, density_np, min_h, min_w):
    h, w = img_np.shape[:2]
    pad_h, pad_w = max(0, min_h - h), max(0, min_w - w)
    if pad_h or pad_w:
        img_np = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
        density_np = np.pad(density_np, ((0, pad_h), (0, pad_w)), mode="constant")
    return img_np, density_np


def pad_to_multiple(img_np, density_np, multiple):
    h, w = img_np.shape[:2]
    new_h = int(np.ceil(h / multiple) * multiple)
    new_w = int(np.ceil(w / multiple) * multiple)
    pad_h, pad_w = new_h - h, new_w - w
    if pad_h or pad_w:
        img_np = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
        density_np = np.pad(density_np, ((0, pad_h), (0, pad_w)), mode="constant")
    return img_np, density_np


def random_crop(img_np, density_np, patch_size):
    h, w = img_np.shape[:2]
    top = random.randint(0, h - patch_size)
    left = random.randint(0, w - patch_size)
    img_c = img_np[top:top + patch_size, left:left + patch_size, :]
    density_c = density_np[top:top + patch_size, left:left + patch_size]
    return img_c, density_c


def sum_pool(density, stride):
    # density: (B, 1, H, W) -> (B, 1, H/stride, W/stride), sum-preserving
    pooled = F.avg_pool2d(density, kernel_size=stride, stride=stride)
    return pooled * (stride * stride)


def random_rescale(img_np, density_np, scale_range):
    # Count-preserving rescale: bilinear-resize the density map, then renormalise it so it still
    # sums to the original head count.
    s = random.uniform(*scale_range)
    h, w = img_np.shape[:2]
    new_h, new_w = max(1, int(round(h * s))), max(1, int(round(w * s)))
    if (new_h, new_w) == (h, w):
        return img_np, density_np
    count = float(density_np.sum())
    img_np = cv2.resize(img_np, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    density_np = cv2.resize(density_np, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    new_sum = float(density_np.sum())
    if new_sum > 0:
        density_np = density_np * (count / new_sum)
    return img_np, density_np.astype(np.float32)


class ShanghaiTechDataset(Dataset):
    def __init__(self, image_paths, density_dir, patch_size, output_stride,
                 density_scale, train=True, crops_per_image=1, scale_range=None,
                 gray_prob=0.0):
        self.image_paths = image_paths
        self.density_dir = density_dir
        self.patch_size = patch_size
        self.output_stride = output_stride
        self.density_scale = density_scale
        self.train = train
        self.crops_per_image = crops_per_image if train else 1
        self.scale_range = scale_range
        self.gray_prob = gray_prob

    def __len__(self):
        return len(self.image_paths) * self.crops_per_image

    def __getitem__(self, idx):
        img_path = self.image_paths[idx % len(self.image_paths)]
        fname = os.path.splitext(os.path.basename(img_path))[0]
        img = np.array(Image.open(img_path).convert("RGB"), dtype=np.uint8)
        density = load_density_map(self.density_dir, fname).astype(np.float32)

        if self.train:
            if self.scale_range is not None:
                img, density = random_rescale(img, density, self.scale_range)
            img, density = pad_to_min_size(img, density, self.patch_size, self.patch_size)
            img, density = random_crop(img, density, self.patch_size)
            if random.random() < 0.5:
                img = np.ascontiguousarray(img[:, ::-1, :])
                density = np.ascontiguousarray(density[:, ::-1])
            if random.random() < self.gray_prob:
                img = np.repeat(np.asarray(Image.fromarray(img).convert("L"))[..., None], 3, axis=2)
        else:
            # Only pad to a multiple of output_stride (< 8 px) so the sum-pooled GT loses no heads
            # at the border. Do NOT pad to a multiple of patch_size here: that reflect-padding
            # duplicated real people into a region whose GT density is zero, inflating the error.
            img, density = pad_to_multiple(img, density, self.output_stride)

        img_tensor = _to_tensor_norm(img)
        density_tensor = torch.from_numpy(density.copy()).unsqueeze(0).unsqueeze(0).float()
        density_tensor = density_tensor * self.density_scale
        density_ds = sum_pool(density_tensor, self.output_stride).squeeze(0)
        raw_count = torch.tensor(float(density.sum()), dtype=torch.float32)
        return img_tensor, density_ds, raw_count


In [ ]:
def stratified_train_val_split(image_paths, val_split, seed=42):
    idx = list(range(len(image_paths)))
    random.Random(seed).shuffle(idx)
    n_val = max(1, int(len(idx) * val_split))
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return [image_paths[i] for i in train_idx], [image_paths[i] for i in val_idx]


train_datasets, val_datasets, test_datasets_by_part = [], [], {}
for p in active_parts:
    part_train_paths, part_val_paths = stratified_train_val_split(
        train_image_paths_by_part[p], cfg.val_split)
    train_datasets.append(ShanghaiTechDataset(
        part_train_paths, train_density_dir_by_part[p], cfg.patch_size,
        cfg.output_stride, cfg.density_scale, train=True,
        crops_per_image=cfg.crops_per_image, scale_range=cfg.scale_aug,
        gray_prob=cfg.gray_aug_prob))
    val_datasets.append(ShanghaiTechDataset(
        part_val_paths, train_density_dir_by_part[p], cfg.patch_size,
        cfg.output_stride, cfg.density_scale, train=False))
    test_datasets_by_part[p] = ShanghaiTechDataset(
        test_image_paths_by_part[p], test_density_dir_by_part[p], cfg.patch_size,
        cfg.output_stride, cfg.density_scale, train=False)
    print(f"part_{p}: {len(part_train_paths)} train (x{cfg.crops_per_image} crops/epoch) / {len(part_val_paths)} val "
          f"/ {len(test_datasets_by_part[p])} test")

# A single part just falls back to its own dataset; "both" concatenates across parts so the
# rest of the notebook (training loop, diagnostics, visualisation) needs no further changes --
# ConcatDataset supports the same __len__/__getitem__ interface as a plain Dataset.
train_dataset = train_datasets[0] if len(train_datasets) == 1 else ConcatDataset(train_datasets)
val_dataset = val_datasets[0] if len(val_datasets) == 1 else ConcatDataset(val_datasets)
test_dataset = (next(iter(test_datasets_by_part.values())) if len(test_datasets_by_part) == 1
                else ConcatDataset(list(test_datasets_by_part.values())))

print(f"\ntotal: train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True,
                          persistent_workers=cfg.num_workers > 0, drop_last=True)
# Validation/test images keep their native (padded) resolution, which varies per sample,
# so they are loaded one at a time and processed with tiled sliding-window inference.
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)
# Kept separately too so Section 11 can report per-part test metrics alongside the combined one.


def dataset_image_path(dataset, idx):
    """Image file behind dataset[idx], for a ShanghaiTechDataset or a ConcatDataset of them."""
    if isinstance(dataset, ConcatDataset):
        prev = 0
        for sub_dataset, cum in zip(dataset.datasets, dataset.cumulative_sizes):
            if idx < cum:
                return sub_dataset.image_paths[idx - prev]
            prev = cum
    return dataset.image_paths[idx % len(dataset.image_paths)]


save_json({p: {"val": [os.path.basename(x) for x in ds.image_paths]}
           for p, ds in zip(active_parts, val_datasets)}, "data_split.json")
test_loaders_by_part = {
    p: DataLoader(ds, batch_size=1, shuffle=False, num_workers=cfg.num_workers, pin_memory=True)
    for p, ds in test_datasets_by_part.items()
}


## 6. Model

### 6.1 Building blocks

Positional encoding, dilated pyramid pooling, spatial transformer and graph reasoning (used by the
`"hybrid"` model), plus the two blocks of the lightweight model:

- **`AdaptiveRFSelect`**: the adaptive receptive-field selection idea behind LigMSANet. Several
  parallel *depthwise* 3x3 branches with different dilations (receptive fields 3, 5, 7 and 11 px at
  stride 8) see the same features. Instead of summing or concatenating them, as fixed multi-column
  and ASPP designs do, the block *learns how much of each receptive field to use*. The weights are a
  softmax over branches that combines a **per-channel** term (selective-kernel style, from global
  context) and a **per-pixel** term (from local content). A far-away region of tiny heads can
  pick the small receptive fields while a nearby region of large heads picks the large ones,
  inside the same image. Depthwise branches keep each block to about 15k parameters.
- **`TinyAttention`**: ECA channel attention (a 3-tap 1D convolution over the pooled channel
  descriptor) followed by a CBAM-style spatial gate (one 7x7 conv on the channel mean/max maps):
  about 100 parameters in total. It suppresses background clutter (sky, buildings, foliage)
  that otherwise leaks false density.


In [ ]:
class PositionalEncoding2D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        channels = int(np.ceil(channels / 4) * 2)
        self.channels = channels
        inv_freq = 1.0 / (10000 ** (torch.arange(0, channels, 2).float() / channels))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, x):
        b, c, h, w = x.shape
        pos_h = torch.arange(h, device=x.device, dtype=self.inv_freq.dtype)
        pos_w = torch.arange(w, device=x.device, dtype=self.inv_freq.dtype)
        sin_inp_h = torch.einsum("i,j->ij", pos_h, self.inv_freq)
        sin_inp_w = torch.einsum("i,j->ij", pos_w, self.inv_freq)
        emb_h = torch.cat([sin_inp_h.sin(), sin_inp_h.cos()], dim=-1)  # (H, channels)
        emb_w = torch.cat([sin_inp_w.sin(), sin_inp_w.cos()], dim=-1)  # (W, channels)
        emb = torch.zeros(2 * self.channels, h, w, device=x.device, dtype=x.dtype)
        emb[:self.channels] = emb_h.t().unsqueeze(2).expand(-1, -1, w)
        emb[self.channels:2 * self.channels] = emb_w.t().unsqueeze(1).expand(-1, h, -1)
        emb = emb[:c].unsqueeze(0)
        return x + emb


class DilatedPyramidPooling(nn.Module):
    """ASPP-style module: parallel dilated-conv branches at several rates plus a
    global-pooling branch, fused together -- covers both the 'dilated convolutions'
    and 'pyramid structure' multi-scale-context requirements in one block."""

    def __init__(self, in_channels, out_channels, dilations=(1, 2, 3, 4)):
        super().__init__()
        branch_ch = out_channels // (len(dilations) + 1)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, branch_ch, kernel_size=3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(branch_ch),
                nn.ReLU(inplace=True),
            ) for d in dilations
        ])
        self.global_branch = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, branch_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(branch_ch),
            nn.ReLU(inplace=True),
        )
        total_ch = branch_ch * (len(dilations) + 1)
        self.fuse = nn.Sequential(
            nn.Conv2d(total_ch, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        h, w = x.shape[-2:]
        feats = [b(x) for b in self.branches]
        g = self.global_branch(x)
        g = F.interpolate(g, size=(h, w), mode="bilinear", align_corners=False)
        feats.append(g)
        return self.fuse(torch.cat(feats, dim=1))


class SpatialTransformer(nn.Module):
    """Learns a per-image affine warp so downstream layers see a perspective/scale
    -normalised feature map -- useful because crowd scenes are shot from many angles
    and distances.

    [PERF FIX] The warped features are added back as a *residual* (through a zero-initialised
    1x1 conv) instead of replacing the input. The predicted density map must stay pixel-aligned
    with the ground truth; when the warp replaced the features outright, any learned zoom/shift
    moved the predicted heads away from their GT locations (and `padding_mode="border"`
    smeared edge features across the vacated area), which hurt both MSE and the count. The
    residual form keeps an aligned identity path while still letting the STN contribute."""

    def __init__(self, in_channels):
        super().__init__()
        self.loc_net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(4),
        )
        self.fc_loc = nn.Sequential(
            nn.Linear(16 * 4 * 4, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 6),
        )
        self.fc_loc[-1].weight.data.zero_()
        self.fc_loc[-1].bias.data.copy_(torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float32))
        self.proj = nn.Conv2d(in_channels, in_channels, kernel_size=1)
        nn.init.zeros_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

    def forward(self, x):
        xs = self.loc_net(x)
        xs = xs.reshape(xs.size(0), -1)
        theta = self.fc_loc(xs).view(-1, 2, 3)
        # affine_grid/grid_sample are done in float32 for accurate sampling coordinates
        grid = F.affine_grid(theta.float(), x.size(), align_corners=False)
        warped = F.grid_sample(x.float(), grid, align_corners=False, padding_mode="zeros").to(x.dtype)
        return x + self.proj(warped)


class AdaptiveRFSelect(nn.Module):
    """Adaptive receptive-field selection (LigMSANet-style, selective-kernel mechanism).

    K depthwise-separable branches with different dilations; their outputs are fused with weights
    softmax_k(channel_logit_k[c] + spatial_logit_k[x, y]), i.e. the receptive field is chosen per
    channel AND per pixel. Residual, so stacking several blocks is stable."""

    def __init__(self, channels, dilations=(1, 2, 3, 5), reduction=4):
        super().__init__()
        self.k = len(dilations)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=3, padding=d, dilation=d,
                          groups=channels, bias=False),
                nn.BatchNorm2d(channels),
                nn.ReLU(inplace=True),
            ) for d in dilations
        ])
        hidden = max(channels // reduction, 8)
        self.squeeze = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, hidden, kernel_size=1), nn.ReLU(inplace=True),
        )
        self.channel_logits = nn.Conv2d(hidden, channels * self.k, kernel_size=1)
        self.spatial_logits = nn.Conv2d(channels, self.k, kernel_size=3, padding=1)
        self.project = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=1, bias=False), nn.BatchNorm2d(channels),
        )
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        b, c, h, w = x.shape
        feats = torch.stack([branch(x) for branch in self.branches], dim=1)   # (B, K, C, H, W)
        u = feats.sum(dim=1)
        ch = self.channel_logits(self.squeeze(u)).view(b, self.k, c, 1, 1)     # per channel
        sp = self.spatial_logits(u).view(b, self.k, 1, h, w)                   # per pixel
        weights = torch.softmax(ch + sp, dim=1)
        selected = (weights * feats).sum(dim=1)
        return self.act(x + self.project(selected))


class TinyAttention(nn.Module):
    """ECA channel attention + CBAM-style spatial attention, ~100 parameters."""

    def __init__(self, eca_kernel=3, spatial_kernel=7):
        super().__init__()
        self.eca = nn.Conv1d(1, 1, kernel_size=eca_kernel, padding=eca_kernel // 2, bias=False)
        self.spatial = nn.Conv2d(2, 1, kernel_size=spatial_kernel, padding=spatial_kernel // 2,
                                 bias=False)

    def forward(self, x, return_maps=False):
        y = x.mean(dim=(2, 3)).unsqueeze(1)                          # (B, 1, C)
        ca = torch.sigmoid(self.eca(y))                              # channel weights (B, 1, C)
        x = x * ca.transpose(1, 2).unsqueeze(-1)                     # (B, C, 1, 1)
        s = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        sa = torch.sigmoid(self.spatial(s))                          # spatial attention (B, 1, H, W)
        if return_maps:  # used by the XAI analysis (Section 22)
            return x * sa, ca.squeeze(1), sa
        return x * sa


class ReliabilityGate(nn.Module):
    """Per-pixel reliability gate r(x) in [0, 1] applied after the attention (the third part of
    the "DAA" block in the RMSC-style study, Section 18). Only used when daa="full"; the main model
    of this notebook does not have it. The last conv starts at zero weight and bias 3, so the gate
    starts almost fully open (r = 0.95) and learns where features are unreliable."""

    def __init__(self, channels, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(hidden, 1, kernel_size=1),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.constant_(self.net[-1].bias, 3.0)

    def forward(self, x, return_map=False):
        r = torch.sigmoid(self.net(x))
        return (x * r, r) if return_map else x * r


class TransformerContext(nn.Module):
    """Global self-attention over the (coarse, stride-8) spatial grid so every location
    can attend to crowd context anywhere in the patch."""

    def __init__(self, channels, num_layers=2, num_heads=4, ff_dim=512, dropout=0.1):
        super().__init__()
        self.pos_enc = PositionalEncoding2D(channels)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=channels, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        b, c, h, w = x.shape
        tokens = self.pos_enc(x).flatten(2).transpose(1, 2)  # (B, H*W, C)
        out = self.transformer(tokens)
        return out.transpose(1, 2).reshape(b, c, h, w)


class GraphOcclusionModule(nn.Module):
    """Graph-based occlusion handling: pools the feature map to a coarse grid of
    'region' nodes, builds a soft top-k similarity graph between them, and propagates
    messages so a heavily-occluded region can borrow evidence from visually-similar,
    less-occluded regions elsewhere in the scene."""

    def __init__(self, channels, grid_size=16, k=8):
        super().__init__()
        self.grid_size = grid_size
        self.k = k
        self.pool = nn.AdaptiveAvgPool2d(grid_size)
        self.node_transform = nn.Linear(channels, channels)
        self.message_transform = nn.Linear(channels, channels)
        self.update = nn.Sequential(nn.Linear(channels * 2, channels), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(
            nn.Conv2d(channels * 2, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        b, c, h, w = x.shape
        g = self.grid_size
        nodes = self.pool(x)
        nodes_flat = nodes.flatten(2).transpose(1, 2)  # (B, N, C), N = g*g

        q = self.node_transform(nodes_flat)
        q_norm = F.normalize(q, dim=-1)
        sim = torch.bmm(q_norm, q_norm.transpose(1, 2))  # (B, N, N)

        topk = min(self.k, sim.size(-1))
        topk_val, topk_idx = sim.topk(topk, dim=-1)
        mask = torch.full_like(sim, float("-inf"))
        mask.scatter_(-1, topk_idx, topk_val)
        adj = F.softmax(mask, dim=-1)

        messages = self.message_transform(nodes_flat)
        agg = torch.bmm(adj, messages)
        updated = self.update(torch.cat([nodes_flat, agg], dim=-1))
        updated = updated.transpose(1, 2).reshape(b, c, g, g)

        updated_up = F.interpolate(updated, size=(h, w), mode="bilinear", align_corners=False)
        return self.fuse(torch.cat([x, updated_up], dim=1))


### 6.2 Putting it together

**`"hybrid"` model (original):**

A VGG16 backbone contributes two feature depths (`conv3_3` at stride 4 and `conv4_3` at
stride 8); they are fused into one multi-scale feature map before entering the
dilated-pyramid → spatial-transformer → transformer → graph-reasoning pipeline and a small
convolutional decoder head that regresses the density map.

### 6.3 `"light"` model: `LightARFSCounter` (default, < 1M parameters)

```
image ─► MobileNetV2 (ImageNet) ─┬─ stride 8  (32 ch) ─ 1x1 ─┐
                                 ├─ stride 16 (96 ch) ─ 1x1 ─┼─ upsample + add ─► ARFS x3 ─► TinyAttention ─► decoder ─► density (stride 8)
                                 └─ stride 32 (160 ch) ─ 1x1 ┘
```

The truncated MobileNetV2 is about 0.70M of the parameters and the whole head is about 0.1M. The
three backbone depths give a top-down multi-scale pyramid; the ARFS blocks then choose the
receptive field per channel and per pixel, and the attention gate suppresses background. The
output stride is 8, like the hybrid model, so the data pipeline, tiled inference, losses and
evaluation are unchanged.


In [ ]:
class HybridCNNTransformerCrowdCounter(nn.Module):  # model_type = "hybrid"
    def __init__(self, pretrained=True, freeze_stage1=True, transformer_layers=2,
                 transformer_heads=4, transformer_ff_dim=512, graph_grid_size=16,
                 graph_k=8, use_grad_checkpoint=False):
        super().__init__()
        weights = tv_models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None
        vgg_features = tv_models.vgg16(weights=weights).features

        self.stage3 = vgg_features[:16]   # -> stride 4,  256 channels (conv3_3 + relu)
        self.stage4 = vgg_features[16:23]  # -> stride 8,  512 channels (pool3 + conv4 block)

        if freeze_stage1:
            for p in vgg_features[:10].parameters():  # conv1 + conv2 blocks
                p.requires_grad_(False)

        self.lateral3 = nn.Conv2d(256, 256, kernel_size=1)
        self.down3 = nn.Conv2d(256, 256, kernel_size=3, stride=2, padding=1)  # stride4 -> stride8
        self.lateral4 = nn.Conv2d(512, 256, kernel_size=1)
        self.fuse_ms = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        self.dppm = DilatedPyramidPooling(256, 256)
        self.stn = SpatialTransformer(256)
        self.transformer = TransformerContext(256, transformer_layers, transformer_heads,
                                               transformer_ff_dim)
        self.graph = GraphOcclusionModule(256, graph_grid_size, graph_k)

        self.decoder = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1),
            nn.ReLU(inplace=True),  # density values must be non-negative
        )

        self.use_grad_checkpoint = use_grad_checkpoint

    def backbone_parameters(self):
        return [p for stage in (self.stage3, self.stage4) for p in stage.parameters()]

    def _maybe_checkpoint(self, module, x):
        if self.use_grad_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(module, x, use_reentrant=False)
        return module(x)

    def forward(self, x):
        # Only the VGG backbone (the bulk of the FLOPs) runs under mixed precision. Everything after
        # it -- fusion, DPPM, STN, transformer, graph module, decoder -- runs in float32: it works on
        # a small stride-8 map so it is cheap, and it contains the numerically delicate parts
        # (attention softmax, affine sampling, top-k graph softmax, density regression) that
        # overflow/NaN or lose precision in float16/bfloat16.
        f3 = self._maybe_checkpoint(self.stage3, x)
        f4 = self._maybe_checkpoint(self.stage4, f3)

        with torch.autocast(device_type=x.device.type, enabled=False):
            f3, f4 = f3.float(), f4.float()
            f3_ds = self.down3(self.lateral3(f3))
            f4_p = self.lateral4(f4)
            fused = self.fuse_ms(torch.cat([f3_ds, f4_p], dim=1))

            x = self.dppm(fused)
            x = self.stn(x)
            x = self._maybe_checkpoint(self.transformer, x)
            x = self.graph(x)
            return self.decoder(x)


class LightARFSCounter(nn.Module):  # model_type = "light"
    """MobileNetV2 front-end + adaptive receptive-field selection + tiny attention (< 1M params).

    The defaults are this notebook's main model (unchanged). The extra options exist only for the
    RMSC-style study (Sections 18-26): use_msrm=False removes the ARFS blocks (the multi-scale
    receptive-field module, "MSRM"); daa="none" removes the attention, daa="full" adds the
    ReliabilityGate after it (channel -> spatial -> reliability, the "DAA" block); fusion="concat" /
    "adaptive" replaces the additive multi-scale fusion. New modules are registered after the
    decoder, so the default model has exactly the same parameters, order and state_dict keys."""

    def __init__(self, pretrained=True, freeze_stem=True, width=64, n_blocks=3,
                 dilations=(1, 2, 3, 5), use_msrm=True, daa="tiny", fusion="add"):
        super().__init__()
        assert daa in ("none", "tiny", "full"), daa
        assert fusion in ("add", "concat", "adaptive"), fusion
        weights = tv_models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
        mb = tv_models.mobilenet_v2(weights=weights).features
        self.stage8 = mb[:7]      # -> stride 8,  32 channels
        self.stage16 = mb[7:14]   # -> stride 16, 96 channels
        self.stage32 = mb[14:15]  # -> stride 32, 160 channels
        if freeze_stem:
            for p in mb[:2].parameters():
                p.requires_grad_(False)

        def lateral(cin):
            return nn.Sequential(nn.Conv2d(cin, width, kernel_size=1, bias=False),
                                 nn.BatchNorm2d(width), nn.ReLU(inplace=True))

        self.lat8, self.lat16, self.lat32 = lateral(32), lateral(96), lateral(160)
        self.blocks = (nn.Sequential(*[AdaptiveRFSelect(width, dilations) for _ in range(n_blocks)])
                       if use_msrm else nn.Identity())
        self.attention = TinyAttention() if daa != "none" else None
        self.decoder = nn.Sequential(
            nn.Conv2d(width, width // 2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(width // 2), nn.ReLU(inplace=True),
            nn.Conv2d(width // 2, 1, kernel_size=1),
            nn.ReLU(inplace=True),  # density values must be non-negative
        )
        # ---- study-only options (registered last; absent from the default model) ----
        self.fusion, self.daa = fusion, daa
        if fusion == "concat":
            self.fuse = nn.Sequential(nn.Conv2d(3 * width, width, kernel_size=1, bias=False),
                                      nn.BatchNorm2d(width), nn.ReLU(inplace=True))
        elif fusion == "adaptive":  # softmax weights per scale from global context (as RMSC-Net)
            self.fuse_weights = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                              nn.Linear(3 * width, 3))
        self.reliability = ReliabilityGate(width) if daa == "full" else None

    def backbone_parameters(self):
        return [p for stage in (self.stage8, self.stage16, self.stage32) for p in stage.parameters()]

    def _head(self, f8, f16, f32, maps=None):
        # As in the hybrid model, only the backbone runs under mixed precision; the small head
        # (softmax-based selection, attention gates, density regression) runs in float32.
        with torch.autocast(device_type=f8.device.type, enabled=False):
            f8, f16, f32 = f8.float(), f16.float(), f32.float()
            size = f8.shape[-2:]
            a8 = self.lat8(f8)
            a16 = F.interpolate(self.lat16(f16), size=size, mode="bilinear", align_corners=False)
            a32 = F.interpolate(self.lat32(f32), size=size, mode="bilinear", align_corners=False)
            if self.fusion == "add":
                fused = a8 + a16 + a32
            elif self.fusion == "concat":
                fused = self.fuse(torch.cat([a8, a16, a32], dim=1))
            else:
                w = torch.softmax(self.fuse_weights(torch.cat([a8, a16, a32], dim=1)), dim=1)
                fused = w[:, 0:1, None, None] * a8 + w[:, 1:2, None, None] * a16 + w[:, 2:3, None, None] * a32
            x = self.blocks(fused)
            ca = sa = rel = None
            if self.attention is not None:
                if maps is None:
                    x = self.attention(x)
                else:
                    x, ca, sa = self.attention(x, return_maps=True)
            if self.reliability is not None:
                x, rel = self.reliability(x, return_map=True)
            density = self.decoder(x)
            if maps is not None:
                maps.update(f8=f8, f16=f16, f32=f32, fused=fused, eca=ca, spatial=sa,
                            reliability=rel, daa=x, density=density)
            return density

    def forward(self, x):
        f8 = self.stage8(x)
        f16 = self.stage16(f8)
        f32 = self.stage32(f16)
        return self._head(f8, f16, f32)

    def forward_features(self, x):
        """Density map plus intermediate maps: encoder features f8/f16/f32, fused features, ECA
        channel weights (eca), spatial attention, reliability map and the features entering the
        decoder (daa). Used for feature consistency (Section 19) and XAI (Section 22)."""
        f8 = self.stage8(x)
        f16 = self.stage16(f8)
        f32 = self.stage32(f16)
        maps = {}
        self._head(f8, f16, f32, maps)
        return maps


def build_model(cfg):
    if cfg.model_type == "light":
        return LightARFSCounter(
            pretrained=cfg.pretrained_backbone,
            freeze_stem=cfg.freeze_backbone_stage1,
            width=cfg.light_width,
            n_blocks=cfg.light_blocks,
            dilations=cfg.light_dilations,
        )
    if cfg.model_type == "hybrid":
        return HybridCNNTransformerCrowdCounter(
            pretrained=cfg.pretrained_backbone,
            freeze_stage1=cfg.freeze_backbone_stage1,
            transformer_layers=cfg.transformer_layers,
            transformer_heads=cfg.transformer_heads,
            transformer_ff_dim=cfg.transformer_ff_dim,
            graph_grid_size=cfg.graph_grid_size,
            graph_k=cfg.graph_k,
            use_grad_checkpoint=cfg.use_grad_checkpoint,
        )
    raise ValueError(f"unknown cfg.model_type: {cfg.model_type!r}")


model = build_model(cfg).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {cfg.model_type} | total params: {n_params/1e6:.3f}M | trainable: {n_trainable/1e6:.3f}M")
if cfg.model_type == "light":
    print("  per part:", {name: f"{sum(p.numel() for p in m.parameters())/1e3:.1f}K"
                         for name, m in model.named_children()})
    assert n_params < cfg.max_params, (
        f"{n_params:,} parameters exceeds the {cfg.max_params:,} budget -- "
        "lower cfg.light_width / cfg.light_blocks")

with torch.no_grad():
    test_out = model(torch.randn(2, 3, cfg.patch_size, cfg.patch_size, device=DEVICE))
print("Output shape for a", cfg.patch_size, "x", cfg.patch_size, "input:", tuple(test_out.shape))

def get_model_for_state_dict(model):
    # torch.compile() wraps the model in an OptimizedModule whose state_dict can carry an
    # "_orig_mod." key prefix. Always saving/loading through the unwrapped module keeps
    # checkpoints compatible regardless of whether compile is enabled in a given session.
    return getattr(model, "_orig_mod", model)


if cfg.use_torch_compile and DEVICE.type == "cuda":
    # torch.compile only fails when the compiled model is first CALLED (e.g. Triton rejecting bfloat16 on
    # a T4 with newer PyTorch), which used to crash a later cell. So both the training graph (forward +
    # backward) and the inference graph are tried here, on the exact shapes used later, and the model
    # falls back to eager mode if either fails. The weights and BatchNorm statistics are restored
    # afterwards and the random streams are untouched, so this probe does not change training.
    _saved_state = copy.deepcopy(model.state_dict())
    try:
        compiled = torch.compile(model)
        with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
            probe = torch.zeros(cfg.batch_size, 3, cfg.patch_size, cfg.patch_size, device=DEVICE)
            amp_on = cfg.amp and DEVICE.type == "cuda"
            compiled.train()
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=amp_on):
                out = compiled(probe)
            out.float().sum().backward()
            compiled.eval()
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=amp_on):
                compiled(probe)
        model = compiled
        print("torch.compile enabled (training and inference graphs tested)")
    except Exception as e:
        print(f"torch.compile does not work here ({type(e).__name__}: {str(e).splitlines()[0][:200]}) "
              "-- continuing uncompiled (same results, somewhat slower).")
    model.zero_grad(set_to_none=True)
    get_model_for_state_dict(model).load_state_dict(_saved_state)
    get_model_for_state_dict(model).train()
    del _saved_state



### 6.4 Model summary (saved to Drive)

Parameters, FLOPs (counted with `torch.utils.flop_counter`, FLOPs = 2 x MACs) at a 1024x768 input
(the resolution used in the paper's comparison table) and the speed of one forward pass, measured
on the uncompiled model. The cell runs inside `torch.random.fork_rng`, so the random tensors it
draws do not shift the random stream that training uses. Saved as `results/model_summary.json`.


In [ ]:
def model_complexity_summary(net, input_hw=(768, 1024), n_runs=20):
    net = get_model_for_state_dict(net)  # uncompiled module
    h, w = input_hw
    summary = {"model_type": cfg.model_type, "total_params": n_params, "trainable_params": n_trainable,
               "input_hw": [h, w], "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"}
    was_training = net.training
    devices = [torch.cuda.current_device()] if DEVICE.type == "cuda" else []
    with torch.random.fork_rng(devices=devices), torch.no_grad():
        net.eval()
        x = torch.randn(1, 3, h, w, device=DEVICE)
        try:
            from torch.utils.flop_counter import FlopCounterMode
            counter = FlopCounterMode(display=False)
            with counter:
                net(x)
            flops = counter.get_total_flops()
            summary[f"GFLOPs_{w}x{h}"] = flops / 1e9
            summary[f"GMACs_{w}x{h}"] = flops / 2e9
        except Exception as e:
            print(f"FLOP count skipped ({e})")
        try:
            amp_on = cfg.amp and DEVICE.type == "cuda"
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=amp_on):
                for _ in range(3):  # warm-up
                    net(x)
                if DEVICE.type == "cuda":
                    torch.cuda.synchronize()
                t0 = time.perf_counter()
                for _ in range(n_runs):
                    net(x)
                if DEVICE.type == "cuda":
                    torch.cuda.synchronize()
            summary[f"fps_{w}x{h}"] = n_runs / (time.perf_counter() - t0)
        except Exception as e:  # e.g. OOM for the "hybrid" model's full-image attention
            print(f"FPS measurement skipped ({e})")
        del x
    net.train(was_training)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return summary


model_summary = model_complexity_summary(model)
for k, v in model_summary.items():
    print(f"{k:22s}: {v:,.3f}" if isinstance(v, float) else f"{k:22s}: {v}")
save_json(model_summary, "model_summary.json")
update_summary("model", model_summary)


## 7. Loss functions

Three complementary, cheap-to-compute terms combined into one objective:

1. **Pixel-wise MSE** (Euclidean loss) — the classic density-map regression loss.
2. **SSIM loss** — rewards local structural/pattern similarity between predicted and
   ground-truth density maps, which plain MSE ignores (it is a purely point-wise loss).
3. **Direct count L1 loss** — penalises the difference between the *summed* predicted and
   ground-truth counts, directly optimising the metric (MAE) crowd counting is judged on.

`L = lambda_mse * MSE + lambda_ssim * (1 - SSIM) + lambda_count * |count_pred - count_gt|`


In [ ]:
class SSIMLoss(nn.Module):
    def __init__(self, window_size=11, channel=1, sigma=1.5):
        super().__init__()
        self.window_size = window_size
        self.channel = channel
        gauss = torch.tensor(
            [math.exp(-((x - window_size // 2) ** 2) / float(2 * sigma ** 2)) for x in range(window_size)]
        )
        gauss = gauss / gauss.sum()
        _2d = gauss.unsqueeze(1) @ gauss.unsqueeze(0)
        window = _2d.float().unsqueeze(0).unsqueeze(0).expand(channel, 1, window_size, window_size)
        self.register_buffer("window", window.contiguous())

    def forward(self, pred, target):
        window = self.window.to(dtype=pred.dtype)
        c1, c2 = 0.01 ** 2, 0.03 ** 2
        pad = self.window_size // 2
        mu1 = F.conv2d(pred, window, padding=pad, groups=self.channel)
        mu2 = F.conv2d(target, window, padding=pad, groups=self.channel)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
        sigma1_sq = F.conv2d(pred * pred, window, padding=pad, groups=self.channel) - mu1_sq
        sigma2_sq = F.conv2d(target * target, window, padding=pad, groups=self.channel) - mu2_sq
        sigma12 = F.conv2d(pred * target, window, padding=pad, groups=self.channel) - mu1_mu2
        ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / \
                   ((mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2) + 1e-12)
        return 1 - ssim_map.mean()


class CrowdCountingLoss(nn.Module):
    def __init__(self, lambda_mse=1.0, lambda_ssim=0.1, lambda_count=0.02):
        super().__init__()
        self.lambda_mse = lambda_mse
        self.lambda_ssim = lambda_ssim
        self.lambda_count = lambda_count
        self.mse = nn.MSELoss()
        self.ssim = SSIMLoss()

    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        ssim_loss = self.ssim(pred, target)
        pred_count = pred.sum(dim=[1, 2, 3])
        target_count = target.sum(dim=[1, 2, 3])
        count_loss = F.l1_loss(pred_count, target_count)
        total = (self.lambda_mse * mse_loss
                 + self.lambda_ssim * ssim_loss
                 + self.lambda_count * count_loss)
        parts = {
            "mse": mse_loss.item(), "ssim": ssim_loss.item(),
            "count": count_loss.item(), "total": total.item(),
        }
        return total, parts


def compute_mae_mse(pred, target, density_scale):
    pred_count = pred.sum(dim=[1, 2, 3]) / density_scale
    target_count = target.sum(dim=[1, 2, 3]) / density_scale
    mae = (pred_count - target_count).abs()
    se = (pred_count - target_count) ** 2
    return mae, se


criterion = CrowdCountingLoss(cfg.lambda_mse, cfg.lambda_ssim, cfg.lambda_count).to(DEVICE)


## 8. Tiled (sliding-window) inference

Full-resolution test images can be large, and self-attention cost grows quadratically with
token count — feeding a whole image through the transformer at once would spike memory usage.
Instead, `tiled_predict` walks the image with **overlapping** `patch_size` tiles
(`cfg.infer_overlap`, 50% by default; the last row/column of tiles is aligned to the image edge),
runs the model on each tile independently (exactly like training), and **averages** the
overlapping tile outputs back into one density map, which is then cropped to the real image area.

[PERF FIX] The previous version (a) required the image to be reflect-padded up to a multiple of
`patch_size`, so mirrored copies of real people in the padded strip were counted against a
zero GT, and (b) used non-overlapping tiles, so every head cut by a tile border was seen by the
network with only half its context (and zero-padding artefacts at every tile edge). Both
directly inflated MAE/RMSE. This bounds peak memory to a single
patch's cost no matter how large the source image is, and is reused for validation, test-set
evaluation, and ad-hoc inference on new images.


In [ ]:
def _tile_starts(length, patch_size, step):
    if length <= patch_size:
        return [0]
    starts = list(range(0, length - patch_size + 1, step))
    if starts[-1] != length - patch_size:
        starts.append(length - patch_size)  # last tile flush with the image edge
    return starts


@torch.no_grad()
def _tiled_predict_single(model, img, patch_size, output_stride, device, amp, chunk_size,
                          overlap=0.5, bn_train_mode=False):
    # bn_train_mode is only used by the Section 10.2 diagnostic (forces BatchNorm to use each
    # chunk's own batch statistics); normal inference always runs in eval mode.
    model.train(bn_train_mode)
    img = img.to(device)
    _, _, H, W = img.shape  # H, W are multiples of output_stride

    # Images smaller than a tile are padded with zeros -- i.e. the ImageNet-mean colour after
    # normalisation, a neutral people-free region -- and the prediction is cropped back below.
    pad_h, pad_w = max(0, patch_size - H), max(0, patch_size - W)
    if pad_h or pad_w:
        img = F.pad(img, (0, pad_w, 0, pad_h), value=0.0)
    Hp, Wp = img.shape[-2:]

    step = int(patch_size * (1 - overlap)) // output_stride * output_stride
    step = max(output_stride, min(step, patch_size))
    positions = [(y, x) for y in _tile_starts(Hp, patch_size, step)
                 for x in _tile_starts(Wp, patch_size, step)]

    tile_out = patch_size // output_stride
    pred_sum = torch.zeros(1, 1, Hp // output_stride, Wp // output_stride, device=device)
    weight = torch.zeros_like(pred_sum)

    # Tiles are batched (chunk_size at a time) into single forward passes -- far better GPU
    # utilisation than one tile per call -- while capping peak memory regardless of image size.
    for start in range(0, len(positions), chunk_size):
        chunk_pos = positions[start:start + chunk_size]
        batch = torch.cat([img[:, :, y:y + patch_size, x:x + patch_size] for y, x in chunk_pos], dim=0)
        n = batch.size(0)
        if bn_train_mode:
            if n == 1:  # train-mode BatchNorm needs > 1 value per channel
                batch = batch.repeat(2, 1, 1, 1)
        elif n < chunk_size:
            # Pad the last, partial chunk to a fixed batch size: in eval mode the extra rows don't
            # affect the real ones, and a constant input shape stops torch.compile from
            # recompiling for every leftover chunk size.
            batch = torch.cat([batch, batch.new_zeros(chunk_size - n, *batch.shape[1:])], dim=0)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(amp and device.type == "cuda")):
            out = model(batch)
        out = out.float()[:n]
        for (y, x), p in zip(chunk_pos, out):
            oy, ox = y // output_stride, x // output_stride
            pred_sum[:, :, oy:oy + tile_out, ox:ox + tile_out] += p.unsqueeze(0)
            weight[:, :, oy:oy + tile_out, ox:ox + tile_out] += 1

    model.eval()
    pred = pred_sum / weight.clamp_min(1.0)
    return pred[:, :, :H // output_stride, :W // output_stride]


@torch.no_grad()
def tiled_predict(model, img, patch_size, output_stride, device, amp=True, tta=False, chunk_size=8,
                  overlap=None):
    overlap = cfg.infer_overlap if overlap is None else overlap
    pred = _tiled_predict_single(model, img, patch_size, output_stride, device, amp, chunk_size, overlap)
    if tta:
        # Average with the horizontal-flip prediction (flipped back to align) -- a cheap,
        # well-established accuracy boost at test/inference time, at 2x the compute cost.
        flipped = torch.flip(img, dims=[3])
        pred_flipped = _tiled_predict_single(model, flipped, patch_size, output_stride, device,
                                              amp, chunk_size, overlap)
        pred = (pred + torch.flip(pred_flipped, dims=[3])) / 2
    return pred


@torch.no_grad()
def predict_density(model, img, tta=False, infer_scale=1.0, infer_threshold=0.0, device=None):
    """tiled_predict, optionally on an enlarged copy of the image (Section 10.3).

    infer_scale = 1.0 (default) is exactly tiled_predict. Otherwise the image is enlarged by
    infer_scale, counted, and the density map is pooled back to the normal stride-8 grid with its
    total count preserved. With infer_threshold > 0 only images whose normal-size prediction is at
    least that many people are enlarged. Never used during training."""
    device = device or DEVICE
    img = img.to(device)
    pred = tiled_predict(model, img, cfg.patch_size, cfg.output_stride, device,
                         amp=cfg.amp, tta=tta, chunk_size=cfg.batch_size)
    if infer_scale == 1.0:
        return pred
    if infer_threshold > 0 and pred.sum().item() / cfg.density_scale < infer_threshold:
        return pred
    H, W = img.shape[-2:]
    m = cfg.output_stride
    nh, nw = int(math.ceil(H * infer_scale / m) * m), int(math.ceil(W * infer_scale / m) * m)
    big_img = F.interpolate(img, size=(nh, nw), mode="bilinear", align_corners=False)
    big = tiled_predict(model, big_img, cfg.patch_size, cfg.output_stride, device,
                        amp=cfg.amp, tta=tta, chunk_size=cfg.batch_size)
    small = F.adaptive_avg_pool2d(big, pred.shape[-2:])
    return small * (big.sum() / small.sum().clamp_min(1e-8))


## 9. Training & validation loops

Mixed precision (`torch.cuda.amp`), optional gradient accumulation, and periodic
`torch.cuda.empty_cache()` calls keep memory usage in check during training.


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, cfg, epoch):
    model.train()
    running = {"loss": 0.0, "mae": 0.0, "mse": 0.0, "ssim": 0.0, "count": 0.0}
    n_samples, n_skipped, n_consecutive_bad = 0, 0, 0
    optimizer.zero_grad()
    pbar = tqdm(loader, desc=f"Epoch {epoch} [train]", leave=False)
    for step, (imgs, densities, _) in enumerate(pbar):
        imgs = imgs.to(device, non_blocking=True)
        densities = densities.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(cfg.amp and device.type == "cuda")):
            preds = model(imgs)
        # The loss (squares/divisions inside SSIM) is computed in float32 regardless of the
        # autocast dtype used for the forward pass -- squaring already density_scale-scaled
        # values can overflow float16's ~65504 max, which otherwise silently turns into NaN.
        loss, parts = criterion(preds.float(), densities.float())

        if not torch.isfinite(loss):
            n_skipped += 1
            n_consecutive_bad += 1
            if n_skipped <= 3:
                # Say where the NaN/inf comes from instead of just that it happened.
                print(f"  [warn] non-finite loss at epoch {epoch} step {step} -- skipping batch "
                      f"(inputs finite={torch.isfinite(imgs).all().item()}, "
                      f"targets finite={torch.isfinite(densities).all().item()}, "
                      f"preds finite={torch.isfinite(preds).all().item()}, "
                      f"weights finite={all(torch.isfinite(p).all().item() for p in model.parameters())})")
            optimizer.zero_grad(set_to_none=True)
            if n_consecutive_bad >= 20:
                raise RuntimeError(
                    f"20 consecutive non-finite batches in epoch {epoch}: training is not recoverable "
                    "this way. If 'weights finite=False' above, reload a checkpoint (Section 10.1); "
                    "otherwise the forward pass itself overflows -- set cfg.amp=False to confirm.")
            continue
        n_consecutive_bad = 0

        loss_scaled = loss / cfg.grad_accum_steps
        scaler.scale(loss_scaled).backward()
        if (step + 1) % cfg.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        with torch.no_grad():
            mae, _ = compute_mae_mse(preds.float(), densities.float(), cfg.density_scale)

        bs = imgs.size(0)
        running["loss"] += parts["total"] * bs
        running["mae"] += mae.sum().item()
        running["mse"] += parts["mse"] * bs
        running["ssim"] += parts["ssim"] * bs
        running["count"] += parts["count"] * bs
        n_samples += bs
        pbar.set_postfix(loss=f"{parts['total']:.4f}", mae=f"{mae.mean().item():.2f}",
                          lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    if n_skipped:
        print(f"  Epoch {epoch}: skipped {n_skipped} non-finite batch(es).")
    torch.cuda.empty_cache()
    n_samples = max(n_samples, 1)
    return {k: v / n_samples for k, v in running.items()}


@torch.no_grad()
def validate(model, loader, criterion, device, cfg, desc="Validate", tta=False,
             infer_scale=1.0, infer_threshold=0.0):
    model.eval()
    running = {"loss": 0.0, "mae": 0.0, "se": 0.0, "mse": 0.0, "ssim": 0.0, "count": 0.0}
    n = 0
    for imgs, densities, _ in tqdm(loader, desc=desc, leave=False):
        imgs = imgs.to(device, non_blocking=True)
        densities = densities.to(device, non_blocking=True)
        # predict_density with the default infer_scale=1.0 is exactly tiled_predict
        preds = predict_density(model, imgs, tta=tta, infer_scale=infer_scale,
                                infer_threshold=infer_threshold, device=device)
        loss, parts = criterion(preds, densities)
        mae, se = compute_mae_mse(preds, densities, cfg.density_scale)

        bs = imgs.size(0)
        running["loss"] += parts["total"] * bs
        running["mae"] += mae.sum().item()
        running["se"] += se.sum().item()
        running["mse"] += parts["mse"] * bs
        running["ssim"] += parts["ssim"] * bs
        running["count"] += parts["count"] * bs
        n += bs

    torch.cuda.empty_cache()
    return {
        "loss": running["loss"] / n, "mae": running["mae"] / n,
        "rmse": math.sqrt(running["se"] / n), "mse": running["mse"] / n,
        "ssim": running["ssim"] / n, "count": running["count"] / n,
    }


## 10. Train

- **Adaptive learning rate**: `ReduceLROnPlateau` halves the LR whenever the *smoothed*
  (moving average over `cfg.val_mae_smoothing` epochs) validation MAE stalls, so the schedule adapts
  to actual training progress without being fooled by single noisy epochs.
- **Differential learning rates**: the pretrained VGG layers train at `base_lr * backbone_lr_mult`,
  the newly-initialised modules at `base_lr`.
- **Surviving GPU timeouts**: a Colab runtime that times out or disconnects is destroyed, along
  with everything in memory and on its local disk, so no code can keep running inside it. What
  this notebook does instead: every epoch the *complete* training state (weights, optimizer,
  LR scheduler, grad-scaler, loss/MAE history, best-so-far MAE and all RNG states) is written to
  **Google Drive** (`cfg.ckpt_dir`). It is written to a temp file and atomically renamed, and the
  previous copy is kept as `last_prev.pth`, so a disconnect in the middle of a save can't corrupt it.
  After a timeout, reconnect and use **Runtime → Run all**: the training cell finds `last.pth` and
  continues from the next epoch with identical optimizer/scheduler state, then the evaluation
  cells run as usual. At most one epoch of work is lost. `cfg.max_session_hours` can also stop
  training cleanly before a known session limit.
- **Bounded training**: `cfg.epochs = 150` is only the upper limit (early stopping ended the
  previous run at epoch 113). `cfg.epochs_per_session = N` additionally ends each Colab session
  after N epochs with the state saved; *Run all* continues from the next epoch.
- **Drive results**: besides the checkpoints, `results/training_history.csv`,
  `results/training_status.json` and `results/figures/training_curves.png` are rewritten every epoch.
- **Early stopping**: training ends once the 5-epoch moving average of validation MAE has not
  improved for `cfg.early_stop_patience` epochs (never before `cfg.min_epochs`). The decision is
  recomputed from the saved history, so it survives resumes, and *Run all* on a run that already
  stopped goes straight to evaluation. `cfg.epochs` is just the upper limit.
- **Best-model checkpointing**: after every epoch, the current validation MAE is compared
  against the *best* MAE seen in any previous epoch (a strictly stronger guarantee than only
  checking the immediately preceding epoch) — the checkpoint is overwritten only on improvement.
- **Visibility**: loss and MAE (the regression stand-in for "accuracy" in a counting task) are
  printed and plotted for both the training and validation sets every epoch.
- **Numerical stability**: gradients are clipped to `cfg.grad_clip_norm`, the loss is always
  computed in float32 even under mixed precision, and mixed precision prefers `bfloat16` over
  `float16` when available -- together these prevent the exploding-gradient / fp16-overflow NaNs
  that (once they occur) permanently corrupt the model's weights and `BatchNorm` running stats.
- **LR warmup**: for the first `cfg.warmup_epochs`, the LR ramps up linearly and the scheduler doesn't react yet, so its best-so-far tracking starts from a stabilized model rather than the noisiest opening epochs.
- **Per-term loss breakdown**: each epoch also prints the MSE/SSIM/count components of the loss (train and val), stored in `history`, so you can see which term is driving progress rather than only the weighted total.


In [ ]:
def build_optimizer(model):
    # [PERF FIX] Two parameter groups: pretrained backbone at a 10x lower LR than the new modules.
    # Group 0 is the head, so optimizer.param_groups[0]["lr"] (logged/plotted) is the head LR.
    m = get_model_for_state_dict(model)
    backbone_params = [p for p in m.backbone_parameters() if p.requires_grad]
    backbone_ids = {id(p) for p in backbone_params}
    head_params = [p for p in m.parameters() if p.requires_grad and id(p) not in backbone_ids]
    head_lr, backbone_lr = cfg.base_lr, cfg.base_lr * cfg.backbone_lr_mult
    return torch.optim.AdamW([
        {"params": head_params, "lr": head_lr, "group_base_lr": head_lr},
        {"params": backbone_params, "lr": backbone_lr, "group_base_lr": backbone_lr},
    ], weight_decay=cfg.weight_decay)


def build_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=cfg.lr_factor, patience=cfg.lr_patience, min_lr=cfg.lr_min
    )


optimizer = build_optimizer(model)
scheduler = build_scheduler(optimizer)
# GradScaler's loss-scaling only matters for float16 (it is a documented no-op when
# enabled=False), so bfloat16 runs simply skip it -- there is nothing for it to protect against.
scaler = torch.amp.GradScaler("cuda", enabled=(cfg.amp and DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))

HISTORY_KEYS = ["train_loss", "val_loss", "train_mae", "val_mae", "lr",
                "train_mse", "train_ssim", "train_count", "val_mse", "val_ssim", "val_count"]
history = {k: [] for k in HISTORY_KEYS}
best_val_mae = float("inf")
best_ckpt_path = os.path.join(cfg.ckpt_dir, "best_model.pth")
last_ckpt_path = os.path.join(cfg.ckpt_dir, "last.pth")
prev_ckpt_path = os.path.join(cfg.ckpt_dir, "last_prev.pth")


# ---------------------------------------------------------------- checkpoint I/O
def atomic_save(obj, path):
    # Write to a temp file, then rename over the target: a runtime that dies mid-write leaves the
    # previous checkpoint intact instead of a truncated, unloadable file.
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def save_training_state(epoch, model, optimizer, scheduler, scaler, history, best_val_mae):
    state = {
        "epoch": epoch,
        "model_type": cfg.model_type,
        "model_state": get_model_for_state_dict(model).state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "history": history,
        "best_val_mae": best_val_mae,
        "cfg": asdict(cfg),
        "rng": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        },
    }
    if os.path.exists(last_ckpt_path):
        shutil.copyfile(last_ckpt_path, prev_ckpt_path)  # keep one older good copy
    atomic_save(state, last_ckpt_path)


def load_training_state():
    for path in (last_ckpt_path, prev_ckpt_path):
        if not os.path.exists(path):
            continue
        try:
            # weights_only=False: the file also holds history/RNG state (Python/NumPy objects). It is
            # our own checkpoint, so this is safe; never use it on checkpoints from untrusted sources.
            return torch.load(path, map_location=DEVICE, weights_only=False), path
        except Exception as e:
            print(f"Could not read {path} ({e}); trying an older copy.")
    return None, None


def restore_training_state(state, model, optimizer, scheduler, scaler):
    if state.get("model_type") != cfg.model_type:
        raise RuntimeError(
            f"{last_ckpt_path} belongs to a '{state.get('model_type')}' model but cfg.model_type is "
            f"'{cfg.model_type}'. Use a different cfg.run_name for a different model.")
    get_model_for_state_dict(model).load_state_dict(state["model_state"])
    optimizer.load_state_dict(state["optimizer_state"])
    scheduler.load_state_dict(state["scheduler_state"])
    scaler.load_state_dict(state["scaler_state"])
    rng = state["rng"]
    random.setstate(rng["python"])
    np.random.set_state(rng["numpy"])
    torch.set_rng_state(rng["torch"].cpu())
    if rng["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in rng["cuda"]])
    return state["epoch"], state["history"], state["best_val_mae"]


def save_history_csv(history):
    # Rewritten on Drive after every epoch, so the curves survive a disconnect even without last.pth.
    n = len(history["val_mae"])
    df = pd.DataFrame({"epoch": list(range(1, n + 1)), **{k: history[k][:n] for k in HISTORY_KEYS}})
    save_csv(df, "training_history.csv", verbose=False, index=False)


def save_training_status(history, finished, reason=""):
    vals = history["val_mae"]
    status = {"epochs_done": len(vals), "max_epochs": cfg.epochs, "finished": finished, "reason": reason,
              "best_val_mae": float(min(vals)) if vals else None,
              "best_epoch": int(np.argmin(vals)) + 1 if vals else None,
              "last_ckpt": last_ckpt_path, "best_ckpt": best_ckpt_path, "updated": _now()}
    save_json(status, "training_status.json", verbose=False)
    return status


def plot_history(history, save_name="training_curves.png"):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

    axes[1].plot(history["train_mae"], label="train")
    axes[1].plot(history["val_mae"], label="val")
    axes[1].set_title("MAE (count error)"); axes[1].set_xlabel("epoch"); axes[1].legend()

    axes[2].plot(history["lr"])
    axes[2].set_title("Learning rate"); axes[2].set_xlabel("epoch"); axes[2].set_yscale("log")
    plt.tight_layout()
    if save_name:
        save_fig(fig, save_name, verbose=False)
    plt.show()


def set_warmup_lr(optimizer, epoch, warmup_epochs, base_lr):
    factor = min(1.0, epoch / max(1, warmup_epochs))
    for pg in optimizer.param_groups:
        pg["lr"] = pg.get("group_base_lr", base_lr) * factor


def early_stop_status(history):
    """Decide early stopping from the saved val-MAE history alone, so it needs no extra state and
    behaves identically after a resume (history is part of last.pth)."""
    vals = history["val_mae"]
    if not cfg.early_stop or len(vals) < max(cfg.min_epochs, 1):
        return False, ""
    k = max(cfg.val_mae_smoothing, 1)  # same moving average that drives ReduceLROnPlateau
    smoothed = [float(np.mean(vals[max(0, i - k + 1):i + 1])) for i in range(len(vals))]
    best_i, best = 0, smoothed[0]
    for i, v in enumerate(smoothed):
        if v < best - cfg.early_stop_min_delta:
            best_i, best = i, v
    since = len(smoothed) - 1 - best_i
    if since >= cfg.early_stop_patience:
        return True, (f"smoothed val MAE has not improved for {since} epochs "
                      f"(best {best:.2f} at epoch {best_i + 1})")
    return False, f"early stopping: {since}/{cfg.early_stop_patience} epochs without improvement"


def run_training(model, optimizer, scheduler, scaler, start_epoch, end_epoch, history, best_val_mae):
    session_start = time.time()
    for epoch in range(start_epoch, end_epoch + 1):
        in_warmup = epoch <= cfg.warmup_epochs
        if in_warmup:
            set_warmup_lr(optimizer, epoch, cfg.warmup_epochs, cfg.base_lr)

        train_stats = train_one_epoch(model, train_loader, optimizer, criterion,
                                       scaler, DEVICE, cfg, epoch)
        val_stats = validate(model, val_loader, criterion, DEVICE, cfg)

        # The scheduler only reacts once warmup is done, so its "best so far" tracking starts
        # from a stabilized model rather than the noisy first few epochs at a ramping-up LR.
        # [PERF FIX] Step on a moving average of recent val MAEs rather than the raw, very noisy
        # per-epoch value, which previously drove the LR to its floor far too early.
        if not in_warmup:
            recent = history["val_mae"][-(cfg.val_mae_smoothing - 1):] if cfg.val_mae_smoothing > 1 else []
            smoothed_val_mae = float(np.mean(recent + [val_stats["mae"]]))
            scheduler.step(smoothed_val_mae)
        current_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(train_stats["loss"])
        history["val_loss"].append(val_stats["loss"])
        history["train_mae"].append(train_stats["mae"])
        history["val_mae"].append(val_stats["mae"])
        history["lr"].append(current_lr)
        history["train_mse"].append(train_stats["mse"])
        history["train_ssim"].append(train_stats["ssim"])
        history["train_count"].append(train_stats["count"])
        history["val_mse"].append(val_stats["mse"])
        history["val_ssim"].append(val_stats["ssim"])
        history["val_count"].append(val_stats["count"])

        improved = val_stats["mae"] < best_val_mae
        if improved:
            best_val_mae = val_stats["mae"]
            atomic_save({
                "epoch": epoch, "model_type": cfg.model_type,
                "model_state": get_model_for_state_dict(model).state_dict(),
                "optimizer_state": optimizer.state_dict(), "val_mae": val_stats["mae"],
                "cfg": asdict(cfg),
            }, best_ckpt_path)
        # Full resumable state, on Drive, every cfg.ckpt_every_epochs epochs (and always at the end).
        if epoch % cfg.ckpt_every_epochs == 0 or epoch == end_epoch:
            save_training_state(epoch, model, optimizer, scheduler, scaler, history, best_val_mae)
        if cfg.snapshot_every_epochs and epoch % cfg.snapshot_every_epochs == 0:
            atomic_save({"epoch": epoch, "model_type": cfg.model_type,
                         "model_state": get_model_for_state_dict(model).state_dict(),
                         "val_mae": val_stats["mae"]},
                        os.path.join(cfg.ckpt_dir, f"snapshot_epoch{epoch:03d}.pth"))
        save_history_csv(history)
        save_training_status(history, finished=False, reason=f"in progress (epoch {epoch})")

        clear_output(wait=True)
        plot_history(history)
        flag = " ** new best, saved **" if improved else ""
        warmup_flag = " (warmup)" if in_warmup else ""
        print(f"Epoch {epoch}/{cfg.epochs} | "
              f"train_loss={train_stats['loss']:.4f} val_loss={val_stats['loss']:.4f} | "
              f"train_mae={train_stats['mae']:.2f} val_mae={val_stats['mae']:.2f} "
              f"val_rmse={val_stats['rmse']:.2f} | lr={current_lr:.2e}{warmup_flag}{flag}")
        print(f"    loss terms -- train: mse={train_stats['mse']:.3f} ssim={train_stats['ssim']:.3f} "
              f"count={train_stats['count']:.3f} | val: mse={val_stats['mse']:.3f} "
              f"ssim={val_stats['ssim']:.3f} count={val_stats['count']:.3f}")
        print(f"    state saved to {cfg.ckpt_dir} (best val MAE so far {best_val_mae:.2f})")

        stop, reason = early_stop_status(history)
        if reason:
            print("    " + reason)
        if stop and epoch < end_epoch:
            save_training_state(epoch, model, optimizer, scheduler, scaler, history, best_val_mae)
            print(f"Early stopping at epoch {epoch}: {reason}. best_model.pth (val MAE "
                  f"{best_val_mae:.2f}) is used for evaluation below.")
            break

        elapsed_h = (time.time() - session_start) / 3600
        if cfg.max_session_hours and elapsed_h >= cfg.max_session_hours and epoch < end_epoch:
            save_training_state(epoch, model, optimizer, scheduler, scaler, history, best_val_mae)
            print(f"Stopping after {elapsed_h:.2f} h (cfg.max_session_hours). State is saved -- "
                  "reconnect and Run all to continue from epoch", epoch + 1)
            break
        epochs_this_session = epoch - start_epoch + 1
        if cfg.epochs_per_session and epochs_this_session >= cfg.epochs_per_session and epoch < end_epoch:
            save_training_state(epoch, model, optimizer, scheduler, scaler, history, best_val_mae)
            print(f"Trained {epochs_this_session} epoch(s) this session (cfg.epochs_per_session). State "
                  f"is saved on Drive -- Run all to continue from epoch {epoch + 1}.")
            break
    return best_val_mae


# ---------------------------------------------------------------- train (auto-resume)
start_epoch = 1
if cfg.auto_resume:
    resume_state, resume_path = load_training_state()
    if resume_state is not None:
        done_epoch, history, best_val_mae = restore_training_state(
            resume_state, model, optimizer, scheduler, scaler)
        start_epoch = done_epoch + 1
        if os.path.exists(best_ckpt_path):  # e.g. resumed from the backup copy: never let a worse
            best_val_mae = min(best_val_mae,  # epoch overwrite a better saved best_model.pth
                               torch.load(best_ckpt_path, map_location="cpu")["val_mae"])
        print(f"Resumed from {resume_path}: {done_epoch} epoch(s) done, "
              f"best val MAE {best_val_mae:.2f}, LR {optimizer.param_groups[0]['lr']:.2e}")
        del resume_state
    else:
        print(f"No checkpoint in {cfg.ckpt_dir} -- starting a new run.")

already_stopped, stop_reason = early_stop_status(history)
if start_epoch > cfg.epochs or already_stopped:
    print("Training already finished:",
          stop_reason if already_stopped else f"all {cfg.epochs} epochs done",
          "-- set cfg.early_stop=False / raise cfg.epochs to train longer.")
    if history["val_mae"]:
        plot_history(history)
else:
    best_val_mae = run_training(model, optimizer, scheduler, scaler, start_epoch, cfg.epochs,
                                history, best_val_mae)

# ---------------------------------------------------------------- training status on Drive
stopped, stop_reason = early_stop_status(history)
TRAINING_FINISHED = stopped or len(history["val_mae"]) >= cfg.epochs
if history["val_mae"]:
    save_history_csv(history)
status = save_training_status(
    history, TRAINING_FINISHED,
    stop_reason if stopped else (f"all {cfg.epochs} epochs done" if TRAINING_FINISHED
                                 else "paused -- Run all to continue"))
if TRAINING_FINISHED:
    update_summary("training", status)
else:
    print(f"\nTraining is NOT finished ({status['epochs_done']}/{cfg.epochs} epochs). The cells below "
          "evaluate the best checkpoint so far; Run all again to continue training.")


### 10.1 Resuming after a diverged (NaN) or interrupted run

If `train_loss` / `val_loss` ever print as `nan`, the model's weights -- and, subtly, every
`BatchNorm2d` layer's running statistics, which update during the forward pass itself and are
**not** protected by `GradScaler` -- are permanently corrupted from that point on. No amount of
further training or learning-rate decay recovers them, because NaNs propagate through every
subsequent forward pass. That is why the loss can stay `nan` for the rest of a run even while the
scheduler keeps dutifully reducing the learning rate.

Two things are needed:

1. **Prevention** (already built into `train_one_epoch`/`tiled_predict` above): the loss is
   computed in float32 even under mixed precision (the SSIM term squares and sums already
   `density_scale`-scaled values, which can silently overflow float16's ~65504 max), gradients
   are clipped to `cfg.grad_clip_norm`, and mixed precision prefers `bfloat16` over `float16`
   whenever the GPU supports it (bf16 shares float32's exponent range, so it cannot overflow the
   way float16 can).
2. **Recovery**: reload the last checkpoint saved *before* the divergence. `best_model.pth` is
   only overwritten when validation MAE improves, so — unlike `last_model.pth`, which is
   overwritten every epoch including the NaN one — it is very likely still a healthy,
   pre-divergence checkpoint.

This is only for NaN recovery: an ordinary timeout or disconnect is handled automatically by
the training cell above (it resumes from `last.pth`). The cell below therefore does nothing
unless you set `RECOVER_FROM_BEST = True`, so **Runtime → Run all** after a timeout doesn't
accidentally restart training from the best checkpoint.


In [ ]:
RECOVER_FROM_BEST = False  # set True only to recover from a NaN-diverged run (see above)

if RECOVER_FROM_BEST:
    resume_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    get_model_for_state_dict(model).load_state_dict(resume_ckpt["model_state"])
    print(f"Restored weights from epoch {resume_ckpt['epoch']} (val_mae={resume_ckpt['val_mae']:.2f})")

    optimizer = build_optimizer(model)
    optimizer.load_state_dict(resume_ckpt["optimizer_state"])
    scheduler = build_scheduler(optimizer)
    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.amp and DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))

    # Drop any epochs recorded after the restored checkpoint (e.g. the NaN epoch itself) so the
    # plotted history stays consistent with the restored weights.
    resume_epoch = resume_ckpt["epoch"]
    for key in history:
        history[key] = history[key][:resume_epoch]
    best_val_mae = resume_ckpt["val_mae"]

    best_val_mae = run_training(model, optimizer, scheduler, scaler,
                                 resume_epoch + 1, cfg.epochs, history, best_val_mae)
else:
    print("Skipped (RECOVER_FROM_BEST = False).")


### 10.2 Diagnosing the train/val gap

Validation MAE (~500) sitting suspiciously close to Part A's *average* image count, while
training MAE (~75) looks fine, has more than one possible explanation -- rather than guess, this
runs the current best checkpoint on a handful of validation images twice:

1. through the normal **eval-mode BatchNorm** path (`tiled_predict`, using the running
   statistics accumulated during training) -- exactly what `validate()` uses, and
2. with BatchNorm **forced into train mode** on that same tile (using the tile's own batch
   statistics instead of the running average).

- If the **train-mode-BN error is much lower**, the running statistics haven't calibrated well
  for whole-image, eval-mode inference at `batch_size=8` -- the fix is to replace `BatchNorm2d`
  with `GroupNorm` in the custom modules, which doesn't depend on batch statistics at all.
- If **both errors are similarly large**, this is a genuine generalization gap (or a scale
  mismatch between patch-level training MAE and whole-image validation MAE) rather than a
  BatchNorm artifact, and a different fix is needed (more augmentation/regularization, or a
  less aggressively-decayed LR schedule so the model actually finishes converging).


In [ ]:
def summarize_batchnorm_stats(diag_model):
    print(f"{'layer':40s} {'mean(run_mean)':>15s} {'std(run_mean)':>15s} {'mean(run_var)':>15s} {'std(run_var)':>15s}")
    for name, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d):
            rm, rv = m.running_mean, m.running_var
            print(f"{name:40s} {rm.mean().item():15.4f} {rm.std().item():15.4f} "
                  f"{rv.mean().item():15.4f} {rv.std().item():15.4f}")


@torch.no_grad()
def tiled_predict_bn_mode(model, img, patch_size, output_stride, device, amp, bn_train_mode,
                           chunk_size=8):
    # Same overlapping sliding-window inference as tiled_predict, but lets us force BatchNorm into
    # "train mode" (each chunk's own batch statistics) instead of the eval-mode running
    # statistics, purely to test whether the running stats explain the train/val gap.
    return _tiled_predict_single(model, img, patch_size, output_stride, device, amp, chunk_size,
                                 overlap=cfg.infer_overlap, bn_train_mode=bn_train_mode)


diag_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
get_model_for_state_dict(model).load_state_dict(diag_ckpt["model_state"])
model.eval()
# The diagnostic runs BatchNorm in train mode on odd batch sizes, which the compiled graphs were never
# tested on, so it uses the uncompiled module (same weights, same results).
diag_model = get_model_for_state_dict(model)
print(f"Diagnosing checkpoint from epoch {diag_ckpt['epoch']} (val_mae={diag_ckpt['val_mae']:.2f})\n")

summarize_batchnorm_stats(diag_model)

n_diag = min(6, len(val_dataset))
diag_idx = random.sample(range(len(val_dataset)), n_diag)

eval_errs, trainbn_errs = [], []
print(f"\n{'idx':>5s} {'gt_count':>10s} {'eval_mode':>10s} {'eval_err':>10s} {'trainBN_mode':>13s} {'trainBN_err':>12s}")
for idx in diag_idx:
    img_tensor, density_gt, _ = val_dataset[idx]
    gt_count = density_gt.sum().item() / cfg.density_scale

    pred_eval = tiled_predict_bn_mode(diag_model, img_tensor.unsqueeze(0), cfg.patch_size,
                                       cfg.output_stride, DEVICE, cfg.amp, bn_train_mode=False)
    pred_trainbn = tiled_predict_bn_mode(diag_model, img_tensor.unsqueeze(0), cfg.patch_size,
                                          cfg.output_stride, DEVICE, cfg.amp, bn_train_mode=True)

    eval_count = pred_eval.sum().item() / cfg.density_scale
    trainbn_count = pred_trainbn.sum().item() / cfg.density_scale
    eval_err = abs(eval_count - gt_count)
    trainbn_err = abs(trainbn_count - gt_count)
    eval_errs.append(eval_err)
    trainbn_errs.append(trainbn_err)

    print(f"{idx:5d} {gt_count:10.1f} {eval_count:10.1f} {eval_err:10.1f} {trainbn_count:13.1f} {trainbn_err:12.1f}")

print(f"\nMean abs error -- eval-mode BN (what validate() uses): {np.mean(eval_errs):.1f}")
print(f"Mean abs error -- train-mode BN (tile's own batch stats): {np.mean(trainbn_errs):.1f}")
print("\nIf train-mode-BN error is dramatically lower -> BatchNorm running stats explain the gap")
print("(switch BatchNorm2d -> GroupNorm). If both are similarly large -> a genuine generalization")
print("gap or patch-vs-whole-image scale mismatch instead; inspect the visualization below.")


In [ ]:
def _diag_resolve_image_path(dataset, idx):
    # Local copy of the same lookup used in Section 12 (defined later in the notebook, so not
    # reused directly here) -- supports a plain ShanghaiTechDataset or a ConcatDataset of them.
    if isinstance(dataset, ConcatDataset):
        prev = 0
        for sub_dataset, cum in zip(dataset.datasets, dataset.cumulative_sizes):
            if idx < cum:
                return sub_dataset.image_paths[idx - prev]
            prev = cum
    return dataset.image_paths[idx]


def visualize_bn_diagnosis(model, dataset, idx, cfg, device):
    img_tensor, density_gt, _ = dataset[idx]
    pred_eval = tiled_predict_bn_mode(model, img_tensor.unsqueeze(0), cfg.patch_size,
                                       cfg.output_stride, device, cfg.amp, bn_train_mode=False)
    pred_trainbn = tiled_predict_bn_mode(model, img_tensor.unsqueeze(0), cfg.patch_size,
                                          cfg.output_stride, device, cfg.amp, bn_train_mode=True)

    gt_count = density_gt.sum().item() / cfg.density_scale
    eval_count = pred_eval.sum().item() / cfg.density_scale
    trainbn_count = pred_trainbn.sum().item() / cfg.density_scale

    # For display only, crop the reflect-padding (added so full images tile evenly for
    # sliding-window inference) back off -- otherwise the input panel shows a visible mirrored
    # duplicate whenever the padding is a large fraction of image size. Counts above are
    # computed on the full, uncropped tensors, matching how validate()'s MAE is calculated.
    with Image.open(_diag_resolve_image_path(dataset, idx)) as im:
        orig_w, orig_h = im.size
    ds_h, ds_w = orig_h // cfg.output_stride, orig_w // cfg.output_stride

    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_np = np.clip(img_np, 0, 1)[:orig_h, :orig_w]
    density_gt_disp = density_gt[:, :ds_h, :ds_w]
    pred_eval_disp = pred_eval[:, :, :ds_h, :ds_w]
    pred_trainbn_disp = pred_trainbn[:, :, :ds_h, :ds_w]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    axes[0].imshow(img_np); axes[0].set_title("Input"); axes[0].axis("off")
    axes[1].imshow(density_gt_disp.squeeze(0).numpy(), cmap="jet")
    axes[1].set_title(f"GT (count={gt_count:.1f})"); axes[1].axis("off")
    axes[2].imshow(pred_eval_disp.squeeze().cpu().numpy(), cmap="jet")
    axes[2].set_title(f"Eval-mode BN (count={eval_count:.1f})"); axes[2].axis("off")
    axes[3].imshow(pred_trainbn_disp.squeeze().cpu().numpy(), cmap="jet")
    axes[3].set_title(f"Train-mode BN (count={trainbn_count:.1f})"); axes[3].axis("off")
    plt.tight_layout()
    plt.show()


visualize_bn_diagnosis(diag_model, val_dataset, diag_idx[0], cfg, DEVICE)


### 10.3 Stage I diagnostics on the validation set (per image)

Ported from RMSC-Net (Section 9d of `RMSC_Net_Colab.ipynb`). This runs the best checkpoint
(`best_model.pth`) on every **validation** image, never the test set, with the same inference as the
test set (overlapping tiles, flip TTA if `cfg.use_tta`), and shows where the error comes from:

- **bias**: mean signed error, i.e. a systematic over- or under-count;
- **density thirds**: MAE and bias for the sparse / medium / dense thirds of the validation images;
- whether **a few images dominate**: share of the total absolute error from the 5 worst images;
- a **predicted-vs-ground-truth** scatter plot.

It also tries **enlarged inference** for very dense scenes, without retraining: each image is counted
at its normal size and enlarged by `cfg.diag_infer_scales`, optionally only when the normal-size
prediction is at least `cfg.diag_infer_thresholds` people. The setting with the lowest validation MAE
is kept for Section 11, where it is reported *next to* the normal-size result. On a tie, normal size
wins. Everything is saved to Drive: `stage1_val_diagnostics.csv`, `stage1_val_infer_scale_search.csv`,
`figures/stage1_val_pred_vs_gt.png`, and `summary.json`.


In [ ]:
INFER_SCALES = [float(s) for s in cfg.diag_infer_scales]
INFER_THRESHOLDS = [float(t) for t in cfg.diag_infer_thresholds]


def _pred_col(scale):
    return f"pred_x{scale:g}"


@torch.no_grad()
def per_image_counts(model, dataset, scales=(1.0,), tta=False, desc="per-image"):
    """One row per image: GT count and the predicted count at each inference scale."""
    model.eval()
    rows = []
    for idx in tqdm(range(len(dataset)), desc=desc, leave=False):
        img, density_gt, _ = dataset[idx]
        img = img.unsqueeze(0).to(DEVICE)
        row = {"image": os.path.basename(dataset_image_path(dataset, idx)),
               "gt_count": density_gt.sum().item() / cfg.density_scale}
        for s in scales:
            row[_pred_col(s)] = predict_density(model, img, tta=tta, infer_scale=s).sum().item() / cfg.density_scale
        rows.append(row)
    return pd.DataFrame(rows)


def apply_infer_setting(counts, scale, threshold):
    """Counts under (scale, threshold): enlarge only images whose normal-size count >= threshold."""
    normal = counts[_pred_col(1.0)].to_numpy()
    if scale == 1.0:
        return normal
    big = counts[_pred_col(scale)].to_numpy()
    return big if threshold <= 0 else np.where(normal >= threshold, big, normal)


def choose_infer_scale(counts, verbose=True):
    """Grid-search the enlargement factor / threshold on the VALIDATION counts only."""
    gt = counts["gt_count"].to_numpy()
    rows = []
    for s in INFER_SCALES:
        for t in (INFER_THRESHOLDS if s != 1.0 else [0.0]):
            p = apply_infer_setting(counts, s, t)
            rows.append({"scale": s, "threshold": t, "val_MAE": float(np.abs(p - gt).mean()),
                         "val_RMSE": float(np.sqrt(((p - gt) ** 2).mean()))})
    # stable sort: on ties the simplest setting (normal size, listed first) wins
    grid = pd.DataFrame(rows).sort_values("val_MAE", kind="mergesort").reset_index(drop=True)
    if verbose:
        print("Enlarged-inference search on the validation set (best first):")
        print(grid.head(8).round(2).to_string(index=False))
    best = grid.iloc[0]
    return float(best["scale"]), float(best["threshold"]), grid


INFER_SCALE, INFER_THRESHOLD = 1.0, 0.0   # used by Section 11; replaced by the search below
if os.path.exists(best_ckpt_path):
    diag_ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
    get_model_for_state_dict(model).load_state_dict(diag_ckpt["model_state"])
    model.eval()
    scales = sorted(set([1.0] + INFER_SCALES))
    val_counts = per_image_counts(model, val_dataset, scales, tta=cfg.use_tta, desc="Val diagnostics")

    diag = val_counts.copy()
    diag["pred_count"] = diag[_pred_col(1.0)]
    diag["error"] = diag["pred_count"] - diag["gt_count"]
    diag["abs_error"] = diag["error"].abs()
    diag["rel_error_%"] = 100 * diag["error"] / diag["gt_count"].clip(lower=1)
    diag = diag.sort_values("gt_count").reset_index(drop=True)
    diag["density_third"] = pd.qcut(diag["gt_count"].rank(method="first"), 3,
                                    labels=["sparse", "medium", "dense"])
    save_csv(diag, "stage1_val_diagnostics.csv", index=False)

    mae, bias = diag["abs_error"].mean(), diag["error"].mean()
    rmse = float(np.sqrt((diag["error"] ** 2).mean()))
    top5_share = diag["abs_error"].nlargest(5).sum() / max(diag["abs_error"].sum(), 1e-8) * 100
    print(f"Stage I best checkpoint (epoch {diag_ckpt['epoch']}, val MAE {diag_ckpt['val_mae']:.2f} without TTA) "
          f"on {len(diag)} validation images{' with flip TTA' if cfg.use_tta else ''}")
    print(f"  MAE {mae:.2f} | RMSE {rmse:.2f} | mean signed error (bias) {bias:+.2f} "
          f"({'over' if bias > 0 else 'under'}-counting on average)")
    print(f"  over-counted {int((diag['error'] > 0).sum())} images, under-counted {int((diag['error'] < 0).sum())}")
    print(f"  the 5 worst images account for {top5_share:.0f}% of the total absolute error")
    by_third = diag.groupby("density_third", observed=True).agg(
        images=("gt_count", "size"), gt_range_min=("gt_count", "min"), gt_range_max=("gt_count", "max"),
        MAE=("abs_error", "mean"), bias=("error", "mean"), rel_bias_pct=("rel_error_%", "mean"))
    save_csv(by_third.reset_index(), "stage1_val_diagnostics_by_third.csv", index=False)
    print("\nBy density third:")
    print(by_third.round(1).to_string())
    print("\nPer image (sorted by ground-truth count):")
    print(diag[["image", "gt_count", "pred_count", "error", "rel_error_%"]].round(1).to_string(index=False))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(diag["gt_count"], diag["pred_count"], s=25)
    lim = [0, max(diag["gt_count"].max(), diag["pred_count"].max()) * 1.05]
    ax.plot(lim, lim, "r--", lw=1, label="perfect")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("ground-truth count"); ax.set_ylabel("predicted count")
    ax.set_title(f"Stage I best (epoch {diag_ckpt['epoch']}) - validation, MAE {mae:.1f}")
    ax.legend()
    save_fig(fig, "stage1_val_pred_vs_gt.png")
    plt.show()
    update_summary("stage1_val_diagnostics", {"epoch": diag_ckpt["epoch"], "MAE": mae, "RMSE": rmse,
                                              "bias": bias, "top5_error_share_pct": top5_share,
                                              "tta": cfg.use_tta})

    print()
    INFER_SCALE, INFER_THRESHOLD, grid = choose_infer_scale(val_counts)
    save_csv(grid, "stage1_val_infer_scale_search.csv", index=False)
    val_counts["pred_enlarged"] = apply_infer_setting(val_counts, INFER_SCALE, INFER_THRESHOLD)
    by_gt = val_counts.sort_values("gt_count").reset_index(drop=True)
    by_gt["density_third"] = diag["density_third"].values
    by_gt["abs_err"] = (by_gt["pred_enlarged"] - by_gt["gt_count"]).abs()
    print(f"\nBest on validation: scale {INFER_SCALE:g} for images predicted >= {INFER_THRESHOLD:.0f} people "
          f"-> val MAE {by_gt['abs_err'].mean():.2f} (was {mae:.2f} at normal size)")
    print(by_gt.groupby("density_third", observed=True)["abs_err"].mean().round(1).to_string())
    update_summary("stage1_val_infer_scale", {"scale": INFER_SCALE, "threshold": INFER_THRESHOLD,
                                              "val_MAE": float(by_gt["abs_err"].mean()),
                                              "val_MAE_normal": mae, "chosen_on": "validation set"})
else:
    print("[SKIP] no best_model.pth yet -- train first (Section 10).")


## 11. Evaluate the best model on the held-out test set

Uses test-time augmentation (`cfg.use_tta`) when enabled -- each prediction is averaged with its horizontal-flip counterpart, a small but nearly-free accuracy boost since it's only applied here and at real inference, never during per-epoch validation.

The headline MAE / RMSE use normal-size inference, exactly as before. If Section 10.3 chose an
enlarged-inference setting on the validation set, the test result with that setting is printed
**separately**. Per-image predictions (`test_per_image.csv`) and metrics (`test_metrics.json`, MAE,
RMSE, NAE) are saved to Drive.


In [ ]:
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
get_model_for_state_dict(model).load_state_dict(ckpt["model_state"])
print(f"Loaded best model from epoch {ckpt['epoch']} (val_mae={ckpt['val_mae']:.2f}) -- {best_ckpt_path}")

test_stats = validate(model, test_loader, criterion, DEVICE, cfg, desc="Test", tta=cfg.use_tta)
label = "Test set (combined)" if len(test_loaders_by_part) > 1 else "Test set"
print(f"{label}  ->  loss={test_stats['loss']:.4f}  MAE={test_stats['mae']:.2f}  "
      f"RMSE={test_stats['rmse']:.2f}")

if len(test_loaders_by_part) > 1:
    for p, loader in test_loaders_by_part.items():
        p_stats = validate(model, loader, criterion, DEVICE, cfg, desc=f"Test part_{p}",
                            tta=cfg.use_tta)
        print(f"  part_{p}  ->  loss={p_stats['loss']:.4f}  MAE={p_stats['mae']:.2f}  "
              f"RMSE={p_stats['rmse']:.2f}")

# ---- per-image predictions + metrics on Drive ----
infer_scale = float(globals().get("INFER_SCALE", 1.0))
infer_threshold = float(globals().get("INFER_THRESHOLD", 0.0))
scales = [1.0] if infer_scale == 1.0 else [1.0, infer_scale]
test_counts = per_image_counts(model, test_dataset, scales, tta=cfg.use_tta, desc="Test per-image")
test_counts["pred_count"] = test_counts[_pred_col(1.0)]
test_counts["error"] = test_counts["pred_count"] - test_counts["gt_count"]


def count_metrics(pred, gt):
    err = np.asarray(pred) - np.asarray(gt)
    return {"MAE": float(np.abs(err).mean()), "RMSE": float(np.sqrt((err ** 2).mean())),
            "NAE": float((np.abs(err) / np.maximum(np.asarray(gt), 1)).mean())}


test_metrics = {"epoch": int(ckpt["epoch"]), "val_mae": float(ckpt["val_mae"]), "tta": cfg.use_tta,
                "training_finished": bool(globals().get("TRAINING_FINISHED", True)),
                "loss": test_stats["loss"],
                **count_metrics(test_counts["pred_count"], test_counts["gt_count"])}
print(f"\nNormal-size inference (headline): MAE={test_metrics['MAE']:.2f}  RMSE={test_metrics['RMSE']:.2f}  "
      f"NAE={test_metrics['NAE']:.4f}")
if infer_scale != 1.0:
    test_counts["pred_enlarged"] = apply_infer_setting(test_counts, infer_scale, infer_threshold)
    enl = count_metrics(test_counts["pred_enlarged"], test_counts["gt_count"])
    test_metrics["enlarged"] = dict(enl, scale=infer_scale, threshold=infer_threshold,
                                    chosen_on="validation set")
    print(f"Enlarged inference (scale {infer_scale:g}, images >= {infer_threshold:.0f} people, chosen on "
          f"validation): MAE={enl['MAE']:.2f}  RMSE={enl['RMSE']:.2f}  NAE={enl['NAE']:.4f}")
else:
    print("Validation search kept normal-size inference -- no separate enlarged result.")

save_csv(test_counts, "test_per_image.csv", index=False)
save_json(test_metrics, "test_metrics.json")
update_summary("test", test_metrics)
if not test_metrics["training_finished"]:
    print("\nNote: training is not finished yet -- these are the numbers of the best checkpoint so far.")


## 12. Qualitative visualisation

Compare a few test images against their ground-truth and predicted density maps (with
predicted vs. actual head counts).


In [ ]:
def _resolve_image_path(dataset, idx):
    # Supports both a plain ShanghaiTechDataset and a ConcatDataset of them (cfg.part="both"),
    # so the original (pre-padding) image size can be looked up for display cropping below.
    if isinstance(dataset, ConcatDataset):
        prev = 0
        for sub_dataset, cum in zip(dataset.datasets, dataset.cumulative_sizes):
            if idx < cum:
                return sub_dataset.image_paths[idx - prev]
            prev = cum
    return dataset.image_paths[idx]


def visualize_samples(model, dataset, cfg, device, n=3):
    idxs = random.sample(range(len(dataset)), min(n, len(dataset)))
    fig, axes = plt.subplots(len(idxs), 3, figsize=(12, 4 * len(idxs)))
    if len(idxs) == 1:
        axes = axes[None, :]

    for row, idx in enumerate(idxs):
        img_tensor, density_gt, _ = dataset[idx]
        pred = tiled_predict(model, img_tensor.unsqueeze(0), cfg.patch_size,
                              cfg.output_stride, device, amp=cfg.amp)

        # Counts are computed on the full (padded) tensors, matching how Section 11's MAE/RMSE
        # are actually calculated -- the reflect-padded region contributes exactly zero GT
        # density, so this is identical to counting only the real image content.
        gt_count = density_gt.sum().item() / cfg.density_scale
        pred_count = pred.sum().item() / cfg.density_scale

        # For display only, crop the reflect-padding (added by pad_to_multiple so full images
        # tile evenly for sliding-window inference) back off -- otherwise the input panel shows
        # a visible mirrored duplicate whenever the padding is a large fraction of image size.
        with Image.open(_resolve_image_path(dataset, idx)) as im:
            orig_w, orig_h = im.size
        ds_h, ds_w = orig_h // cfg.output_stride, orig_w // cfg.output_stride

        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        img_np = np.clip(img_np, 0, 1)[:orig_h, :orig_w]
        density_gt_disp = density_gt[:, :ds_h, :ds_w]
        pred_disp = pred[:, :, :ds_h, :ds_w]

        axes[row, 0].imshow(img_np); axes[row, 0].set_title("Input"); axes[row, 0].axis("off")
        axes[row, 1].imshow(density_gt_disp.squeeze(0).numpy(), cmap="jet")
        axes[row, 1].set_title(f"GT density (count={gt_count:.1f})"); axes[row, 1].axis("off")
        axes[row, 2].imshow(pred_disp.squeeze().cpu().numpy(), cmap="jet")
        axes[row, 2].set_title(f"Predicted density (count={pred_count:.1f})"); axes[row, 2].axis("off")

    plt.tight_layout()
    plt.show()


visualize_samples(model, test_dataset, cfg, DEVICE, n=3)


## 13. Inference on a new image

Memory-efficient, patch-based inference for any arbitrary image (not necessarily from
ShanghaiTech).


Also applies test-time augmentation by default (`cfg.use_tta`), matching Section 11.


In [ ]:
def predict_count(model, image_path, cfg, device):
    img = np.array(Image.open(image_path).convert("RGB"), dtype=np.uint8)
    h, w = img.shape[:2]
    # Only pad to a multiple of output_stride; tiled_predict handles tiling itself. Reflect-padding
    # up to a multiple of patch_size duplicated real people into the padding and over-counted.
    new_h = int(np.ceil(h / cfg.output_stride) * cfg.output_stride)
    new_w = int(np.ceil(w / cfg.output_stride) * cfg.output_stride)
    img_padded = np.pad(img, ((0, new_h - h), (0, new_w - w), (0, 0)), mode="reflect")

    img_tensor = _to_tensor_norm(img_padded).unsqueeze(0)
    pred_density = tiled_predict(model, img_tensor, cfg.patch_size, cfg.output_stride,
                                  device, amp=cfg.amp, tta=cfg.use_tta, chunk_size=cfg.batch_size)
    count = pred_density.sum().item() / cfg.density_scale
    return count, pred_density.squeeze().cpu().numpy()


# Example:
# count, density_map = predict_count(model, "/content/some_crowd_photo.jpg", cfg, DEVICE)
# print(f"Estimated crowd count: {count:.1f}")


## 14. Memory-optimisation techniques used in this notebook

- **Patch-based training** (`patch_size`-sized random crops) and **tiled sliding-window
  inference** (`tiled_predict`) — bound peak activation memory (including the O(N²)
  self-attention cost) to a single patch regardless of source image resolution.
- **Automatic mixed precision** (`torch.autocast` + `GradScaler`) — halves activation memory
  and speeds up matmuls on modern GPUs.
- **Gradient accumulation** (`grad_accum_steps`) — simulates a larger effective batch size
  without holding all samples' activations at once.
- **Partial backbone freezing** (`freeze_backbone_stage1`) — no gradients/optimizer state are
  kept for the frozen early VGG layers.
- **Optional gradient checkpointing** (`use_grad_checkpoint`) — trades recompute for memory on
  the backbone/transformer if you hit OOM on a smaller GPU.
- **Sum-pooled, downsampled density targets** — the loss/metrics operate on a `1/output_stride`
  resolution target instead of a full-resolution one, shrinking both target and prediction
  tensors substantially.
- **Disk-cached density maps** (`.npy` per image) — the expensive Gaussian-splat step runs once,
  not every epoch.
- **DataLoader tuning** — `pin_memory`, `persistent_workers`, and modest `num_workers` overlap
  host-side preprocessing with GPU compute without duplicating worker startup cost each epoch.
- Periodic `torch.cuda.empty_cache()` after each epoch's train/val pass.
- **Numerically-stable mixed precision** (`bfloat16` when available, float32 loss computation, gradient clipping) -- prevents the NaN corruption described in Section 10.1, which otherwise silently wastes an entire run's compute.

- **Chunked, overlapping tiled inference** -- `tiled_predict` batches an image's (50%-overlapping) tiles `cfg.batch_size` at a time into single forward passes (instead of one tile per forward call), improving GPU utilisation while keeping peak memory bounded to a known-safe chunk, regardless of how many tiles a large image needs.
- **`torch.compile()`** (`cfg.use_torch_compile`) -- JIT-compiles the model for faster train/inference with a safe fallback to eager mode if compilation isn't available; checkpoints are always saved/loaded through the unwrapped module so they stay compatible whether or not compile is enabled in a given session.

## 15. Checkpoints on Google Drive

With `cfg.use_drive = True` (default) everything is already written to Google Drive during training
(`last.pth`, `last_prev.pth`, `best_model.pth`, periodic `snapshot_epochNNN.pth`), and all tables,
figures and metrics go to `results/` in the same folder (`summary.json` collects the headline
numbers). The cell below lists what is there.


In [ ]:
for root, _, files in os.walk(cfg.ckpt_dir):
    for f in sorted(files):
        path = os.path.join(root, f)
        print(f"{os.path.relpath(path, cfg.ckpt_dir):48s} {os.path.getsize(path) / 1e6:8.2f} MB")


# Part II: RMSC-style robustness study, explainability, ablations and statistics

Sections 16-26 turn the LightARFS model into an RMSC-Net-style study. The **main model trained in
Section 10 is not modified**, and its results in Section 11 stay the headline clean numbers. The study
trains its own variants. Each one is stored on Drive under `<ckpt_dir>/rmsc_study/<name>_seed<seed>/`
and resumes epoch by epoch after a disconnect, like the main run.

How the RMSC-Net components map onto this model:

| RMSC-Net component | Here |
|---|---|
| Encoder | ImageNet MobileNetV2 (strides 8/16/32) + 1x1 laterals, multi-scale fusion (addition by default) |
| **MSRM** (multi-scale receptive-field module) | the 3 adaptive receptive-field selection (ARFS) blocks, dilations `cfg.light_dilations` |
| **DAA** (degradation-aware attention) | ECA channel attention -> spatial attention -> **reliability gate** (`ReliabilityGate`, study variants only) |
| **Consistency** learning | Stage II: clean supervision + density consistency + feature consistency between clean and corrupted views |
| **Adversarial training** | Stage III: PGD adversarial views with supervision + consistency |

Section overview: 16 corruptions, 17 adversarial attacks, 18 model variants, 19 experiment runner,
20 training of all experiments, 21 robustness evaluation, 22 explainable-AI analysis, 23 ablation,
24 additional ablations, 25 statistical validation, 26 results and discussion.

**Compute budget.** A Stage I variant costs about as much as the main run, and the study has about
20 experiments (see the list in Section 19). Use `cfg.abl_experiments` to train a subset, use
`cfg.abl_epochs_per_session` / `cfg.max_session_hours` to spread training over several sessions, and
just **Run all** again to continue. The analysis sections use whatever has finished so far and say
what is still missing.

## 16. Environmental corruptions

The 12 corruption types x 5 severities are the same as in RMSC-Net (Section 10 of
`RMSC_Net_Colab.ipynb`), so results can be compared. Note that `frost`, `snow` and `rain` are
simplified there (additive noise of increasing strength), not the ImageNet-C renderings. For evaluation,
each (image, corruption, severity) uses its own fixed random seed, so every model sees exactly the
same corrupted images. This is what makes the per-image comparisons in Section 25 paired.


In [ ]:
import zlib
import uuid
import copy
from scipy import stats as sps

SEVERITY_PARAMS = {
    "gaussian_noise":   [0.04, 0.06, 0.08, 0.12, 0.18],
    "shot_noise":       [60, 25, 12, 5, 3],
    "impulse_noise":    [0.03, 0.06, 0.09, 0.17, 0.27],
    "defocus_blur":     [1, 2, 3, 4, 6],
    "motion_blur":      [3, 5, 8, 12, 15],
    "fog":              [0.2, 0.3, 0.5, 0.7, 0.9],
    "frost":            [0.1, 0.15, 0.2, 0.3, 0.4],
    "snow":             [0.05, 0.1, 0.15, 0.2, 0.3],
    "rain":             [0.05, 0.1, 0.15, 0.2, 0.3],
    "brightness":       [0.1, 0.2, 0.35, 0.5, 0.65],
    "contrast":         [0.75, 0.5, 0.4, 0.3, 0.15],
    "jpeg_compression": [80, 60, 40, 25, 10],
}
CORRUPTION_TYPES = list(SEVERITY_PARAMS)
_MEAN_T = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
_STD_T = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)


def apply_corruption(img_np, corruption_type, severity, rng=None):
    """uint8 (H, W, 3) -> corrupted uint8 (H, W, 3). Same definitions as RMSC-Net."""
    rng = np.random if rng is None else rng
    p = SEVERITY_PARAMS[corruption_type][severity - 1]
    img = img_np.astype(np.float32) / 255.0
    if corruption_type == "gaussian_noise":
        img = img + rng.randn(*img.shape).astype(np.float32) * p
    elif corruption_type == "shot_noise":
        img = rng.poisson(np.clip(img, 0, 1) * p).astype(np.float32) / p
    elif corruption_type == "impulse_noise":
        mask = rng.random_sample(img.shape[:2]) < p
        img[mask] = rng.randint(0, 2, (int(mask.sum()), 1)).astype(np.float32)
    elif corruption_type == "defocus_blur":
        k = 2 * p + 1
        kk = np.zeros((k, k), np.float32)
        cv2.circle(kk, (p, p), p, 1, -1)
        kk /= kk.sum()
        img = cv2.filter2D(img, -1, kk)
    elif corruption_type == "motion_blur":
        kernel = np.zeros((p, p), np.float32)
        kernel[p // 2, :] = 1.0 / p
        img = cv2.filter2D(img, -1, kernel)
    elif corruption_type == "fog":
        img = img * (1 - p) + p
    elif corruption_type in ("frost", "snow", "rain"):
        img = img + rng.randn(*img.shape).astype(np.float32) * p
    elif corruption_type == "brightness":
        img = img + p
    elif corruption_type == "contrast":
        mean = img.mean(axis=(0, 1), keepdims=True)
        img = (img - mean) * p + mean
    elif corruption_type == "jpeg_compression":
        u8 = (np.clip(img, 0, 1) * 255).round().astype(np.uint8)
        _, enc = cv2.imencode(".jpg", cv2.cvtColor(u8, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), int(p)])
        img = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return (np.clip(img, 0, 1) * 255).round().astype(np.uint8)


def to_uint8(img_norm):
    """Normalised (3, H, W) tensor -> uint8 (H, W, 3)."""
    x = (img_norm.detach().float().cpu() * _STD_T[0] + _MEAN_T[0]).clamp(0, 1)
    return (x.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)


def from_uint8(arr, device):
    t = torch.from_numpy(np.ascontiguousarray(arr)).permute(2, 0, 1).float().div(255.0).unsqueeze(0)
    return ((t - _MEAN_T) / _STD_T).to(device)


def eval_seed(*keys):
    """Deterministic seed for one (image, perturbation) pair, identical for every model."""
    return zlib.crc32("|".join(map(str, keys)).encode())


def corrupt_image(img_norm, ctype, severity, seed):
    """(1, 3, H, W) normalised -> same image with one corruption, reproducible via `seed`."""
    rng = np.random.RandomState(seed % (2 ** 32))
    return from_uint8(apply_corruption(to_uint8(img_norm[0]), ctype, severity, rng), img_norm.device)


def random_corrupt_batch(imgs, rng):
    """Training: a random corruption type and severity for every sample of the batch."""
    out = []
    for i in range(imgs.size(0)):
        ctype = CORRUPTION_TYPES[rng.randint(len(CORRUPTION_TYPES))]
        sev = int(rng.randint(1, 6))
        out.append(from_uint8(apply_corruption(to_uint8(imgs[i]), ctype, sev, rng), imgs.device))
    return torch.cat(out)


# preview: one test image under every corruption (severity 3)
_img, _, _ = test_dataset[0]
fig, axes = plt.subplots(2, 7, figsize=(28, 8))
axes = axes.ravel()
axes[0].imshow(to_uint8(_img)); axes[0].set_title("clean"); axes[0].axis("off")
for ax, ct in zip(axes[1:], CORRUPTION_TYPES):
    ax.imshow(to_uint8(corrupt_image(_img.unsqueeze(0), ct, 3, eval_seed("preview", ct))[0]))
    ax.set_title(f"{ct} (s3)"); ax.axis("off")
axes[-1].axis("off")
plt.tight_layout()
save_fig(fig, "corruption_examples.png")
plt.show()


## 17. Adversarial attacks (FGSM, PGD, BIM)

White-box, untargeted attacks that maximise the density-map MSE. ε and the step size are in pixel
units of [0, 1] images, and adversarial images are kept inside the valid image range (same
conventions as RMSC-Net). For whole test images the attack goes through `tiled_forward`, a
differentiable copy of the sliding-window inference used for evaluation. This means the attack
targets exactly the prediction that is scored. PGD's random start uses a fixed per-image seed, so
every model is attacked reproducibly.


In [ ]:
def tiled_forward(model, img, overlap=None, chunk_size=None):
    """Differentiable version of _tiled_predict_single (eval-mode BatchNorm, no flip TTA)."""
    overlap = cfg.infer_overlap if overlap is None else overlap
    chunk_size = chunk_size or cfg.batch_size
    ps, st = cfg.patch_size, cfg.output_stride
    _, _, H, W = img.shape
    pad_h, pad_w = max(0, ps - H), max(0, ps - W)
    x = F.pad(img, (0, pad_w, 0, pad_h), value=0.0) if (pad_h or pad_w) else img
    Hp, Wp = x.shape[-2:]
    step = max(st, min(int(ps * (1 - overlap)) // st * st, ps))
    positions = [(y, xx) for y in _tile_starts(Hp, ps, step) for xx in _tile_starts(Wp, ps, step)]
    t = ps // st
    pred_sum = torch.zeros(1, 1, Hp // st, Wp // st, device=img.device)
    weight = torch.zeros_like(pred_sum)
    amp_on = cfg.amp and img.device.type == "cuda"
    for s in range(0, len(positions), chunk_size):
        cp = positions[s:s + chunk_size]
        batch = torch.cat([x[:, :, y:y + ps, xx:xx + ps] for y, xx in cp], dim=0)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=amp_on):
            out = model(batch)
        out = out.float()
        for (y, xx), p in zip(cp, out):
            oy, ox = y // st, xx // st
            pred_sum[:, :, oy:oy + t, ox:ox + t] = pred_sum[:, :, oy:oy + t, ox:ox + t] + p.unsqueeze(0)
            weight[:, :, oy:oy + t, ox:ox + t] += 1
    return (pred_sum / weight.clamp_min(1.0))[:, :, :H // st, :W // st]


def _forward_any(model, x, tiled):
    if tiled:
        return tiled_forward(model, x)
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(cfg.amp and x.device.type == "cuda")):
        return model(x).float()


def _px(v, ref):  # pixel units -> normalised units, per channel
    return v / _STD_T.to(ref.device)


def _project(adv, clean, eps):
    e = _px(eps, clean)
    adv = torch.max(torch.min(adv, clean + e), clean - e)                        # L_inf ball
    lo = (0 - _MEAN_T.to(clean.device)) / _STD_T.to(clean.device)
    hi = (1 - _MEAN_T.to(clean.device)) / _STD_T.to(clean.device)
    return torch.max(torch.min(adv, hi), lo).detach()                            # valid image range


def _input_grad(model, x, target, tiled):
    x = x.clone().detach().requires_grad_(True)
    loss = F.mse_loss(_forward_any(model, x, tiled), target)
    g, = torch.autograd.grad(loss, x)   # gradient w.r.t. the input only
    return g


def _attack(fn):
    def wrapped(model, x, target, eps, *args, **kwargs):
        was_training = model.training
        model.eval()                    # attacks always use eval-mode BatchNorm
        try:
            return fn(model, x, target, eps, *args, **kwargs)
        finally:
            model.train(was_training)
    return wrapped


@_attack
def fgsm_attack(model, x, target, eps, tiled=True, generator=None):
    return _project(x + _px(eps, x) * _input_grad(model, x, target, tiled).sign(), x, eps)


@_attack
def pgd_attack(model, x, target, eps, alpha=None, steps=None, tiled=True, generator=None):
    alpha = eps / 4 if alpha is None else alpha
    steps = cfg.adv_eval_steps if steps is None else steps
    noise = torch.rand(x.shape, generator=generator, device=x.device) * 2 - 1
    adv = _project(x + _px(eps, x) * noise, x, eps)
    for _ in range(steps):
        adv = _project(adv + _px(alpha, x) * _input_grad(model, adv, target, tiled).sign(), x, eps)
    return adv


@_attack
def bim_attack(model, x, target, eps, alpha=None, steps=10, tiled=True, generator=None):
    alpha = eps / 10 if alpha is None else alpha
    adv = x.clone().detach()
    for _ in range(steps):
        adv = _project(adv + _px(alpha, x) * _input_grad(model, adv, target, tiled).sign(), x, eps)
    return adv


ATTACKS = {"fgsm": fgsm_attack, "pgd": pgd_attack, "bim": bim_attack}
print(f"Attacks ready: {list(ATTACKS)} | training PGD eps={cfg.adv_eps * 255:.0f}/255, "
      f"alpha={cfg.adv_alpha * 255:.0f}/255, steps={cfg.adv_train_steps}; eval PGD steps={cfg.adv_eval_steps}")


## 18. Model variants for the study

All variants share the main model's backbone, width, decoder, data pipeline, loss and optimiser.
Only the components below change. Every variant stays under the 1M-parameter budget.


In [ ]:
MAIN_NAME = "lightarfs_main"
MAIN_MODEL_SPEC = dict(msrm=True, daa="tiny", fusion="add", dilations=tuple(cfg.light_dilations))
FULL_SPEC = dict(msrm=True, daa="full", fusion="add", dilations=tuple(cfg.light_dilations))


def build_variant(spec, pretrained=None):
    return LightARFSCounter(
        pretrained=cfg.pretrained_backbone if pretrained is None else pretrained,
        freeze_stem=cfg.freeze_backbone_stage1, width=cfg.light_width, n_blocks=cfg.light_blocks,
        dilations=tuple(spec["dilations"]), use_msrm=spec["msrm"], daa=spec["daa"], fusion=spec["fusion"])


_COMPLEXITY_CACHE = os.path.join(RESULT_DIR, "variant_complexity.json")


def variant_complexity(spec):
    """Parameters and GFLOPs (1024x768 input, FLOPs = 2 x MACs) of a variant, cached on Drive."""
    key = json.dumps(spec, sort_keys=True)
    cache = json.load(open(_COMPLEXITY_CACHE)) if os.path.exists(_COMPLEXITY_CACHE) else {}
    if key not in cache:
        m = build_variant(spec, pretrained=False).to(DEVICE).eval()
        info = {"params": sum(p.numel() for p in m.parameters())}
        try:
            from torch.utils.flop_counter import FlopCounterMode
            with torch.no_grad(), torch.random.fork_rng(devices=[torch.cuda.current_device()] if DEVICE.type == "cuda" else []):
                counter = FlopCounterMode(display=False)
                with counter:
                    m(torch.zeros(1, 3, 768, 1024, device=DEVICE))
            info["GFLOPs"] = counter.get_total_flops() / 1e9
        except Exception as e:
            print(f"FLOP count skipped ({e})")
            info["GFLOPs"] = float("nan")
        cache[key] = info
        _atomic_write_text(_COMPLEXITY_CACHE, json.dumps(cache, indent=2))
        del m
    return cache[key]


if cfg.model_type != "light":
    print("[NOTE] The study (Sections 16-26) is built on the light model; cfg.model_type is "
          f"'{cfg.model_type}', so the main run is left out of the comparisons.")
for label, spec in [("main LightARFS", MAIN_MODEL_SPEC), ("Baseline (no MSRM, no DAA)", dict(FULL_SPEC, msrm=False, daa="none")),
                    ("Full (MSRM + DAA with reliability gate)", FULL_SPEC)]:
    info = variant_complexity(spec)
    assert info["params"] < cfg.max_params, f"{label}: {info['params']:,} params exceeds the budget"
    print(f"{label:42s} params {info['params']:>9,}  GFLOPs@1024x768 {info['GFLOPs']:.2f}")


## 19. Experiment registry and training runner

| Experiment | Stage | Starts from | What it trains | Used in |
|---|---|---|---|---|
| `baseline` | I | ImageNet | no MSRM, no DAA | 23 |
| `model_a` | I | ImageNet | + MSRM | 23 |
| `model_b` | I | ImageNet | + MSRM + DAA (with reliability gate) | 23, 24 |
| `dil_<rates>` | I | ImageNet | `model_b` with other dilation rates | 24 |
| `fusion_concat`, `fusion_adaptive` | I | ImageNet | `model_b` with other fusion | 24 |
| `model_c` | II | `model_b` | complete consistency objective (density + count + consistency + feature consistency) | 23, 24 |
| `loss_d`, `loss_dc`, `loss_dcons`, `loss_dfeat` | II | `model_b` | density only / + count / + consistency / + feature consistency | 24 |
| `rob_corrupt_p1`, `rob_adv_p1` | II | `model_b` | corruption augmentation / adversarial training (phase 1) | 24 |
| `rmsc_full` | III | `model_c` | adversarial training with consistency = **RMSC-Net** | 22-26 |
| `rob_clean`, `rob_corrupt`, `rob_adv`, `rob_corrupt_adv` | III | Stage II run | clean / corruption / adversarial / corruption + adversarial (phase 2) | 24 |

Design choices, so that comparisons are fair:

- **Stage I** variants use exactly the main recipe (`train_one_epoch`, the same loss, LRs, warmup,
  `ReduceLROnPlateau` and early stopping), with `cfg.abl_s1_max_epochs` as the upper limit.
- **Loss-function ablation:** every loss variant fine-tunes the same `model_b` checkpoint for the
  same Stage II budget. *Density* = the pixel MSE + SSIM terms of the main loss, *count* = its count
  L1 term, *consistency* = MSE between the corrupted-view and the (stop-gradient) clean-view density
  map, *feature consistency* = 1 - cosine similarity of the features entering the decoder. Corrupted
  views are only used by the consistency terms.
- **Robust-training ablation:** every variant gets the same two phases (Stage II budget, then
  Stage III budget) starting from `model_b`. Corruption augmentation and adversarial training
  supervise the perturbed view directly (clean and perturbed loss averaged). The complete
  consistency framework is `model_c` -> `rmsc_full`.
- The best checkpoint of every stage is chosen on the **clean validation MAE**, never on the test set.
- Lineage: every best checkpoint gets a unique id. If a parent is retrained, its children notice
  this and retrain from the new parent.


In [ ]:
STUDY_DIR = os.path.join(cfg.ckpt_dir, "rmsc_study")
os.makedirs(STUDY_DIR, exist_ok=True)

MODES = {  # Stage II/III objectives (Stage I always uses the main loss)
    "clean_d":         dict(count=False),
    "clean_dc":        dict(count=True),
    "complete":        dict(count=True, view="corrupt", cons=True, feat=True),
    "d_cons":          dict(count=False, view="corrupt", cons=True),
    "d_feat":          dict(count=False, view="corrupt", feat=True),
    "corrupt_aug":     dict(count=True, view="corrupt", sup_view=True),
    "adv":             dict(count=True, view="adv", sup_view=True),
    "adv_consistency": dict(count=True, view="adv", sup_view=True, cons=True, feat=True),
}

EXPERIMENTS = {}


def _exp(name, stage, label, model=None, parent=None, mode=None):
    if model is None:
        model = EXPERIMENTS[parent]["model"]
    EXPERIMENTS[name] = dict(stage=stage, label=label, model=model, parent=parent, mode=mode)


_exp("baseline", 1, "Baseline", dict(FULL_SPEC, msrm=False, daa="none"))
_exp("model_a", 1, "Model A (+MSRM)", dict(FULL_SPEC, daa="none"))
_exp("model_b", 1, "Model B (+MSRM +DAA)", dict(FULL_SPEC))
_exp("model_c", 2, "Model C (+consistency)", parent="model_b", mode="complete")
_exp("rmsc_full", 3, "RMSC-Net", parent="model_c", mode="adv_consistency")
DILATION_EXPS = {}
for d in cfg.abl_dilation_sets:
    d = tuple(int(v) for v in d)
    if d != tuple(FULL_SPEC["dilations"]):
        name = "dil_" + "-".join(map(str, d))
        _exp(name, 1, f"dilations {d}", dict(FULL_SPEC, dilations=d))
        DILATION_EXPS[name] = d
_exp("fusion_concat", 1, "concatenation fusion", dict(FULL_SPEC, fusion="concat"))
_exp("fusion_adaptive", 1, "adaptive weighted fusion", dict(FULL_SPEC, fusion="adaptive"))
_exp("loss_d", 2, "density only", parent="model_b", mode="clean_d")
_exp("loss_dc", 2, "density + count", parent="model_b", mode="clean_dc")
_exp("loss_dcons", 2, "density + consistency", parent="model_b", mode="d_cons")
_exp("loss_dfeat", 2, "density + feature consistency", parent="model_b", mode="d_feat")
_exp("rob_corrupt_p1", 2, "corruption augmentation (phase 1)", parent="model_b", mode="corrupt_aug")
_exp("rob_adv_p1", 2, "adversarial training (phase 1)", parent="model_b", mode="adv")
_exp("rob_clean", 3, "clean training", parent="loss_dc", mode="clean_dc")
_exp("rob_corrupt", 3, "corruption augmentation", parent="rob_corrupt_p1", mode="corrupt_aug")
_exp("rob_adv", 3, "adversarial training", parent="rob_adv_p1", mode="adv")
_exp("rob_corrupt_adv", 3, "corruption + adversarial training", parent="rob_corrupt_p1", mode="adv")

# main ablation path first, so the core table fills up before the additional ablations
EXPERIMENT_ORDER = (["baseline", "model_a", "model_b", "model_c", "rmsc_full"] + list(DILATION_EXPS)
                    + ["fusion_concat", "fusion_adaptive", "loss_d", "loss_dc", "loss_dcons", "loss_dfeat",
                       "rob_corrupt_p1", "rob_adv_p1", "rob_clean", "rob_corrupt", "rob_adv", "rob_corrupt_adv"])
assert sorted(EXPERIMENT_ORDER) == sorted(EXPERIMENTS)


def exp_dir(name, seed):
    d = os.path.join(STUDY_DIR, f"{name}_seed{seed}")
    os.makedirs(d, exist_ok=True)
    return d


def exp_best_path(name, seed):
    return os.path.join(exp_dir(name, seed), "best.pth")


def exp_signature(name):
    s = EXPERIMENTS[name]
    return json.dumps({"stage": s["stage"], "model": s["model"], "parent": s["parent"], "mode": s["mode"]},
                      sort_keys=True, default=list)


def exp_done_info(name, seed):
    p = os.path.join(exp_dir(name, seed), "done.json")
    return json.load(open(p)) if os.path.exists(p) else None


def exp_parent_id(name, seed):
    s = EXPERIMENTS[name]
    if s["stage"] == 1:
        return "imagenet" if cfg.pretrained_backbone else "random-init"
    info = exp_done_info(s["parent"], seed) if exp_is_done(s["parent"], seed) else None
    return info["best_ckpt_id"] if info else None


def exp_is_done(name, seed):
    info = exp_done_info(name, seed)
    return bool(info) and info.get("signature") == exp_signature(name) and \
        info.get("parent_id") == exp_parent_id(name, seed) and os.path.exists(exp_best_path(name, seed))


def early_stop_generic(vals, patience, min_epochs):
    if not cfg.early_stop or len(vals) < max(min_epochs, 1):
        return False, ""
    k = max(cfg.val_mae_smoothing, 1)
    smoothed = [float(np.mean(vals[max(0, i - k + 1):i + 1])) for i in range(len(vals))]
    best_i, best = 0, smoothed[0]   # same rule as early_stop_status (Section 10)
    for i, v in enumerate(smoothed):
        if v < best - cfg.early_stop_min_delta:
            best_i, best = i, v
    since = len(smoothed) - 1 - best_i
    if since >= patience:
        return True, f"smoothed val MAE not improved for {since} epochs (best at epoch {best_i + 1})"
    return False, ""


def train_one_epoch_robust(model, loader, optimizer, scaler, mode, epoch, rng, tag=""):
    m = MODES[mode]
    sup = CrowdCountingLoss(cfg.lambda_mse, cfg.lambda_ssim, cfg.lambda_count if m.get("count") else 0.0).to(DEVICE)
    amp_on = cfg.amp and DEVICE.type == "cuda"
    model.train()
    running = {k: 0.0 for k in ("loss", "mae", "mse", "ssim", "count")}
    n, n_bad = 0, 0
    for imgs, dens, _ in tqdm(loader, desc=f"{tag} ep{epoch} [{mode}]", leave=False):
        imgs = imgs.to(DEVICE, non_blocking=True)
        dens = dens.to(DEVICE, non_blocking=True)
        B = imgs.size(0)
        view = None
        if m.get("view") == "corrupt":
            view = random_corrupt_batch(imgs, rng)
        elif m.get("view") == "adv":
            view = pgd_attack(model, imgs, dens, cfg.adv_eps, alpha=cfg.adv_alpha,
                              steps=cfg.adv_train_steps, tiled=False)
        x = imgs if view is None else torch.cat([imgs, view], dim=0)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=amp_on):
            out = model.forward_features(x) if m.get("feat") else {"density": model(x)}
        pred = out["density"].float()
        loss, parts = sup(pred[:B], dens)
        if view is not None:
            pv = pred[B:]
            if m.get("sup_view"):
                lv, _ = sup(pv, dens)
                loss = 0.5 * (loss + lv)
            if m.get("cons"):
                loss = loss + cfg.lambda_cons * F.mse_loss(pv, pred[:B].detach())
            if m.get("feat"):
                f = out["daa"].float()
                loss = loss + cfg.lambda_feat * (1 - F.cosine_similarity(f[B:], f[:B].detach(), dim=1)).mean()
        optimizer.zero_grad(set_to_none=True)
        if not torch.isfinite(loss):
            n_bad += 1
            if n_bad >= 20:
                raise RuntimeError(f"20 non-finite batches in {tag} epoch {epoch}")
            continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()
        with torch.no_grad():
            mae, _ = compute_mae_mse(pred[:B].detach(), dens.float(), cfg.density_scale)
        running["loss"] += loss.item() * B
        running["mae"] += mae.sum().item()
        for k in ("mse", "ssim", "count"):
            running[k] += parts[k] * B
        n += B
    if n_bad:
        print(f"  {tag} epoch {epoch}: skipped {n_bad} non-finite batch(es).")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return {k: v / max(n, 1) for k, v in running.items()}


def _budget_exhausted(budget):
    if cfg.abl_epochs_per_session and budget["epochs"] >= cfg.abl_epochs_per_session:
        return True
    return bool(cfg.max_session_hours) and (time.time() - budget["t0"]) / 3600 >= cfg.max_session_hours


def train_experiment(name, seed, budget):
    """Train one experiment to completion (or until the session budget runs out). Resumable."""
    spec = EXPERIMENTS[name]
    if exp_is_done(name, seed):
        return "done"
    pid = exp_parent_id(name, seed)
    if pid is None:
        return "waiting for " + spec["parent"]
    d, sig, stage = exp_dir(name, seed), exp_signature(name), spec["stage"]
    last, prev, best_p = os.path.join(d, "last.pth"), os.path.join(d, "last_prev.pth"), exp_best_path(name, seed)

    set_seed(seed + 1000 * (stage - 1))
    model = build_variant(spec["model"], pretrained=(stage == 1 and cfg.pretrained_backbone)).to(DEVICE)
    if stage == 1:
        max_epochs, patience, min_ep, warmup = cfg.abl_s1_max_epochs, cfg.early_stop_patience, cfg.min_epochs, cfg.warmup_epochs
        optimizer = build_optimizer(model)
    else:
        parent_ck = torch.load(exp_best_path(spec["parent"], seed), map_location=DEVICE, weights_only=False)
        model.load_state_dict(parent_ck["model_state"])
        max_epochs = cfg.abl_s2_epochs if stage == 2 else cfg.abl_s3_epochs
        patience, min_ep, warmup = cfg.abl_s23_patience, cfg.abl_s23_min_epochs, 0
        optimizer = build_optimizer(model)
        mult = cfg.abl_s2_lr_mult if stage == 2 else cfg.abl_s3_lr_mult
        for pg in optimizer.param_groups:
            pg["group_base_lr"] = pg["group_base_lr"] * mult
            pg["lr"] = pg["group_base_lr"]
    scheduler = build_scheduler(optimizer)
    scaler = torch.amp.GradScaler("cuda", enabled=(cfg.amp and DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))
    crit = CrowdCountingLoss(cfg.lambda_mse, cfg.lambda_ssim, cfg.lambda_count).to(DEVICE)

    history, best, start = {k: [] for k in HISTORY_KEYS}, float("inf"), 1
    for path in (last, prev):
        if not os.path.exists(path):
            continue
        try:
            st = torch.load(path, map_location=DEVICE, weights_only=False)
        except Exception as e:
            print(f"  could not read {path} ({e})")
            continue
        if st.get("signature") == sig and st.get("parent_id") == pid:
            model.load_state_dict(st["model_state"]); optimizer.load_state_dict(st["optimizer_state"])
            scheduler.load_state_dict(st["scheduler_state"]); scaler.load_state_dict(st["scaler_state"])
            random.setstate(st["rng"]["python"]); np.random.set_state(st["rng"]["numpy"])
            torch.set_rng_state(st["rng"]["torch"].cpu())
            if st["rng"]["cuda"] is not None and torch.cuda.is_available():
                torch.cuda.set_rng_state_all([s.cpu() for s in st["rng"]["cuda"]])
            history, best, start = st["history"], st["best_val_mae"], st["epoch"] + 1
            print(f"  resumed {name} (seed {seed}) from {os.path.basename(path)}: {st['epoch']} epoch(s) done")
        else:
            print(f"  {name}: saved progress belongs to an older setup/parent -- restarting.")
        break

    stopped, reason = early_stop_generic(history["val_mae"], patience, min_ep)
    epoch = start - 1
    for epoch in range(start, max_epochs + 1):
        if stopped:
            break
        if _budget_exhausted(budget):
            return "paused"
        t0 = time.time()
        if stage == 1:
            if epoch <= warmup:
                set_warmup_lr(optimizer, epoch, warmup, cfg.base_lr)
            tr = train_one_epoch(model, train_loader, optimizer, crit, scaler, DEVICE, cfg, epoch)
        else:
            rng = np.random.RandomState((seed * 100003 + stage * 1009 + epoch) % (2 ** 32))
            tr = train_one_epoch_robust(model, train_loader, optimizer, scaler, spec["mode"], epoch, rng, tag=name)
        va = validate(model, val_loader, crit, DEVICE, cfg, desc=f"{name} val")
        if not (stage == 1 and epoch <= warmup):
            k = max(cfg.val_mae_smoothing, 1)
            scheduler.step(float(np.mean(history["val_mae"][-(k - 1):] + [va["mae"]]) if k > 1 else va["mae"]))
        for key, val in (("train_loss", tr["loss"]), ("val_loss", va["loss"]), ("train_mae", tr["mae"]),
                         ("val_mae", va["mae"]), ("lr", optimizer.param_groups[0]["lr"]),
                         ("train_mse", tr["mse"]), ("train_ssim", tr["ssim"]), ("train_count", tr["count"]),
                         ("val_mse", va["mse"]), ("val_ssim", va["ssim"]), ("val_count", va["count"])):
            history[key].append(val)
        if va["mae"] < best:
            best = va["mae"]
            atomic_save({"epoch": epoch, "val_mae": va["mae"], "ckpt_id": uuid.uuid4().hex,
                         "model_state": model.state_dict(), "spec": spec, "seed": seed}, best_p)
        if os.path.exists(last):
            shutil.copyfile(last, prev)
        atomic_save({"epoch": epoch, "signature": sig, "parent_id": pid, "model_state": model.state_dict(),
                     "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),
                     "scaler_state": scaler.state_dict(), "history": history, "best_val_mae": best,
                     "rng": {"python": random.getstate(), "numpy": np.random.get_state(),
                             "torch": torch.get_rng_state(),
                             "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}}, last)
        n_ep = len(history["val_mae"])
        _atomic_write_text(os.path.join(d, "history.csv"), pd.DataFrame(
            {"epoch": range(1, n_ep + 1), **{k: history[k][:n_ep] for k in HISTORY_KEYS}}).to_csv(index=False))
        budget["epochs"] += 1
        print(f"  [{name} s{seed}] stage {stage} epoch {epoch}/{max_epochs}  train_mae={tr['mae']:.2f}  "
              f"val_mae={va['mae']:.2f}  best={best:.2f}  lr={optimizer.param_groups[0]['lr']:.2e}  "
              f"{time.time() - t0:.0f}s")
        stopped, reason = early_stop_generic(history["val_mae"], patience, min_ep)
        if stopped:
            print(f"  [{name}] early stopping: {reason}")

    if not os.path.exists(best_p):
        return "no checkpoint"
    best_ck = torch.load(best_p, map_location="cpu", weights_only=False)
    info = {"signature": sig, "parent_id": pid, "best_ckpt_id": best_ck["ckpt_id"], "best_epoch": best_ck["epoch"],
            "best_val_mae": float(best_ck["val_mae"]), "epochs_run": len(history["val_mae"]),
            "stopped_early": bool(stopped), "reason": reason, "finished": _now()}
    _atomic_write_text(os.path.join(d, "done.json"), json.dumps(info, indent=2))
    del model, optimizer
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return "done"


def study_status():
    rows = []
    for seed in cfg.abl_seeds:
        for name in EXPERIMENT_ORDER:
            info = exp_done_info(name, seed) if exp_is_done(name, seed) else None
            last = os.path.join(STUDY_DIR, f"{name}_seed{seed}", "history.csv")
            n_ep = len(pd.read_csv(last)) if (not info and os.path.exists(last)) else None
            rows.append({"experiment": name, "seed": seed, "label": EXPERIMENTS[name]["label"],
                         "stage": EXPERIMENTS[name]["stage"],
                         "state": "done" if info else (f"in progress ({n_ep} ep)" if n_ep else "not started"),
                         "best_val_mae": info["best_val_mae"] if info else None,
                         "best_epoch": info["best_epoch"] if info else None})
    return pd.DataFrame(rows)


print(f"{len(EXPERIMENTS)} experiments registered: {', '.join(EXPERIMENT_ORDER)}")


## 20. Train the study's experiments (resumable, on Drive)

This runs every experiment in `EXPERIMENT_ORDER` (or `cfg.abl_experiments`) for every seed in
`cfg.abl_seeds`. Finished experiments are skipped and an interrupted one resumes from its next
epoch. When the session budget (`cfg.abl_epochs_per_session` / `cfg.max_session_hours`) is used up,
the cell stops cleanly and **Run all** continues next time. This cell waits until the main run
(Section 10) has finished, so it never competes with it.


In [ ]:
STUDY_BUDGET = {"epochs": 0, "t0": time.time()}
if not cfg.run_rmsc_study:
    print("cfg.run_rmsc_study = False -- not training; the analysis below uses what is on Drive.")
elif not globals().get("TRAINING_FINISHED", True):
    print("The main run (Section 10) is not finished yet -- the study starts after it. Run all again.")
else:
    names = [n for n in EXPERIMENT_ORDER if not cfg.abl_experiments or n in cfg.abl_experiments]
    paused = False
    for seed in cfg.abl_seeds:
        for name in names:
            print(f"--- {name} (seed {seed}): {EXPERIMENTS[name]['label']}")
            state = train_experiment(name, seed, STUDY_BUDGET)
            print(f"    -> {state}")
            if state == "paused":
                paused = True
                break
        if paused:
            print("\nSession budget used up -- progress is saved on Drive. Run all again to continue.")
            break

status_df = study_status()
save_csv(status_df, "rmsc_study_status.csv", index=False)
print(status_df.to_string(index=False))


## 21. Robustness evaluation

Every trained model (the main run plus all finished experiments) is evaluated on the **test set**
with its best (validation-selected) checkpoint. Per-image predictions are cached on Drive next to
each checkpoint (`eval/<name>_test.csv`), one column per protocol. Cached columns are reused, a
new checkpoint invalidates the cache, and an interrupted evaluation continues where it stopped.

| Column | Protocol |
|---|---|
| `cleantta` | clean image, overlapping tiles + flip TTA (`cfg.use_tta`), same as Section 11 |
| `clean` | clean image without TTA (reference for every perturbed protocol below) |
| `corr:<type>:<s>` | one of the 12 corruptions at severity s |
| `adv:<fgsm/pgd/bim>:<e>` | white-box attack with ε = e/255 |
| `scale:<s>` | image rescaled by s; the true count is unchanged |
| `occ:<p>` | grey rectangles hide a fraction p of the image; scored against the visible people only (`gtocc:<p>`) |

The ablation columns (Section 23/24) are clean, corruptions at `cfg.abl_corruption_severities`, and
PGD at `cfg.adv_eps`. The models in `cfg.robust_full_models` additionally get the full sweep: 12 x 5
corruptions, FGSM/PGD/BIM at `cfg.robust_eps_255`, scales, occlusion, and cross-part generalisation
(a Part A model evaluated on the Part B test set, and vice versa).

**Robustness degradation**: RD = (MAE_perturbed - MAE_clean) / MAE_clean x 100, with both terms
computed without TTA.


In [ ]:
_EVAL_MODEL_CACHE = {}


def load_eval_model(name, seed=None):
    """Best checkpoint of `name` (an experiment or the main run) as an eval-mode model, or (None, None)."""
    seed = cfg.abl_seeds[0] if seed is None else seed
    key = (name, seed)
    if name == MAIN_NAME:
        if cfg.model_type != "light" or not os.path.exists(best_ckpt_path):
            return None, None
        ck = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
        spec, ckpt_id = MAIN_MODEL_SPEC, f"main@{ck['epoch']}:{float(ck['val_mae']):.6f}"
    else:
        if not exp_is_done(name, seed):
            return None, None
        ck = torch.load(exp_best_path(name, seed), map_location=DEVICE, weights_only=False)
        spec, ckpt_id = EXPERIMENTS[name]["model"], ck["ckpt_id"]
    if key in _EVAL_MODEL_CACHE and _EVAL_MODEL_CACHE[key][1]["ckpt_id"] == ckpt_id:
        return _EVAL_MODEL_CACHE[key]
    m = build_variant(spec, pretrained=False).to(DEVICE)
    m.load_state_dict(ck["model_state"])
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)  # attacks/saliency only need gradients w.r.t. the input
    info = {"ckpt_id": ckpt_id, "spec": spec, "epoch": int(ck["epoch"]), "val_mae": float(ck["val_mae"])}
    _EVAL_MODEL_CACHE.clear()  # keep at most one model in memory
    _EVAL_MODEL_CACHE[key] = (m, info)
    return m, info


def model_label(name):
    return "LightARFS (main run)" if name == MAIN_NAME else EXPERIMENTS[name]["label"]


def available_models(names, seed=None):
    seed = cfg.abl_seeds[0] if seed is None else seed
    return [n for n in names if (n == MAIN_NAME and cfg.model_type == "light" and os.path.exists(best_ckpt_path))
            or (n in EXPERIMENTS and exp_is_done(n, seed))]


def _tp(model, x, tta=False):
    return tiled_predict(model, x, cfg.patch_size, cfg.output_stride, DEVICE, amp=cfg.amp, tta=tta,
                         chunk_size=cfg.batch_size)


def occlusion_mask(h, w, frac, seed):
    """1 = visible, 0 = hidden by grey rectangles covering at least `frac` of the image."""
    rng = np.random.RandomState(seed % (2 ** 32))
    mask = np.ones((h, w), np.float32)
    while 1 - mask.mean() < frac:
        rh, rw = int(rng.uniform(0.05, 0.25) * h) + 1, int(rng.uniform(0.05, 0.25) * w) + 1
        y, x = rng.randint(0, max(1, h - rh)), rng.randint(0, max(1, w - rw))
        mask[y:y + rh, x:x + rw] = 0
    return mask


def _n_eval(dataset):
    return len(dataset) if not cfg.robust_eval_max_images else min(len(dataset), cfg.robust_eval_max_images)


def compute_column(model, dataset, col, n):
    kind, *args = col.split(":")
    preds, extra = [], []
    for idx in tqdm(range(n), desc=col, leave=False):
        img, dens, _ = dataset[idx]
        img, dens = img.unsqueeze(0).to(DEVICE), dens.unsqueeze(0).to(DEVICE)
        if kind in ("clean", "cleantta"):
            pred = _tp(model, img, tta=(kind == "cleantta" and cfg.use_tta))
        elif kind == "corr":
            pred = _tp(model, corrupt_image(img, args[0], int(args[1]), eval_seed(idx, col)))
        elif kind == "adv":
            gen = torch.Generator(device=img.device)
            gen.manual_seed(eval_seed(idx, col))
            pred = _tp(model, ATTACKS[args[0]](model, img, dens, float(args[1]) / 255, generator=gen))
        elif kind == "scale":
            s, (H, W) = float(args[0]), img.shape[-2:]
            st = cfg.output_stride
            size = (max(st, int(round(H * s / st)) * st), max(st, int(round(W * s / st)) * st))
            pred = _tp(model, F.interpolate(img, size=size, mode="bilinear", align_corners=False))
        elif kind == "occ":
            H, W = img.shape[-2:]
            mask = torch.from_numpy(occlusion_mask(H, W, float(args[0]), eval_seed(idx, col))).to(DEVICE)[None, None]
            pred = _tp(model, img * mask)      # 0 in normalised space = ImageNet-mean grey
            visible = F.avg_pool2d(mask, cfg.output_stride)
            extra.append((dens * visible).sum().item() / cfg.density_scale)
        else:
            raise ValueError(col)
        preds.append(pred.sum().item() / cfg.density_scale)
    return preds, extra


def model_ckpt_id(name, seed):
    """Identity of the checkpoint that would be evaluated (cheap: no model is built)."""
    if name == MAIN_NAME:
        if cfg.model_type != "light" or not os.path.exists(best_ckpt_path):
            return None
        ck = torch.load(best_ckpt_path, map_location="cpu", weights_only=False)
        return f"main@{ck['epoch']}:{float(ck['val_mae']):.6f}"
    return exp_done_info(name, seed)["best_ckpt_id"] if exp_is_done(name, seed) else None


_EVAL_DF_CACHE = {}


def ensure_eval(name, columns, seed=None, dataset=None, tag="test", verbose=True):
    """Per-image predictions of `name` for every protocol in `columns` (computed only if missing)."""
    seed = cfg.abl_seeds[0] if seed is None else seed
    dataset = test_dataset if dataset is None else dataset
    ckpt_id = model_ckpt_id(name, seed)
    if ckpt_id is None:
        return None
    base = cfg.ckpt_dir if name == MAIN_NAME else exp_dir(name, seed)
    os.makedirs(os.path.join(base, "eval"), exist_ok=True)
    path = os.path.join(base, "eval", f"{name}_{tag}.csv")
    meta_path, n = path + ".json", _n_eval(dataset)
    key = (path, ckpt_id, n)
    df = _EVAL_DF_CACHE.get(key)
    if df is None:
        meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {}
        if os.path.exists(path) and meta.get("ckpt_id") == ckpt_id and meta.get("n") == n:
            df = pd.read_csv(path)
        else:
            df = pd.DataFrame({"image": [os.path.basename(dataset_image_path(dataset, i)) for i in range(n)],
                               "gt": [dataset[i][1].sum().item() / cfg.density_scale for i in range(n)]})
    missing = [c for c in columns if c not in df.columns]
    if missing:
        model, info = load_eval_model(name, seed)
        for col in missing:
            if verbose:
                print(f"  [{name}] {col}")
            preds, extra = compute_column(model, dataset, col, n)
            df[col] = preds
            if extra:
                df["gt" + col] = extra          # e.g. "gtocc:0.2"
            _atomic_write_text(path, df.to_csv(index=False))
            _atomic_write_text(meta_path, json.dumps({"ckpt_id": ckpt_id, "n": n}))
    _EVAL_DF_CACHE[key] = df
    return df


def abs_err(df, col):
    gt = df["gt" + col] if ("gt" + col) in df.columns else df["gt"]
    return (df[col] - gt).abs()


def corr_cols(sevs):
    return [f"corr:{t}:{s}" for t in CORRUPTION_TYPES for s in sevs]


PGD_COL = f"adv:pgd:{int(round(cfg.adv_eps * 255))}"
ABL_COLUMNS = ["cleantta", "clean"] + corr_cols(cfg.abl_corruption_severities) + [PGD_COL]


def per_image_errors(df, metric):
    """Per-image absolute errors for metric 'clean' (with TTA), 'clean_notta', 'corruption' (mean
    over the ablation corruptions) or 'pgd'."""
    if metric == "clean":
        return abs_err(df, "cleantta").to_numpy()
    if metric == "clean_notta":
        return abs_err(df, "clean").to_numpy()
    if metric == "corruption":
        return np.mean([abs_err(df, c).to_numpy() for c in corr_cols(cfg.abl_corruption_severities)], axis=0)
    if metric == "pgd":
        return abs_err(df, PGD_COL).to_numpy()
    raise ValueError(metric)


def summarize_model(name, seed=None):
    df = ensure_eval(name, ABL_COLUMNS, seed=seed, verbose=False)
    if df is None:
        return None
    spec = MAIN_MODEL_SPEC if name == MAIN_NAME else EXPERIMENTS[name]["model"]
    cx = variant_complexity(spec)
    err = df["cleantta"] - df["gt"]
    clean_nt = per_image_errors(df, "clean_notta").mean()
    corr = per_image_errors(df, "corruption").mean()
    return {"MAE": float(err.abs().mean()), "RMSE": float(np.sqrt((err ** 2).mean())),
            "Params (M)": cx["params"] / 1e6, "GFLOPs": cx["GFLOPs"],
            "clean MAE (no TTA)": float(clean_nt), "Corr. MAE": float(corr),
            "Corr. RD %": float((corr - clean_nt) / max(clean_nt, 1e-8) * 100),
            "PGD MAE": float(per_image_errors(df, "pgd").mean()),
            "PGD RD %": float((per_image_errors(df, "pgd").mean() - clean_nt) / max(clean_nt, 1e-8) * 100)}


if cfg.run_robust_eval:
    names = available_models([MAIN_NAME] + EXPERIMENT_ORDER)
    print(f"Ablation protocols ({len(ABL_COLUMNS)} columns) for {len(names)} model(s): {names}")
    for seed in cfg.abl_seeds:
        for name in names if seed == cfg.abl_seeds[0] else available_models(EXPERIMENT_ORDER, seed):
            ensure_eval(name, ABL_COLUMNS, seed=seed)
    rows = [{"model": n, "label": model_label(n), **summarize_model(n)} for n in names]
    if rows:
        robust_summary = pd.DataFrame(rows)
        save_csv(robust_summary, "robustness_summary_all_models.csv", index=False)
        print(robust_summary.round(2).to_string(index=False))
else:
    print("cfg.run_robust_eval = False -- skipped.")


In [ ]:
# ---- full robustness sweep for cfg.robust_full_models ----
EPS_LIST = list(cfg.robust_eps_255)
FULL_COLUMNS = (["cleantta", "clean"] + corr_cols(range(1, 6))
                + [f"adv:{a}:{e}" for a in ATTACKS for e in EPS_LIST]
                + [f"scale:{s:g}" for s in cfg.robust_scales] + [f"occ:{p:g}" for p in cfg.robust_occlusions])
FULL_MODELS = available_models(cfg.robust_full_models) if cfg.run_robust_eval else []
full_eval = {n: ensure_eval(n, FULL_COLUMNS) for n in FULL_MODELS}
print("Full sweep done for:", FULL_MODELS or "none (no trained model in cfg.robust_full_models yet)")

if full_eval:
    # corruption: MAE per type x severity, RD per type
    corr_rows = []
    for n, df in full_eval.items():
        clean_mae = abs_err(df, "clean").mean()
        for t in CORRUPTION_TYPES:
            for s in range(1, 6):
                m = abs_err(df, f"corr:{t}:{s}").mean()
                corr_rows.append({"model": n, "corruption": t, "severity": s, "MAE": m,
                                  "RD %": (m - clean_mae) / max(clean_mae, 1e-8) * 100})
    corr_df = pd.DataFrame(corr_rows)
    save_csv(corr_df, "robust_corruption_full.csv", index=False)
    fig, axes = plt.subplots(1, len(full_eval), figsize=(7 * len(full_eval), 6), squeeze=False)
    vmax = corr_df["MAE"].max()
    for ax, n in zip(axes[0], full_eval):
        piv = corr_df[corr_df.model == n].pivot(index="corruption", columns="severity", values="MAE").loc[CORRUPTION_TYPES]
        im = ax.imshow(piv.values, cmap="YlOrRd", vmin=0, vmax=vmax, aspect="auto")
        ax.set_xticks(range(5)); ax.set_xticklabels([f"s{s}" for s in range(1, 6)])
        ax.set_yticks(range(len(CORRUPTION_TYPES))); ax.set_yticklabels(CORRUPTION_TYPES)
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                ax.text(j, i, f"{piv.values[i, j]:.0f}", ha="center", va="center", fontsize=8)
        ax.set_title(f"{model_label(n)}: corrupted MAE")
    fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8)
    save_fig(fig, "robust_corruption_heatmaps.png"); plt.show()

    rd = corr_df.groupby(["model", "corruption"])["RD %"].mean().unstack(0).loc[CORRUPTION_TYPES]
    fig, ax = plt.subplots(figsize=(14, 4))
    rd.rename(columns=model_label).plot(kind="bar", ax=ax, edgecolor="black")
    ax.axhline(0, color="k", lw=0.8); ax.set_ylabel("RD % (mean over severities)")
    ax.set_title("Robustness degradation per corruption type"); plt.xticks(rotation=40, ha="right")
    plt.tight_layout(); save_fig(fig, "robust_corruption_rd.png"); plt.show()

    # adversarial: MAE vs eps per attack
    adv_rows = [{"model": n, "attack": a, "eps_255": e, "MAE": abs_err(df, f"adv:{a}:{e}").mean(),
                 "clean MAE": abs_err(df, "clean").mean()} for n, df in full_eval.items() for a in ATTACKS for e in EPS_LIST]
    adv_df = pd.DataFrame(adv_rows)
    save_csv(adv_df, "robust_adversarial_full.csv", index=False)
    fig, axes = plt.subplots(1, len(ATTACKS), figsize=(6 * len(ATTACKS), 4), squeeze=False)
    for ax, a in zip(axes[0], ATTACKS):
        for n in full_eval:
            sub = adv_df[(adv_df.model == n) & (adv_df.attack == a)]
            ax.plot([0] + list(sub.eps_255), [sub["clean MAE"].iloc[0]] + list(sub.MAE), marker="o", label=model_label(n))
        ax.set_title(a.upper()); ax.set_xlabel("ε (x/255)"); ax.set_ylabel("MAE"); ax.legend(fontsize=8)
    plt.tight_layout(); save_fig(fig, "robust_adversarial_curves.png"); plt.show()

    # scale and occlusion
    sc_rows, oc_rows = [], []
    for n, df in full_eval.items():
        sc_rows.append({"model": n, "scale": 1.0, "MAE": abs_err(df, "clean").mean()})
        sc_rows += [{"model": n, "scale": s, "MAE": abs_err(df, f"scale:{s:g}").mean()} for s in cfg.robust_scales]
        oc_rows.append({"model": n, "occluded": 0.0, "MAE": abs_err(df, "clean").mean()})
        oc_rows += [{"model": n, "occluded": p, "MAE": abs_err(df, f"occ:{p:g}").mean()} for p in cfg.robust_occlusions]
    scale_df = pd.DataFrame(sc_rows).sort_values(["model", "scale"])
    occ_df = pd.DataFrame(oc_rows).sort_values(["model", "occluded"])
    save_csv(scale_df, "robust_scale.csv", index=False); save_csv(occ_df, "robust_occlusion.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for n in full_eval:
        s = scale_df[scale_df.model == n]; axes[0].plot(s.scale, s.MAE, marker="o", label=model_label(n))
        o = occ_df[occ_df.model == n]; axes[1].plot(o.occluded, o.MAE, marker="o", label=model_label(n))
    axes[0].set_title("Scale robustness (count unchanged)"); axes[0].set_xlabel("image scale"); axes[0].set_ylabel("MAE")
    axes[1].set_title("Occlusion (scored on visible people)"); axes[1].set_xlabel("fraction of image hidden"); axes[1].set_ylabel("MAE")
    for ax in axes:
        ax.legend(fontsize=8)
    plt.tight_layout(); save_fig(fig, "robust_scale_occlusion.png"); plt.show()

    # cross-part generalisation (Part A model on Part B test and vice versa)
    other = {"A": "B", "B": "A"}.get(cfg.part)
    xpart_rows = []
    if other:
        try:
            _, _, o_img_dir, o_gt_dir = resolve_part_dirs(DATASET_ROOT, other)
            o_paths = sorted(glob.glob(os.path.join(o_img_dir, "*.jpg")))
            o_dens = os.path.join(cfg.density_cache_root, f"part_{other}_test")
            precompute_density_maps(o_paths, o_gt_dir, o_dens, part=other)
            o_ds = ShanghaiTechDataset(o_paths, o_dens, cfg.patch_size, cfg.output_stride, cfg.density_scale, train=False)
            for n in full_eval:
                df = ensure_eval(n, ["cleantta"], dataset=o_ds, tag=f"part{other}_test")
                e = df["cleantta"] - df["gt"]
                xpart_rows.append({"model": n, "trained_on": f"part_{cfg.part}", "tested_on": f"part_{other}",
                                   "MAE": e.abs().mean(), "RMSE": float(np.sqrt((e ** 2).mean())),
                                   "in-domain MAE": abs_err(full_eval[n], "cleantta").mean()})
        except Exception as e:
            print(f"Cross-part evaluation skipped ({e})")
    xpart_df = pd.DataFrame(xpart_rows)
    if len(xpart_df):
        save_csv(xpart_df, "robust_cross_part.csv", index=False)
        print(xpart_df.round(2).to_string(index=False))
    update_summary("robustness_full", {n: {"clean_MAE": float(abs_err(df, "cleantta").mean()),
                                           "mean_corr_MAE": float(corr_df[corr_df.model == n].MAE.mean()),
                                           "mean_corr_RD_%": float(corr_df[corr_df.model == n]["RD %"].mean()),
                                           **{f"{a}@{e}/255": float(adv_df[(adv_df.model == n) & (adv_df.attack == a) & (adv_df.eps_255 == e)].MAE.iloc[0])
                                              for a in ATTACKS for e in EPS_LIST}} for n, df in full_eval.items()})


## 22. Explainable AI analysis

For each model in `cfg.xai_models` (default: Model B, clean training, vs RMSC-Net; the main model is
used if neither is trained yet) and for `cfg.xai_n_samples` test images (sparse / medium / dense),
the analysis shows a side-by-side comparison **Clean <-> Corrupted <-> Adversarial**. The rows are
the clean image, the image under `cfg.xai_corruption`, and the image under PGD at ε =
`cfg.adv_eps`. The corrupted and adversarial images are exactly the ones scored in Section 21. The
columns are:

| Panel | What it is |
|---|---|
| Image, GT density | input and ground truth (GT repeated on each row) |
| Predicted density | the scored prediction (overlapping tiles), with its count |
| Error map | predicted - GT density (red = over-count, blue = under-count) |
| DAA spatial attention | per-pixel spatial attention weight (sigmoid, 0-1) |
| Reliability map | output of the reliability gate (only models with DAA "full") |
| Encoder f8 / f16 / f32 | channel-mean absolute activation of the MobileNetV2 features at strides 8 / 16 / 32 |
| Grad x Input | gradient of the predicted count w.r.t. the input, times the input |
| Integrated gradients | IG with a grey (ImageNet-mean) baseline, `cfg.xai_ig_steps` steps; completeness gap printed |

A second figure shows the **channel response**: the ECA channel-attention weights and the mean
activation per channel of the features entering the decoder, for the three views.

Attention and feature maps come from one forward pass over the whole image. The predicted density
and the attributions use the tiled inference that is scored.

**Stability analysis.** For `cfg.xai_stability_images` test images, every corruption (severity
`cfg.xai_stability_severity`) and PGD are compared with the clean view:
- Pearson correlation of the density, spatial-attention and reliability maps;
- cosine similarity of the ECA channel weights;
- linear CKA of the encoder features (f8, f16, f32) and the decoder-input features;
- the count shift.

Values near 1 mean the representation is unchanged by the perturbation. Section 25 tests
RMSC-Net against Model B on these per-image values.


In [ ]:
def xai_forward(model, img):
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(cfg.amp and DEVICE.type == "cuda")):
        maps = model.forward_features(img)
    return {k: (v.float() if torch.is_tensor(v) else v) for k, v in maps.items()}


def grad_x_input(model, img):
    x = img.clone().detach().requires_grad_(True)
    g, = torch.autograd.grad(tiled_forward(model, x).sum() / cfg.density_scale, x)
    return (g * x.detach()).sum(1)[0]                 # signed attribution per pixel (H, W)


def integrated_gradients(model, img, steps):
    baseline = torch.zeros_like(img)                  # 0 in normalised space = ImageNet-mean grey
    total = torch.zeros_like(img)
    for k in range(1, steps + 1):
        x = (baseline + k / steps * (img - baseline)).detach().requires_grad_(True)
        g, = torch.autograd.grad(tiled_forward(model, x).sum() / cfg.density_scale, x)
        total += g
    attr = ((img - baseline) * total / steps).sum(1)[0]
    with torch.no_grad():
        gap = (tiled_forward(model, img).sum() - tiled_forward(model, baseline).sum()).item() / cfg.density_scale
    return attr, attr.sum().item(), gap


def _feat_map(f):
    return f[0].abs().mean(0).cpu().numpy()


def _norm_abs(a, q=99):
    a = np.abs(a)
    return np.clip(a / (np.percentile(a, q) + 1e-8), 0, 1)


def xai_views(model, idx, img, dens):
    ctype, sev = cfg.xai_corruption
    gen = torch.Generator(device=img.device)
    gen.manual_seed(eval_seed(idx, PGD_COL))
    return [("Clean", img),
            (f"Corrupted ({ctype}, s{sev})", corrupt_image(img, ctype, sev, eval_seed(idx, f"corr:{ctype}:{sev}"))),
            (f"Adversarial (PGD {cfg.adv_eps * 255:.0f}/255)", pgd_attack(model, img, dens, cfg.adv_eps, generator=gen))]


def xai_figure(name, model, idx):
    img, dens, _ = test_dataset[idx]
    img, dens = img.unsqueeze(0).to(DEVICE), dens.unsqueeze(0).to(DEVICE)
    gt_map = dens[0, 0].cpu().numpy() / cfg.density_scale
    gt = gt_map.sum()
    views = xai_views(model, idx, img, dens)
    cols = ["Image", "GT density", "Predicted density", "Error (pred - GT)", "DAA spatial attention",
            "Reliability map", "Encoder f8", "Encoder f16", "Encoder f32", "Grad x Input", "Integrated gradients"]
    fig, axes = plt.subplots(len(views), len(cols), figsize=(3.1 * len(cols), 3.0 * len(views)))
    dmax = gt_map.max()
    channel = {}
    for r, (vname, x) in enumerate(views):
        pred = _tp(model, x)[0, 0].cpu().numpy() / cfg.density_scale
        maps = xai_forward(model, x)
        sal = grad_x_input(model, x).cpu().numpy()
        if cfg.xai_ig_steps:
            ig, ig_sum, ig_gap = integrated_gradients(model, x, cfg.xai_ig_steps)
            ig = ig.cpu().numpy()
            print(f"  {vname}: IG completeness  sum(attr)={ig_sum:.1f}  vs  count(x)-count(baseline)={ig_gap:.1f}")
        err = pred - gt_map
        emax = max(np.abs(err).max(), 1e-6)
        panels = [
            (to_uint8(x[0]), None, {}),
            (gt_map, "jet", dict(vmin=0, vmax=dmax)),
            (pred, "jet", dict(vmin=0, vmax=dmax)),
            (err, "bwr", dict(vmin=-emax, vmax=emax)),
            (maps["spatial"][0, 0].cpu().numpy() if maps["spatial"] is not None else None, "viridis", dict(vmin=0, vmax=1)),
            (maps["reliability"][0, 0].cpu().numpy() if maps["reliability"] is not None else None, "viridis", dict(vmin=0, vmax=1)),
            (_feat_map(maps["f8"]), "magma", {}), (_feat_map(maps["f16"]), "magma", {}), (_feat_map(maps["f32"]), "magma", {}),
            (_norm_abs(sal), "inferno", dict(vmin=0, vmax=1)),
            (_norm_abs(ig) if cfg.xai_ig_steps else None, "inferno", dict(vmin=0, vmax=1)),
        ]
        for c, (data, cmap, kw) in enumerate(panels):
            ax = axes[r, c]
            ax.axis("off")
            if data is None:
                ax.text(0.5, 0.5, "n/a for this model", ha="center", va="center", transform=ax.transAxes)
            else:
                ax.imshow(data, cmap=cmap, **kw)
            title = cols[c]
            if c == 2:
                title += f"\ncount {pred.sum():.0f} (GT {gt:.0f})"
            ax.set_title(title, fontsize=9)
        axes[r, 0].text(-0.08, 0.5, vname, rotation=90, ha="right", va="center", transform=axes[r, 0].transAxes, fontsize=10)
        channel[vname] = (None if maps["eca"] is None else maps["eca"][0].cpu().numpy(), maps["daa"][0].mean((1, 2)).cpu().numpy())
    fig.suptitle(f"{model_label(name)} -- test image {os.path.basename(dataset_image_path(test_dataset, idx))} (GT {gt:.0f})", fontsize=12)
    plt.tight_layout()
    save_fig(fig, f"xai_{name}_img{idx}.png")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(16, 3.5))
    for vname, (eca, act) in channel.items():
        if eca is not None:
            axes[0].plot(eca, marker=".", lw=1, label=vname)
        axes[1].plot(act, marker=".", lw=1, label=vname)
    axes[0].set_title("ECA channel-attention weights" if channel["Clean"][0] is not None else "ECA: n/a (no attention)")
    axes[1].set_title("Mean activation per channel (decoder input)")
    for ax in axes:
        ax.set_xlabel("channel"); ax.legend(fontsize=8)
    plt.tight_layout()
    save_fig(fig, f"xai_channels_{name}_img{idx}.png")
    plt.show()
    clean_act = channel["Clean"][1]
    for vname, (_, act) in list(channel.items())[1:]:
        cos = float(np.dot(clean_act, act) / (np.linalg.norm(clean_act) * np.linalg.norm(act) + 1e-12))
        print(f"  channel-response cosine similarity clean vs {vname}: {cos:.3f}")


def linear_cka(a, b):
    X = a.flatten(1).T.double(); Y = b.flatten(1).T.double()
    X = X - X.mean(0); Y = Y - Y.mean(0)
    num = (Y.T @ X).pow(2).sum()
    den = (X.T @ X).pow(2).sum().sqrt() * (Y.T @ Y).pow(2).sum().sqrt()
    return float(num / den) if den > 0 else float("nan")


def _pearson(a, b):
    a, b = a.flatten().double(), b.flatten().double()
    a, b = a - a.mean(), b - b.mean()
    den = a.norm() * b.norm()
    return float((a * b).sum() / den) if den > 0 else float("nan")


def _cos(a, b):
    return float(F.cosine_similarity(a.flatten().double(), b.flatten().double(), dim=0))


def stability_rows(name, model, idxs):
    eps = int(round(cfg.adv_eps * 255))
    perts = [("corr", t) for t in CORRUPTION_TYPES] + [("adv", "pgd")]
    rows = []
    for idx in tqdm(idxs, desc=f"stability {name}", leave=False):
        img, dens, _ = test_dataset[idx]
        img, dens = img.unsqueeze(0).to(DEVICE), dens.unsqueeze(0).to(DEVICE)
        m0, c0 = xai_forward(model, img), _tp(model, img).sum().item() / cfg.density_scale
        for kind, what in perts:
            if kind == "corr":
                sev = cfg.xai_stability_severity
                x = corrupt_image(img, what, sev, eval_seed(idx, f"corr:{what}:{sev}"))
                pname = what
            else:
                gen = torch.Generator(device=img.device)
                gen.manual_seed(eval_seed(idx, f"adv:pgd:{eps}"))
                x = pgd_attack(model, img, dens, cfg.adv_eps, generator=gen)
                pname = f"PGD {eps}/255"
            m1, c1 = xai_forward(model, x), _tp(model, x).sum().item() / cfg.density_scale
            rows.append({
                "model": name, "image_idx": idx, "perturbation": pname, "kind": kind,
                "count_clean": c0, "count_perturbed": c1, "count_shift": abs(c1 - c0),
                "density_corr": _pearson(m0["density"], m1["density"]),
                "spatial_corr": _pearson(m0["spatial"], m1["spatial"]) if m0["spatial"] is not None else np.nan,
                "spatial_mad": float((m0["spatial"] - m1["spatial"]).abs().mean()) if m0["spatial"] is not None else np.nan,
                "reliability_corr": _pearson(m0["reliability"], m1["reliability"]) if m0["reliability"] is not None else np.nan,
                "eca_cos": _cos(m0["eca"], m1["eca"]) if m0["eca"] is not None else np.nan,
                "cka_f8": linear_cka(m0["f8"][0], m1["f8"][0]), "cka_f16": linear_cka(m0["f16"][0], m1["f16"][0]),
                "cka_f32": linear_cka(m0["f32"][0], m1["f32"][0]), "cka_daa": linear_cka(m0["daa"][0], m1["daa"][0]),
            })
    return rows


STABILITY_METRICS = ["density_corr", "spatial_corr", "reliability_corr", "eca_cos", "cka_f8", "cka_f16", "cka_f32", "cka_daa"]
XAI_MODELS = available_models(cfg.xai_models) or available_models([MAIN_NAME])
stability_df = None
if not XAI_MODELS:
    print("[SKIP] no trained model available for the XAI analysis yet.")
else:
    gts = np.array([test_dataset[i][1].sum().item() / cfg.density_scale for i in range(len(test_dataset))])
    order = np.argsort(gts)
    qs = np.linspace(0.15, 0.9, cfg.xai_n_samples) if cfg.xai_n_samples > 1 else [0.5]
    sample_idx = [int(order[int(q * (len(order) - 1))]) for q in qs]
    for name in XAI_MODELS:
        model_x, _ = load_eval_model(name)
        for idx in sample_idx:
            print(f"{model_label(name)} | test image {idx} (GT {gts[idx]:.0f})")
            xai_figure(name, model_x, idx)

    n_stab = min(cfg.xai_stability_images, len(order))
    stab_idx = [int(order[int(i)]) for i in np.linspace(0, len(order) - 1, n_stab)]
    rows = []
    for name in XAI_MODELS:
        model_x, _ = load_eval_model(name)
        rows += stability_rows(name, model_x, stab_idx)
    stability_df = pd.DataFrame(rows)
    save_csv(stability_df, "xai_stability_per_image.csv", index=False)
    stability_df["group"] = np.where(stability_df["kind"] == "corr", "corruptions (mean)", stability_df["perturbation"])
    stab_summary = stability_df.groupby(["model", "group"])[STABILITY_METRICS + ["count_shift"]].agg(["mean", "std"])
    save_csv(stab_summary, "xai_stability_summary.csv")
    print("Representation stability, clean vs perturbed (1 = unchanged):")
    print(stab_summary.round(3).to_string())
    by_type = stability_df[stability_df.kind == "corr"].groupby(["model", "perturbation"])[STABILITY_METRICS].mean()
    save_csv(by_type, "xai_stability_by_corruption.csv")

    groups = list(stability_df["group"].unique())
    fig, axes = plt.subplots(1, len(groups), figsize=(8 * len(groups), 4), squeeze=False)
    for ax, g in zip(axes[0], groups):
        sub = stability_df[stability_df.group == g].groupby("model")[STABILITY_METRICS].mean()
        sub.rename(index=model_label).T.plot(kind="bar", ax=ax, edgecolor="black")
        ax.set_ylim(min(0, np.nanmin(sub.values) - 0.05), 1.05); ax.set_title(f"Stability under {g}")
        ax.set_ylabel("similarity to clean"); ax.tick_params(axis="x", rotation=40)
    plt.tight_layout(); save_fig(fig, "xai_stability.png"); plt.show()

    print("\nObservations (descriptive; Section 25 has the statistical tests):")
    for g in groups:
        sub = stability_df[stability_df.group == g].groupby("model")[STABILITY_METRICS].mean()
        for metric in STABILITY_METRICS:
            vals = sub[metric].dropna()
            if len(vals) == 0:
                continue
            txt = ", ".join(f"{model_label(m)} {v:.3f}" for m, v in vals.items())
            print(f"  {g:22s} {metric:17s}: {txt}")


## 23. Ablation study (component-wise)

Each row adds one component to the previous one. All models are evaluated on the same test images
with the same protocols:
- **MAE / RMSE**: clean test images, with TTA (as in Section 11).
- **Params** and **GFLOPs**: at a 1024x768 input.
- **Robustness**: corrupted MAE (mean over the 12 corruptions at `cfg.abl_corruption_severities`) and PGD MAE at ε = `cfg.adv_eps`, each with its degradation RD relative to clean MAE without TTA.

With several seeds (`cfg.abl_seeds`) the table shows mean ± SD over seeds. The main LightARFS run
is listed below the table as a reference.


In [ ]:
METRIC_COLS = ["MAE", "RMSE", "Params (M)", "GFLOPs", "Corr. MAE", "Corr. RD %", "PGD MAE", "PGD RD %"]


def summarize_over_seeds(name):
    if name == MAIN_NAME:
        s = summarize_model(name)
        return (s, None, 1) if s else (None, None, 0)
    per_seed = [summarize_model(name, seed) for seed in cfg.abl_seeds if exp_is_done(name, seed)]
    per_seed = [s for s in per_seed if s]
    if not per_seed:
        return None, None, 0
    df = pd.DataFrame(per_seed)
    return df.mean().to_dict(), (df.std(ddof=1).to_dict() if len(df) > 1 else None), len(df)


def result_table(rows, flag_cols=()):
    out = []
    for row in rows:
        label, name = row[0], row[1]
        rec = {"Configuration": label, **{c: ("✓" if f else "✗") for c, f in zip(flag_cols, row[2] if len(row) > 2 else ())}}
        mean, sd, n = summarize_over_seeds(name)
        if mean is None:
            rec.update({c: np.nan for c in METRIC_COLS}); rec["note"] = "not trained yet"
        else:
            for c in METRIC_COLS:
                rec[c] = mean[c]
                if sd is not None and c not in ("Params (M)", "GFLOPs"):
                    rec[c + " SD"] = sd[c]
            rec["note"] = f"{n} seed(s)"
        rec["experiment"] = name
        out.append(rec)
    return pd.DataFrame(out)


def latex_table(df, cols, caption, label):
    def fmt(row, c):
        v = row[c]
        if isinstance(v, str):
            return {"✓": r"\checkmark", "✗": r"$\times$"}.get(v, v)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "--"
        sd = row[c + " SD"] if (c + " SD") in row.index else None
        s = f"{v:.3f}" if c == "Params (M)" else f"{v:.1f}"
        return s + (f" $\\pm$ {sd:.1f}" if sd is not None and not np.isnan(sd) else "")
    lines = [r"\begin{table}[t]", r"\centering", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
             r"\setlength{\tabcolsep}{4pt}", r"\begin{tabular}{l" + "c" * (len(cols) - 1) + "}", r"\toprule",
             " & ".join(c.replace("%", r"\%") for c in cols) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        lines.append(" & ".join(fmt(r, c) for c in cols) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(lines)


def save_table(df, base, cols, caption, label):
    save_csv(df, base + ".csv", index=False)
    path = os.path.join(RESULT_DIR, base + ".tex")
    _atomic_write_text(path, latex_table(df, cols, caption, label))
    print(f"[Drive] saved {path}")


def plot_table(df, title, fname):
    d = df.dropna(subset=["MAE"])
    if d.empty:
        return
    fig, ax = plt.subplots(figsize=(max(7, 1.6 * len(d)), 4))
    x = np.arange(len(d)); w = 0.27
    for k, (c, col) in enumerate([("MAE", "tab:blue"), ("Corr. MAE", "tab:orange"), ("PGD MAE", "tab:red")]):
        ax.bar(x + (k - 1) * w, d[c], w, label=c, color=col, edgecolor="black")
    ax.set_xticks(x); ax.set_xticklabels(d["Configuration"], rotation=20, ha="right")
    ax.set_ylabel("MAE"); ax.set_title(title); ax.legend()
    plt.tight_layout(); save_fig(fig, fname); plt.show()


ABLATION_ROWS = [("Baseline", "baseline", (0, 0, 0, 0)), ("Model A", "model_a", (1, 0, 0, 0)),
                 ("Model B", "model_b", (1, 1, 0, 0)), ("Model C", "model_c", (1, 1, 1, 0)),
                 ("RMSC-Net", "rmsc_full", (1, 1, 1, 1))]
FLAG_COLS = ["MSRM", "DAA", "Consistency", "Adv. Training"]
ablation_df = result_table(ABLATION_ROWS, FLAG_COLS)
show_cols = ["Configuration"] + FLAG_COLS + METRIC_COLS + ["note"]
print(ablation_df[show_cols].round(2).to_string(index=False))
save_table(ablation_df, "ablation_main", ["Configuration"] + FLAG_COLS + METRIC_COLS,
           "Component-wise ablation on the ShanghaiTech test set. Robustness: corrupted MAE (12 corruptions) "
           "and PGD MAE, with degradation RD relative to the clean MAE.", "tab:ablation_main")
plot_table(ablation_df, "Component-wise ablation", "ablation_main.png")
ref = result_table([("LightARFS (main run, reference)", MAIN_NAME)])
print("\nReference:"); print(ref[["Configuration"] + METRIC_COLS].round(2).to_string(index=False))
update_summary("ablation_main", {r["Configuration"]: {c: r[c] for c in METRIC_COLS} for _, r in ablation_df.iterrows()})


## 24. Additional ablations

These use the same metrics and protocols as Section 23.

- **Dilation rates**: the ARFS/MSRM dilation set, `cfg.light_dilations` = (1, 2, 3, 5) vs `cfg.abl_dilation_sets`:
  - (1, 2, 3): a smaller receptive field;
  - (1, 2, 4, 8): an exponential, ASPP-like set with the widest receptive field, which may grid (holes in the effective kernel);
  - (1, 3, 5, 7): odd rates with a wide receptive field.

  (1, 2, 3, 5) is the hybrid-dilated-convolution choice: no common factor greater than 1, so there is no gridding, and it covers 3-11 px at stride 8.
- **Fusion strategy**: addition (default) vs concatenation + 1x1 conv vs adaptive softmax-weighted fusion.
- **Loss function** (Stage II fine-tuning of the same Model B checkpoint, same budget): density
  only, + count, + consistency, + feature consistency, complete objective (= Model C).
- **Robust training** (same two-phase budget from Model B): clean, corruption augmentation,
  adversarial, corruption + adversarial, complete consistency framework (= RMSC-Net).


In [ ]:
ADDITIONAL_GROUPS = {
    "Dilation rates": [(f"{tuple(FULL_SPEC['dilations'])} (default)", "model_b")] +
                      [(str(d), n) for n, d in DILATION_EXPS.items()],
    "Fusion strategy": [("Addition (default)", "model_b"), ("Concatenation", "fusion_concat"),
                        ("Adaptive weighted fusion", "fusion_adaptive")],
    "Loss function": [("Density only", "loss_d"), ("Density + count", "loss_dc"),
                      ("Density + consistency", "loss_dcons"), ("Density + feature consistency", "loss_dfeat"),
                      ("Complete objective", "model_c")],
    "Robust training": [("Clean training", "rob_clean"), ("Corruption augmentation", "rob_corrupt"),
                        ("Adversarial training", "rob_adv"), ("Corruption + adversarial", "rob_corrupt_adv"),
                        ("Complete consistency framework", "rmsc_full")],
}
GROUP_REFERENCE = {"Dilation rates": "model_b", "Fusion strategy": "model_b",
                   "Loss function": "model_c", "Robust training": "rmsc_full"}
additional_tables = {}
for group, rows in ADDITIONAL_GROUPS.items():
    df = result_table(rows)
    additional_tables[group] = df
    slug = group.lower().replace(" ", "_")
    print(f"\n=== {group} ===")
    print(df[["Configuration"] + METRIC_COLS + ["note"]].round(2).to_string(index=False))
    save_table(df, f"ablation_{slug}", ["Configuration"] + METRIC_COLS,
               f"Ablation: {group.lower()} (ShanghaiTech test set).", f"tab:ablation_{slug}")
    plot_table(df, f"Ablation: {group}", f"ablation_{slug}.png")


## 25. Statistical validation

**Units of observation.** Every model is evaluated on the same test images with identical
perturbations (fixed per-image seeds), so the per-image absolute count errors of two models are
**paired** observations. All tests compare such pairs; nothing is pooled across unpaired data. With
several training seeds, a per-image error is first averaged over the seeds, and seed-level results
(mean, SD and 95% t-interval of the test MAE over seeds) are reported separately. With a single seed,
the tests describe variability over images for the trained models. They do not cover
training-to-training variability, so add seeds to `cfg.abl_seeds` for that.

**Procedure (fixed in advance; every result is reported, significant or not).**
- **Descriptive**: MAE (= mean absolute error), SD, and 95% bootstrap CI (`cfg.stats_n_bootstrap` resamples, percentile) for every model and metric (clean, corruption, PGD).
- **Pairwise**: paired difference d = error(candidate) - error(reference); negative means the candidate is better. For each pair the notebook reports:
  - the mean ± SD of d and a paired-bootstrap 95% CI;
  - the Shapiro-Wilk test of d;
  - the **paired t-test** with Cohen's d_z;
  - the **Wilcoxon signed-rank test** with the rank-biserial correlation.

  The primary test is the t-test if Shapiro-Wilk does not reject normality at α, otherwise the Wilcoxon test.
- **Multiple comparisons**: Holm-Bonferroni within each family. A family is one comparison set (for example, the component contrasts) on one metric.
- **Repeated measures**: Friedman test with Kendall's W across all models of a family that are evaluated on the same images. A repeated-measures ANOVA is also reported when `statsmodels` is available, but sphericity is not tested and per-image errors are usually skewed, so Friedman is the primary omnibus test.
- **Representation stability**: RMSC-Net vs Model B on the per-image stability values from Section 22, averaged over the corruptions. Higher is better, so d > 0 favours RMSC-Net.


In [ ]:
ALPHA, NBOOT = cfg.stats_alpha, cfg.stats_n_bootstrap
STAT_METRICS = ["clean", "corruption", "pgd"]


def boot_ci(x, fn=np.mean, seed=0):
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(NBOOT, len(x)))
    vals = fn(x[idx], axis=1) if fn in (np.mean,) else np.array([fn(x[i]) for i in idx])
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def holm(pvals):
    p = np.asarray(pvals, float)
    out = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    order = np.argsort(p[ok])
    m = ok.sum()
    adj = np.minimum(1, np.maximum.accumulate((m - np.arange(m)) * p[ok][order]))
    tmp = np.empty(m); tmp[order] = adj
    out[ok] = tmp
    return out


def paired_compare(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = a - b
    n = len(d)
    res = {"n": n, "mean_a": a.mean(), "mean_b": b.mean(), "mean_diff": d.mean(), "sd_diff": d.std(ddof=1) if n > 1 else np.nan}
    res["ci_low"], res["ci_high"] = boot_ci(d)
    res["shapiro_p"] = sps.shapiro(d).pvalue if n >= 3 and d.std() > 0 else np.nan
    if n > 1 and d.std() > 0:
        t = sps.ttest_rel(a, b)
        res.update(t=t.statistic, p_t=t.pvalue, cohen_dz=d.mean() / d.std(ddof=1))
    else:
        res.update(t=np.nan, p_t=np.nan, cohen_dz=np.nan)
    nz = d[d != 0]
    if len(nz) > 0:
        w = sps.wilcoxon(a, b)
        ranks = sps.rankdata(np.abs(nz))
        res.update(W=w.statistic, p_w=w.pvalue, rank_biserial=(ranks[nz > 0].sum() - ranks[nz < 0].sum()) / ranks.sum())
    else:
        res.update(W=np.nan, p_w=np.nan, rank_biserial=np.nan)
    use_t = not np.isnan(res["shapiro_p"]) and res["shapiro_p"] >= ALPHA
    res["primary_test"] = "paired t-test" if use_t else "Wilcoxon signed-rank"
    res["p_primary"] = res["p_t"] if use_t else res["p_w"]
    res["effect_size"] = res["cohen_dz"] if use_t else res["rank_biserial"]
    res["effect_measure"] = "Cohen's d_z" if use_t else "rank-biserial r"
    return res


def errors_for(name, metric):
    """Per-image absolute errors, averaged over the available seeds (pairing stays per image)."""
    if name == MAIN_NAME:
        df = ensure_eval(name, ABL_COLUMNS, verbose=False)
        return None if df is None else per_image_errors(df, metric)
    errs = [per_image_errors(df, metric) for seed in cfg.abl_seeds
            for df in [ensure_eval(name, ABL_COLUMNS, seed=seed, verbose=False)] if df is not None]
    return np.mean(errs, axis=0) if errs else None


COMPONENT_CONTRASTS = [("+MSRM (Model A vs Baseline)", "model_a", "baseline"),
                       ("+DAA (Model B vs Model A)", "model_b", "model_a"),
                       ("+Consistency (Model C vs Model B)", "model_c", "model_b"),
                       ("+Adv. training (RMSC-Net vs Model C)", "rmsc_full", "model_c"),
                       ("RMSC-Net vs Baseline", "rmsc_full", "baseline")]
FAMILIES = {"component": COMPONENT_CONTRASTS,
            "reference": [("RMSC-Net vs LightARFS main", "rmsc_full", MAIN_NAME),
                          ("Model B vs LightARFS main", "model_b", MAIN_NAME)]}
for group, rows in ADDITIONAL_GROUPS.items():
    refn = GROUP_REFERENCE[group]
    FAMILIES[group] = [(f"{lab} vs reference", n, refn) for lab, n in rows if n != refn]

desc_rows, cmp_rows, omni_rows = [], [], []
all_names = available_models([MAIN_NAME] + EXPERIMENT_ORDER)
for name in all_names:
    for metric in STAT_METRICS:
        e = errors_for(name, metric)
        lo, hi = boot_ci(e)
        rlo, rhi = boot_ci(e, fn=lambda v: float(np.sqrt(np.mean(v ** 2))))
        desc_rows.append({"model": name, "label": model_label(name), "metric": metric, "n": len(e),
                          "MAE": e.mean(), "SD": e.std(ddof=1), "MAE_CI_low": lo, "MAE_CI_high": hi,
                          "RMSE": float(np.sqrt(np.mean(e ** 2))), "RMSE_CI_low": rlo, "RMSE_CI_high": rhi})

for fam, contrasts in FAMILIES.items():
    for metric in STAT_METRICS:
        fam_rows = []
        for label, cand, refn in contrasts:
            a, b = errors_for(cand, metric) if cand in all_names else None, errors_for(refn, metric) if refn in all_names else None
            if a is None or b is None:
                continue
            fam_rows.append({"family": fam, "metric": metric, "comparison": label, "candidate": cand,
                             "reference": refn, **paired_compare(a, b)})
        if fam_rows:
            p_adj = holm([r["p_primary"] for r in fam_rows])
            for r, p in zip(fam_rows, p_adj):
                r["p_holm"] = p
                r["significant"] = bool(p < ALPHA) if not np.isnan(p) else False
            cmp_rows += fam_rows
        models = [m for m in dict.fromkeys([c for _, c, _ in contrasts] + [r for _, _, r in contrasts]) if m in all_names]
        if len(models) >= 3:
            mat = np.column_stack([errors_for(m, metric) for m in models])
            fr = sps.friedmanchisquare(*mat.T)
            rec = {"family": fam, "metric": metric, "models": ", ".join(models), "n_images": mat.shape[0],
                   "friedman_chi2": fr.statistic, "friedman_p": fr.pvalue,
                   "kendall_W": fr.statistic / (mat.shape[0] * (mat.shape[1] - 1))}
            try:
                from statsmodels.stats.anova import AnovaRM
                long = pd.DataFrame({"img": np.repeat(np.arange(mat.shape[0]), mat.shape[1]),
                                     "model": np.tile(models, mat.shape[0]), "err": mat.ravel()})
                an = AnovaRM(long, "err", "img", within=["model"]).fit().anova_table
                rec.update(rm_anova_F=float(an["F Value"].iloc[0]), rm_anova_p=float(an["Pr > F"].iloc[0]))
            except Exception as e:
                rec.update(rm_anova_F=np.nan, rm_anova_p=np.nan)
            omni_rows.append(rec)

# representation stability: RMSC-Net vs Model B (higher similarity = more stable)
if stability_df is not None and {"rmsc_full", "model_b"} <= set(stability_df.model):
    fam_rows = []
    for metric in STABILITY_METRICS:
        for g in stability_df["group"].unique():
            sub = stability_df[stability_df.group == g].groupby(["model", "image_idx"])[metric].mean().unstack(0)
            if sub[["rmsc_full", "model_b"]].isna().any().any():
                continue
            fam_rows.append({"family": "stability", "metric": f"{metric} [{g}]", "comparison": "RMSC-Net vs Model B",
                             "candidate": "rmsc_full", "reference": "model_b",
                             **paired_compare(sub["rmsc_full"].to_numpy(), sub["model_b"].to_numpy())})
    if fam_rows:
        for r, p in zip(fam_rows, holm([r["p_primary"] for r in fam_rows])):
            r["p_holm"], r["significant"] = p, bool(p < ALPHA) if not np.isnan(p) else False
        cmp_rows += fam_rows

desc_df, cmp_df, omni_df = pd.DataFrame(desc_rows), pd.DataFrame(cmp_rows), pd.DataFrame(omni_rows)
for df_, nm in ((desc_df, "stats_descriptive.csv"), (cmp_df, "stats_comparisons.csv"), (omni_df, "stats_omnibus.csv")):
    if len(df_):
        save_csv(df_, nm, index=False)

# seed-level results (only with repeated training runs)
seed_rows = []
if len(cfg.abl_seeds) > 1:
    for name in EXPERIMENT_ORDER:
        for metric in STAT_METRICS:
            maes = [per_image_errors(df, metric).mean() for seed in cfg.abl_seeds
                    for df in [ensure_eval(name, ABL_COLUMNS, seed=seed, verbose=False)] if df is not None]
            if len(maes) >= 2:
                m, s = float(np.mean(maes)), float(np.std(maes, ddof=1))
                h = sps.t.ppf(0.975, len(maes) - 1) * s / np.sqrt(len(maes))
                seed_rows.append({"model": name, "metric": metric, "n_seeds": len(maes), "mean_MAE": m, "SD": s,
                                  "CI_low": m - h, "CI_high": m + h})
    if seed_rows:
        save_csv(pd.DataFrame(seed_rows), "stats_seed_level.csv", index=False)
        print(pd.DataFrame(seed_rows).round(2).to_string(index=False))

pd.set_option("display.width", 250)
if len(desc_df):
    print("Descriptive statistics (per-image absolute error):")
    print(desc_df.drop(columns=["model"]).round(2).to_string(index=False))
if len(cmp_df):
    print("\nPaired comparisons (d = candidate - reference; Holm within family x metric):")
    print(cmp_df[["family", "metric", "comparison", "n", "mean_diff", "ci_low", "ci_high", "shapiro_p",
                  "primary_test", "p_primary", "p_holm", "effect_measure", "effect_size", "significant"]]
          .round(4).to_string(index=False))
if len(omni_df):
    print("\nRepeated-measures omnibus tests:")
    print(omni_df.round(4).to_string(index=False))
if not len(cmp_df):
    print("No comparison possible yet -- the experiments of Sections 19-20 have to finish first.")
update_summary("statistics", {"n_comparisons": len(cmp_df),
                              "n_significant_holm": int(cmp_df["significant"].sum()) if len(cmp_df) else 0,
                              "alpha": ALPHA, "correction": "Holm within family x metric"})


## 26. Results and discussion (generated from the saved results)

This cell writes `results/results_discussion.md` from the numbers on Drive: Section 11 (test
metrics), Sections 21-24 (robustness, ablations) and Section 25 (statistics). Every statement
carries its effect size, 95% CI and Holm-adjusted p-value. Non-significant results are reported as
such, and no wording is chosen according to whether a result came out "favourable". The text is a
factual first draft; check it against the figures before using it in the paper. The failure-case
figure is saved as `figures/failure_cases.png`.


In [ ]:
from IPython.display import Markdown, display


def _csv(name):
    p = os.path.join(RESULT_DIR, name)
    return pd.read_csv(p) if os.path.exists(p) else None


def _cmp(family, comparison, metric):
    if cmp_df is None or not len(cmp_df):
        return None
    r = cmp_df[(cmp_df.family == family) & (cmp_df.comparison == comparison) & (cmp_df.metric == metric)]
    return None if r.empty else r.iloc[0]


def _cmp_text(r, unit="MAE"):
    if r is None:
        return "not available yet (experiments missing)"
    verdict = "statistically significant" if r["significant"] else "not statistically significant"
    return (f"Δ{unit} = {r['mean_diff']:+.2f} (95% CI {r['ci_low']:+.2f} to {r['ci_high']:+.2f}; "
            f"{r['primary_test']}, Holm-adjusted p = {r['p_holm']:.3g}; {r['effect_measure']} = {r['effect_size']:+.2f}), "
            f"{verdict} at α = {ALPHA}")


def _mae(name, metric="clean"):
    if desc_df is None or not len(desc_df):
        return None
    r = desc_df[(desc_df.model == name) & (desc_df.metric == metric)]
    return None if r.empty else r.iloc[0]


def _mae_text(name, metric="clean"):
    r = _mae(name, metric)
    if r is None:
        return f"{model_label(name) if (name == MAIN_NAME or name in EXPERIMENTS) else name}: not available"
    return f"{r['label']}: MAE {r['MAE']:.2f} (95% CI {r['MAE_CI_low']:.2f}-{r['MAE_CI_high']:.2f}), RMSE {r['RMSE']:.2f}"


L = ["# Results and discussion", "",
     f"_Generated {_now()} from `{RESULT_DIR}`. Test set: {len(test_dataset)} images of ShanghaiTech part {cfg.part}; "
     f"seeds: {list(cfg.abl_seeds)}. Δ = candidate - reference (negative = lower error)._", ""]

# 1 clean accuracy
tm = json.load(open(os.path.join(RESULT_DIR, "test_metrics.json"))) if os.path.exists(os.path.join(RESULT_DIR, "test_metrics.json")) else None
L += ["## Clean-image accuracy", ""]
if tm:
    L.append(f"- Main LightARFS model (Section 11, headline): MAE {tm['MAE']:.2f}, RMSE {tm['RMSE']:.2f}, NAE {tm['NAE']:.3f} "
             f"(best epoch {tm['epoch']}).")
for n in ("model_b", "model_c", "rmsc_full"):
    L.append(f"- {_mae_text(n)}.")
L.append(f"- RMSC-Net vs the main LightARFS model on clean images: {_cmp_text(_cmp('reference', 'RMSC-Net vs LightARFS main', 'clean'))}.")
L.append("")

# 2 dense scenes
L += ["## Dense-scene performance", ""]
for n in available_models([MAIN_NAME, "model_b", "rmsc_full"]):
    df = ensure_eval(n, ABL_COLUMNS, verbose=False)
    thirds = pd.qcut(df["gt"].rank(method="first"), 3, labels=["sparse", "medium", "dense"])
    err = df["cleantta"] - df["gt"]
    g = pd.DataFrame({"t": thirds, "ae": err.abs(), "e": err, "re": err / df["gt"].clip(lower=1)}).groupby("t", observed=True)
    parts = [f"{t}: MAE {v.ae.mean():.1f}, bias {v.e.mean():+.1f} ({100 * v.re.mean():+.0f}%)" for t, v in g]
    L.append(f"- {model_label(n)} -- " + "; ".join(parts) + ".")
L.append("- A negative bias in the dense third means systematic under-counting of very crowded scenes.")
L.append("")

# 3 environmental robustness
cdf = _csv("robust_corruption_full.csv")
L += ["## Environmental robustness", ""]
if cdf is not None:
    for n, sub in cdf.groupby("model"):
        worst = sub.groupby("corruption")["RD %"].mean().sort_values(ascending=False)
        L.append(f"- {model_label(n)}: mean corrupted MAE {sub.MAE.mean():.1f} (mean RD {sub['RD %'].mean():+.1f}%); "
                 f"hardest: {', '.join(f'{k} ({v:+.0f}%)' for k, v in worst.head(3).items())}; "
                 f"easiest: {', '.join(f'{k} ({v:+.0f}%)' for k, v in worst.tail(2).items())}.")
L.append(f"- Consistency learning on corrupted images (Model C vs Model B): {_cmp_text(_cmp('component', '+Consistency (Model C vs Model B)', 'corruption'))}.")
L.append(f"- RMSC-Net vs Model B is in the robust-training family below; vs the main model: "
         f"{_cmp_text(_cmp('reference', 'RMSC-Net vs LightARFS main', 'corruption'))}.")
L.append("")

# 4 adversarial robustness
adf = _csv("robust_adversarial_full.csv")
L += ["## Adversarial robustness", ""]
if adf is not None:
    for (n, a), sub in adf.groupby(["model", "attack"]):
        L.append(f"- {model_label(n)}, {a.upper()}: " + ", ".join(f"ε={e}/255 MAE {m:.1f}" for e, m in zip(sub.eps_255, sub.MAE))
                 + f" (clean {sub['clean MAE'].iloc[0]:.1f}).")
L.append(f"- Effect of adversarial training under PGD (RMSC-Net vs Model C): {_cmp_text(_cmp('component', '+Adv. training (RMSC-Net vs Model C)', 'pgd'))}.")
L.append("")

# 5 scale / 6 occlusion
for title, fname, col in (("Scale robustness", "robust_scale.csv", "scale"), ("Occlusion handling", "robust_occlusion.csv", "occluded")):
    d = _csv(fname)
    L += [f"## {title}", ""]
    if d is None:
        L.append("- not available yet.")
    else:
        for n, sub in d.groupby("model"):
            L.append(f"- {model_label(n)}: " + ", ".join(f"{col} {v:g}: MAE {m:.1f}" for v, m in zip(sub[col], sub.MAE)) + ".")
    if col == "occluded":
        L.append("- Occlusion is scored against the people that stay visible, so a robust model predicts no density under the occluders.")
    L.append("")

# 7-10 component effectiveness
L += ["## Effectiveness of the components", ""]
for label, _, _ in COMPONENT_CONTRASTS:
    L.append(f"- **{label}**")
    for metric in STAT_METRICS:
        L.append(f"  - {metric}: {_cmp_text(_cmp('component', label, metric))}")
for fam in ("Loss function", "Robust training", "Dilation rates", "Fusion strategy"):
    L.append(f"- **{fam}** (each vs {model_label(GROUP_REFERENCE[fam])}):")
    for label, n in ADDITIONAL_GROUPS[fam]:
        if n == GROUP_REFERENCE[fam]:
            continue
        L.append(f"  - {label}: clean {_cmp_text(_cmp(fam, f'{label} vs reference', 'clean'))}; "
                 f"corruption {_cmp_text(_cmp(fam, f'{label} vs reference', 'corruption'))}; "
                 f"PGD {_cmp_text(_cmp(fam, f'{label} vs reference', 'pgd'))}.")
if omni_df is not None and len(omni_df):
    for _, r in omni_df.iterrows():
        L.append(f"- Friedman ({r['family']}, {r['metric']}, {r['models']}): χ² = {r['friedman_chi2']:.2f}, "
                 f"p = {r['friedman_p']:.3g}, Kendall's W = {r['kendall_W']:.3f}.")
if stability_df is not None:
    st = stability_df.groupby(["model", "group"])[["spatial_corr", "reliability_corr", "cka_daa", "cka_f8"]].mean()
    L.append("- Representation stability (mean similarity clean vs perturbed): " + "; ".join(
        f"{model_label(m)} / {g}: spatial attention r = {v.spatial_corr:.3f}, reliability r = {v.reliability_corr:.3f}, "
        f"decoder-input CKA = {v.cka_daa:.3f}, encoder-f8 CKA = {v.cka_f8:.3f}" for (m, g), v in st.iterrows()) + ".")
    for _, r in (cmp_df[cmp_df.family == "stability"].iterrows() if len(cmp_df) else []):
        L.append(f"  - {r['metric']}: {_cmp_text(r, unit='similarity')}.")
L.append("")

# 11 accuracy vs robustness, 12 accuracy vs complexity
rs = _csv("robustness_summary_all_models.csv")
L += ["## Accuracy-versus-robustness and accuracy-versus-complexity trade-offs", ""]
if rs is not None and len(rs) >= 3:
    rho_c = sps.spearmanr(rs["MAE"], rs["Corr. MAE"])
    rho_a = sps.spearmanr(rs["MAE"], rs["PGD MAE"])
    L.append(f"- Across {len(rs)} trained models, Spearman ρ(clean MAE, corrupted MAE) = {rho_c.correlation:.2f} "
             f"(p = {rho_c.pvalue:.3g}); ρ(clean MAE, PGD MAE) = {rho_a.correlation:.2f} (p = {rho_a.pvalue:.3g}). "
             "(Descriptive: the models are not independent samples.)")
    pareto = []
    for _, r in rs.sort_values("GFLOPs").iterrows():
        if not pareto or r["MAE"] < pareto[-1]["MAE"]:
            pareto.append(r)
    L.append("- Accuracy-complexity Pareto front (GFLOPs at 1024x768 -> clean MAE): " +
             "; ".join(f"{p['label']} {p['GFLOPs']:.2f} GFLOPs / {p['Params (M)']:.3f}M -> {p['MAE']:.1f}" for p in pareto) + ".")
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(rs["MAE"], rs["Corr. MAE"], c="tab:orange", label="corrupted")
    axes[0].scatter(rs["MAE"], rs["PGD MAE"], c="tab:red", label="PGD")
    axes[1].scatter(rs["GFLOPs"], rs["MAE"])
    for _, r in rs.iterrows():
        axes[0].annotate(r["model"], (r["MAE"], r["Corr. MAE"]), fontsize=7)
        axes[1].annotate(r["model"], (r["GFLOPs"], r["MAE"]), fontsize=7)
    axes[0].set_xlabel("clean MAE"); axes[0].set_ylabel("perturbed MAE"); axes[0].legend(); axes[0].set_title("Accuracy vs robustness")
    axes[1].set_xlabel("GFLOPs (1024x768)"); axes[1].set_ylabel("clean MAE"); axes[1].set_title("Accuracy vs complexity")
    plt.tight_layout(); save_fig(fig, "tradeoffs.png"); plt.show()
else:
    L.append("- needs at least 3 trained models.")
L.append("")

# 13 generalisation
L += ["## Generalisation", ""]
for n in available_models([MAIN_NAME, "model_b", "rmsc_full"]):
    _, info = load_eval_model(n)
    df = ensure_eval(n, ABL_COLUMNS, verbose=False)
    L.append(f"- {model_label(n)}: validation MAE {info['val_mae']:.2f} (selection set, no TTA) vs test MAE "
             f"{abs_err(df, 'clean').mean():.2f} (no TTA).")
xp = _csv("robust_cross_part.csv")
if xp is not None:
    for _, r in xp.iterrows():
        L.append(f"- Cross-dataset: {model_label(r['model'])} trained on {r['trained_on']} -> {r['tested_on']}: "
                 f"MAE {r['MAE']:.1f}, RMSE {r['RMSE']:.1f} (in-domain {r['in-domain MAE']:.1f}).")
L.append("")

# 14 failure cases
L += ["## Failure cases", ""]
final = (available_models(["rmsc_full"]) or available_models([MAIN_NAME]) or [None])[0]
if final:
    df = ensure_eval(final, ABL_COLUMNS, verbose=False)
    df = df.assign(err=df["cleantta"] - df["gt"]).assign(ae=lambda d: d.err.abs())
    rho = sps.spearmanr(df["ae"], df["gt"])
    worst = df.sort_values("ae", ascending=False).head(5)
    L.append(f"- {model_label(final)}: the 5 worst images account for {100 * worst.ae.sum() / df.ae.sum():.0f}% of the total "
             f"absolute error; Spearman ρ(|error|, GT count) = {rho.correlation:.2f} (p = {rho.pvalue:.3g}).")
    for _, r in worst.iterrows():
        L.append(f"  - {r['image']}: GT {r['gt']:.0f}, predicted {r['cleantta']:.0f} ({r['err']:+.0f}, {100 * r['err'] / max(r['gt'], 1):+.0f}%)")
    model_f, _ = load_eval_model(final)
    name_to_idx = {os.path.basename(dataset_image_path(test_dataset, i)): i for i in range(len(test_dataset))}
    fig, axes = plt.subplots(min(4, len(worst)), 4, figsize=(16, 3.6 * min(4, len(worst))), squeeze=False)
    for row, (_, r) in enumerate(worst.head(4).iterrows()):
        img, dens, _ = test_dataset[name_to_idx[r["image"]]]
        pred = _tp(model_f, img.unsqueeze(0).to(DEVICE))[0, 0].cpu().numpy() / cfg.density_scale
        gtm = dens[0].numpy() / cfg.density_scale
        e = pred - gtm; em = max(np.abs(e).max(), 1e-6)
        for c, (data, cmap, t) in enumerate([(to_uint8(img), None, r["image"]), (gtm, "jet", f"GT {r['gt']:.0f}"),
                                             (pred, "jet", f"pred {r['cleantta']:.0f}"), (e, "bwr", "error")]):
            axes[row, c].imshow(data, cmap=cmap, **({"vmin": -em, "vmax": em} if cmap == "bwr" else {}))
            axes[row, c].set_title(t); axes[row, c].axis("off")
    plt.tight_layout(); save_fig(fig, "failure_cases.png"); plt.show()
L.append("")
L += ["## Limitations", "",
      f"- Statistical tests are over test images for fixed trained models; with {len(cfg.abl_seeds)} seed(s), "
      "training-run variance is " + ("estimated in `stats_seed_level.csv`." if len(cfg.abl_seeds) > 1 else "not captured -- add seeds to `cfg.abl_seeds`."),
      "- frost / snow / rain are simplified noise models (as in RMSC-Net), not physically rendered weather.",
      "- White-box attacks use the density-MSE objective with the scored tiled inference; adaptive attacks against the specific defence were not run.",
      "- Checkpoints are selected on the clean validation MAE; robustness was never used for model selection.", ""]

report = "\n".join(L)
_atomic_write_text(os.path.join(RESULT_DIR, "results_discussion.md"), report)
print(f"[Drive] saved {os.path.join(RESULT_DIR, 'results_discussion.md')}")
display(Markdown(report))
